# Household Electricity-Related CO₂ Emissions and Explainable AI
## Final May–November 2024 six-hour reconstruction, regression, classification and XAI workflow

### Reproducible execution rule

This notebook is intentionally distributed with **cleared outputs** after the
final methodological revision. First run
`Thesis_Master_Pipeline_WITH_UNCLIPPED_FEATURE_EXPORT.ipynb` far enough to
produce `clean_fm_clusters.*` and `raw_active_features_unclipped.*`, then place
the required six-hour, billing and Wave 1 files in the project folder and use
**Restart Kernel → Run All**. Do not mix outputs from an older Carbon/XAI run
with this revised code.

### Purpose

This notebook adds a carbon-emission and explainable-machine-learning extension
to the completed smart-meter thesis. It does **not** rerun, alter or replace the
locked K-Means clustering.

Two related supervised tasks are retained:

1. **Primary  regression:** estimate continuous mean monthly
   household electricity-related CO₂ emissions.
2. **Secondary classification:** identify households in the relatively
   high-emission group, defined by the development-set 75th percentile.

Both tasks use only direct-target-excluded behavioural load-shape predictors.
The continuous target  uses the
selected Sri Lankan electricity factor of **0.71 kg CO₂/kWh only**.


### Final examiner-critical ML safeguards

The final revision adds only analyses that materially strengthen the supervised
ML evidence:

- **fully nested high-emission target construction:** outer-training Q75 labels
  are retained for honest outer validation, while each inner-training fold now
  derives its own Q75 threshold for hyperparameter tuning;
- **classification-family selection by mean outer-fold PR-AUC**, avoiding pooled
  OOF PR-AUC as the primary family-selection statistic when fold-specific
  training-derived Q75 thresholds vary slightly;
- **one prespecified non-negative regression prediction rule**, with predictions
  clipped at zero before inner-CV scoring, outer-CV evaluation, held-out
  evaluation and regression permutation importance;
- **development-only leave-one-feature-out ablation** of the five behavioural
  predictors, using the same outer folds and locked fold-specific model
  settings;
- **held-out regression residual/error diagnostics**, including error by
  actual-emission quartile; and
- **paired bootstrap confidence intervals** for the matched temporal-only versus
  temporal + Wave 1 performance differences.

These diagnostics are not used to redesign the locked final model after seeing
the held-out test results.

### Period alignment

| Component | Period |
|---|---|
| Locked 15-minute behavioural features and archetypes | May–November 2024 |
| Six-hour reconstructed monthly electricity | Eligible high-quality months within May–November 2024 |
| Electricity-related CO₂ calculation | Mean across each household's eligible months |
| Billing validation of the reconstruction | May–October 2024 |
| Survey variables, when available | Existing Wave 1 household records |

### Full-calendar-month reconstruction and cohort rule

1. Physically valid cumulative-register differences whose endpoints are in the
   same month are retained in full.
2. An interval crossing one adjacent month boundary is allocated by elapsed time
   only when its total gap is no more than **48 hours**. Shorter 8-, 12- and
   24-hour rules remain explicit sensitivity analyses.
3. Negative/reset differences and rates above 24 kWh/hour are excluded.
4. Allocated energy must be conserved.
5. Every included household-month must represent at least 95% of calendar hours.
6. The **actual register readings**, not synthetic allocated boundary points,
   must occur within 6.5 hours of the start and end of the month.
7. A household must contain at least **five valid full months** within the
   seven-month May–November window and must be represented in both the early
   (May–July) and later (August–November) parts of the window.

The minimum-five-month rule avoids the severe complete-case attrition caused by
requiring all seven months while preserving a majority of the study window.
Missing months are not imputed, and reconstructed electricity is not divided by
coverage. The household target is the mean across valid full months; observed
month count and month composition are audited and reported.

### Carbon outcome

The outcome is **household electricity-related operational CO₂ emissions**, not
the household's complete carbon footprint. Emissions are calculated as
reconstructed imported electricity multiplied by **0.71 kg CO₂/kWh**.



In [ ]:
# ============================================================
# STEP 1 — PACKAGE CHECK, IMPORTS AND FIXED CONFIGURATION
# ============================================================

from pathlib import Path
import importlib.util
import hashlib
import json
import math
import pickle
import platform
import warnings

required_packages = {
    "joblib": "joblib",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
}

missing_packages = [
    pip_name
    for import_name, pip_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    raise ModuleNotFoundError(
        "Missing packages: "
        + ", ".join(missing_packages)
        + "\nRun once in a new cell:\n%pip install "
        + " ".join(missing_packages)
    )

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn

from IPython.display import display
from scipy.stats import (
    chi2_contingency,
    kruskal,
    mannwhitneyu,
    pearsonr,
    spearmanr,
)

from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    ParameterGrid,
    cross_val_predict,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "carbon_sixhour_xai_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEACHER_MATCHED_RANDOM_STATE = 2026
TEACHER_REGRESSION_SELECTION_OBJECTIVE = "r2"
TEACHER_REGRESSION_ADD_MISSING_INDICATORS = True
TEACHER_REGRESSION_INCLUDE_LOG_TARGET = True

TARGET_START = pd.Period("2024-05", freq="M")
TARGET_END = pd.Period("2024-11", freq="M")
TARGET_MONTHS = pd.period_range(TARGET_START, TARGET_END, freq="M")
EARLY_TARGET_MONTHS = TARGET_MONTHS[:3]
LATE_TARGET_MONTHS = TARGET_MONTHS[3:]

BILLING_VALIDATION_START = pd.Period("2024-05", freq="M")
BILLING_VALIDATION_END = pd.Period("2024-10", freq="M")

EXPECTED_LOCKED_HOUSEHOLDS = 2227
EXPECTED_SIXHOUR_FILE_COUNT = 5
EXPECTED_CLUSTER_COUNTS = {
    "Peak-Exposed": 145,
    "Typical Household": 1426,
    "Heavy Continuous": 515,
    "Daytime-Active": 141,
}

# Six-hour reconstruction rules supported by the validated audit.
MAX_RATE_KWH_PER_HOUR = 24.0
PRIMARY_MAX_GAP_HOURS = 48.0
SENSITIVITY_MAX_GAP_HOURS = [8.0, 12.0, 24.0, 48.0]
MIN_MONTH_COVERAGE = 0.95
MAX_BOUNDARY_GAP_HOURS = 6.5
PRIMARY_MIN_VALID_MONTHS = 5
MIN_PRIMARY_HOUSEHOLDS = 300

# Prespecified source-method validation gates.
MIN_VALIDATION_HOUSEHOLD_MONTHS = 1000
MIN_VALIDATION_SPEARMAN = 0.85
MAX_VALIDATION_MEDIAN_APE = 0.20

# Only this factor is used anywhere in the final analysis.
GRID_FACTOR_KG_CO2_PER_KWH = 0.71

HIGH_EMISSION_QUANTILE = 0.75
TEST_SIZE = 0.20
OUTER_CV_SPLITS = 5
INNER_CV_SPLITS = 4
FINAL_TUNING_CV_SPLITS = 5
SENSITIVITY_CV_SPLITS = 5
SURVEY_OUTER_CV_SPLITS = 3
SURVEY_INNER_CV_SPLITS = 3
STABILITY_ALTERNATIVE_SEEDS = [17, 73]
STABILITY_OUTER_CV_SPLITS = 3
STABILITY_INNER_CV_SPLITS = 3
MIN_TEST_POSITIVES_FOR_STABLE_POINT_ESTIMATES = 20
BOOTSTRAP_ITERATIONS = 1000
SEARCH_N_JOBS = -1

ARCHETYPE_ORDER = [
    "Peak-Exposed",
    "Typical Household",
    "Heavy Continuous",
    "Daytime-Active",
]

TEMPORAL_SHAPE_FEATURES = [
    "evening_peak_ratio",
    "night_baseload_ratio",
    "load_uniformity",
    "load_factor",
    "leisure_usage_ratio",
]

print("Project folder:", PROJECT_ROOT)
print("Output folder:", OUTPUT_DIR)
print("Target period:", TARGET_START, "to", TARGET_END)
print("Electricity factor:", GRID_FACTOR_KG_CO2_PER_KWH, "kg CO2/kWh")
print("Primary adjacent cross-month maximum gap:", PRIMARY_MAX_GAP_HOURS, "hours")
print("Minimum monthly represented coverage:", MIN_MONTH_COVERAGE)
print("Maximum start/end boundary gap:", MAX_BOUNDARY_GAP_HOURS, "hours")
print("Minimum valid full months per household:", PRIMARY_MIN_VALID_MONTHS)


In [ ]:
# ============================================================
# STEP 2 — LOCATE THE EXISTING PROJECT FILES
# ============================================================

CLUSTER_CANDIDATES = [
    PROJECT_ROOT / "clean_fm_clusters.pkl",
    PROJECT_ROOT / "clean_fm_clusters.csv",
    PROJECT_ROOT / "clean_fm_for_carbon.csv",
    PROJECT_ROOT / "results" / "clusters" / "clean_fm_clusters.pkl",
    PROJECT_ROOT / "results" / "clusters" / "clean_fm_clusters.csv",
]

RAW_FEATURE_CANDIDATES = [
    PROJECT_ROOT / "raw_active_features_unclipped.pkl",
    PROJECT_ROOT / "raw_active_features_unclipped.csv",
    PROJECT_ROOT / "results" / "features" / "raw_active_features_unclipped.pkl",
    PROJECT_ROOT / "results" / "features" / "raw_active_features_unclipped.csv",
]

MONTHLY_BILLING_CANDIDATES = [
    PROJECT_ROOT / "monthly_consumption.csv",
    PROJECT_ROOT / "consumption_data" / "monthly_consumption.csv",
    PROJECT_ROOT
    / "Electricity"
    / "data"
    / "data"
    / "consumption_data"
    / "non_smart_meter"
    / "monthly_consumption.csv",
]

SURVEY_CANDIDATES = [
    PROJECT_ROOT / "survey_outputs" / "socio_demographic_household_dataset_FINAL.csv",
    PROJECT_ROOT / "socio_demographic_household_dataset_FINAL.csv",
    PROJECT_ROOT / "ANALYSIS2_consumption_group_profile_dataset.csv",
    PROJECT_ROOT / "MAIN_ANALYSIS_1413_monthly_smart_wave1_CLEAN.csv",
    PROJECT_ROOT / "wave1_household_features_1413_main.csv",
    PROJECT_ROOT / "model_outputs" / "phase3_final" / "PHASE3_frozen_modelling_source.csv",
    PROJECT_ROOT / "socio_merged.csv",
    PROJECT_ROOT / "results" / "socio" / "socio_merged.csv",
]

WAVE1_RAW_FILENAMES = {
    "household": "w1_household_information_and_history.csv",
    "demographics": "w1_demographics.csv",
    "appliances": "w1_appliances.csv",
    "ac_roster": "w1_ac_roster.csv",
}


def first_recursive_file(filename):
    direct = PROJECT_ROOT / filename

    if direct.exists():
        return direct

    matches = sorted(
        PROJECT_ROOT.rglob(filename),
        key=lambda path: (len(path.parts), str(path).lower()),
    )

    return matches[0] if matches else None


wave1_raw_paths = {
    key: first_recursive_file(filename)
    for key, filename in WAVE1_RAW_FILENAMES.items()
}


def first_existing(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


cluster_path = first_existing(CLUSTER_CANDIDATES)
raw_feature_path = first_existing(RAW_FEATURE_CANDIDATES)
monthly_billing_path = first_existing(MONTHLY_BILLING_CANDIDATES)
survey_paths = [
    candidate
    for candidate in SURVEY_CANDIDATES
    if candidate.exists()
]
survey_path = survey_paths[0] if survey_paths else None
sixhour_files = sorted(PROJECT_ROOT.glob("smart_6hour_*.csv"))

input_audit = pd.DataFrame([
    {
        "input": "locked cluster-feature table",
        "path": str(cluster_path) if cluster_path else None,
        "found": cluster_path is not None,
        "required": True,
    },
    {
        "input": "unclipped active behavioural-feature table",
        "path": str(raw_feature_path) if raw_feature_path else None,
        "found": raw_feature_path is not None,
        "required": True,
    },
    {
        "input": "monthly billing file",
        "path": str(monthly_billing_path) if monthly_billing_path else None,
        "found": monthly_billing_path is not None,
        "required": True,
    },
    {
        "input": "six-hour cumulative-register files",
        "path": "; ".join(str(path) for path in sixhour_files),
        "found": len(sixhour_files) == EXPECTED_SIXHOUR_FILE_COUNT,
        "required": True,
    },
    {
        "input": "clean survey table(s)",
        "path": "; ".join(str(path) for path in survey_paths),
        "found": len(survey_paths) > 0,
        "required": False,
    },
    {
        "input": "raw Wave 1 files for Wave-1 household-context variables",
        "path": "; ".join(
            f"{key}={path}"
            for key, path in wave1_raw_paths.items()
            if path is not None
        ),
        "found": all(
            path is not None
            for path in wave1_raw_paths.values()
        ),
        "required": False,
    },
])

display(input_audit)

if cluster_path is None:
    raise FileNotFoundError(
        "The final clean_fm_clusters.pkl/csv output was not found."
    )

if raw_feature_path is None:
    raise FileNotFoundError(
        "raw_active_features_unclipped.pkl/csv was not found. Run the "
        "updated thesis master notebook through the ghost-purge/clustering "
        "cell once so the pre-clipping feature export is created. The "
        "already clipped clean_fm_clusters file cannot be reversed safely."
    )

if monthly_billing_path is None:
    raise FileNotFoundError("monthly_consumption.csv was not found.")

if len(sixhour_files) != EXPECTED_SIXHOUR_FILE_COUNT:
    raise FileNotFoundError(
        f"Exactly {EXPECTED_SIXHOUR_FILE_COUNT} files named "
        "smart_6hour_*.csv are required in this notebook folder. "
        f"Found {len(sixhour_files)}."
    )

print("Required file check: PASS")

In [ ]:
# ============================================================
# STEP 3 — LOAD AND AUDIT THE LOCKED CLUSTER-FEATURE TABLE
# ============================================================

def load_table(path):
    if path.suffix.lower() in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    return pd.read_csv(path, low_memory=False)


def normalise_household_id(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def ensure_household_id(frame):
    result = frame.copy()

    if "household_ID" not in result.columns:
        result = result.reset_index()

        if "household_ID" not in result.columns:
            result = result.rename(
                columns={result.columns[0]: "household_ID"}
            )

    result["household_ID"] = normalise_household_id(
        result["household_ID"]
    )

    return result


def infer_archetype_map(frame):
    for name_column in [
        "cluster_name",
        "final_cluster_name",
        "archetype",
    ]:
        if name_column not in frame.columns:
            continue

        candidate = (
            frame[["final_cluster", name_column]]
            .dropna()
            .drop_duplicates()
        )

        if (
            candidate["final_cluster"].nunique() == 4
            and candidate[name_column].nunique() == 4
            and set(candidate[name_column]) == set(ARCHETYPE_ORDER)
        ):
            return candidate.set_index(
                "final_cluster"
            )[name_column].to_dict()

    # Exact fallback used in the locked master pipeline.
    medians = frame.groupby(
        "final_cluster",
        observed=True,
    )[
        [
            "evening_peak_ratio",
            "night_baseload_ratio",
            "load_factor",
        ]
    ].median()

    if len(medians) != 4:
        raise ValueError("Exactly four locked clusters are required.")

    remaining = set(medians.index.tolist())
    mapping = {}

    heavy_id = medians["load_factor"].idxmax()
    mapping[heavy_id] = "Heavy Continuous"
    remaining.remove(heavy_id)

    peak_id = medians.loc[
        list(remaining),
        "evening_peak_ratio",
    ].idxmax()
    mapping[peak_id] = "Peak-Exposed"
    remaining.remove(peak_id)

    daytime_id = medians.loc[
        list(remaining),
        "night_baseload_ratio",
    ].idxmin()
    mapping[daytime_id] = "Daytime-Active"
    remaining.remove(daytime_id)

    mapping[next(iter(remaining))] = "Typical Household"
    return mapping


cluster_df = ensure_household_id(load_table(cluster_path))

required_cluster_columns = {
    "household_ID",
    "final_cluster",
    "evening_peak_ratio",
    "night_baseload_ratio",
    "load_uniformity",
    "load_factor",
    "leisure_usage_ratio",
    "log_daily_kwh",
}

missing_cluster_columns = required_cluster_columns.difference(
    cluster_df.columns
)

if missing_cluster_columns:
    raise ValueError(
        "Locked cluster table is missing: "
        + ", ".join(sorted(missing_cluster_columns))
    )

if cluster_df["household_ID"].duplicated().any():
    raise ValueError(
        "The locked cluster table contains duplicate household IDs."
    )

if len(cluster_df) != EXPECTED_LOCKED_HOUSEHOLDS:
    raise RuntimeError(
        f"The wrong clustering output appears to be loaded. "
        f"Found {len(cluster_df):,} households; "
        f"expected {EXPECTED_LOCKED_HOUSEHOLDS:,}."
    )

cluster_name_map = infer_archetype_map(cluster_df)
cluster_df["cluster_name"] = cluster_df["final_cluster"].map(
    cluster_name_map
)

if set(cluster_df["cluster_name"].dropna()) != set(ARCHETYPE_ORDER):
    raise RuntimeError(
        "All four canonical archetype names were not recovered."
    )

for feature in TEMPORAL_SHAPE_FEATURES + ["log_daily_kwh"]:
    cluster_df[feature] = pd.to_numeric(
        cluster_df[feature],
        errors="coerce",
    )

if cluster_df[TEMPORAL_SHAPE_FEATURES].isna().any().any():
    raise RuntimeError(
        "The locked temporal shape features contain missing values."
    )

cluster_population = (
    cluster_df["cluster_name"]
    .value_counts()
    .reindex(ARCHETYPE_ORDER)
    .rename("households")
    .reset_index()
)

observed_cluster_counts = dict(
    zip(
        cluster_population["cluster_name"],
        cluster_population["households"].astype(int),
    )
)

if observed_cluster_counts != EXPECTED_CLUSTER_COUNTS:
    raise RuntimeError(
        "The locked archetype counts do not match the validated master "
        f"pipeline. Observed: {observed_cluster_counts}; "
        f"expected: {EXPECTED_CLUSTER_COUNTS}."
    )

display(cluster_population)
print("Locked cluster-feature audit: PASS")

# ------------------------------------------------------------------
# Load the same household features before thesis-wide clipping.
# These are the only feature values used by the supervised ML section.
# The clipped cluster table remains in use only for locked archetype labels
# and descriptive archetype comparisons.
# ------------------------------------------------------------------
raw_feature_df = ensure_household_id(load_table(raw_feature_path))

required_raw_feature_columns = {
    "household_ID",
    *TEMPORAL_SHAPE_FEATURES,
    "log_daily_kwh",
}

missing_raw_feature_columns = required_raw_feature_columns.difference(
    raw_feature_df.columns
)

if missing_raw_feature_columns:
    raise ValueError(
        "The unclipped feature table is missing: "
        + ", ".join(sorted(missing_raw_feature_columns))
    )

if raw_feature_df["household_ID"].duplicated().any():
    raise RuntimeError(
        "The unclipped feature table contains duplicate household IDs."
    )

for feature in TEMPORAL_SHAPE_FEATURES + ["log_daily_kwh"]:
    raw_feature_df[feature] = pd.to_numeric(
        raw_feature_df[feature],
        errors="coerce",
    )

if raw_feature_df[TEMPORAL_SHAPE_FEATURES].isna().any().any():
    raise RuntimeError(
        "The unclipped temporal features contain missing values."
    )

if not np.isfinite(
    raw_feature_df[TEMPORAL_SHAPE_FEATURES].to_numpy()
).all():
    raise RuntimeError(
        "The unclipped temporal features contain non-finite values."
    )

raw_locked_id_match = (
    set(raw_feature_df["household_ID"])
    == set(cluster_df["household_ID"])
)

if not raw_locked_id_match:
    missing_from_raw = sorted(
        set(cluster_df["household_ID"])
        - set(raw_feature_df["household_ID"])
    )
    extra_in_raw = sorted(
        set(raw_feature_df["household_ID"])
        - set(cluster_df["household_ID"])
    )
    raise RuntimeError(
        "The unclipped and locked-cluster household populations differ. "
        f"Missing from raw: {missing_from_raw[:10]}; "
        f"extra in raw: {extra_in_raw[:10]}."
    )

if len(raw_feature_df) != EXPECTED_LOCKED_HOUSEHOLDS:
    raise RuntimeError(
        f"Found {len(raw_feature_df):,} unclipped feature rows; "
        f"expected {EXPECTED_LOCKED_HOUSEHOLDS:,}."
    )

forbidden_raw_columns = [
    column
    for column in raw_feature_df.columns
    if any(
        term in column.lower()
        for term in [
            "final_cluster",
            "cluster_name",
            "archetype",
            "emission",
            "carbon",
            "high_emission",
            "target",
        ]
    )
]

if forbidden_raw_columns:
    raise RuntimeError(
        "The raw feature export contains forbidden label/target columns: "
        + ", ".join(forbidden_raw_columns)
    )

raw_feature_audit = pd.DataFrame([{
    "households": len(raw_feature_df),
    "household_ids_match_locked_population": raw_locked_id_match,
    "feature_source": "pre-clipping thesis feature export",
    "population_percentile_clipping_reused_for_ml": False,
    "supervised_features": ", ".join(TEMPORAL_SHAPE_FEATURES),
}])

display(raw_feature_audit)
print("Unclipped behavioural-feature audit: PASS")


In [ ]:
# ============================================================
# STEP 4 — CALENDAR-MONTH ALLOCATION FUNCTION + SYNTHETIC TESTS
# ============================================================

def reconstruct_sixhour_months(
    interval_frame,
    cross_month_max_gap_hours,
):
    """
    Reconstruct calendar-month imported electricity from cumulative registers.

    Allocated segment times measure represented calendar hours. Actual meter
    reading endpoints are retained separately. Month-boundary quality is based
    on the nearest real reading to each boundary, not on a synthetic midnight
    split created during cross-month allocation.
    """

    working = interval_frame.copy()

    required = {
        "household_ID",
        "previous_datetime",
        "reading_datetime",
        "gap_hours",
        "total_import_valid_delta_kwh",
        "_interval_id",
    }

    missing = required.difference(working.columns)

    if missing:
        raise ValueError(
            "Reconstruction input is missing: "
            + ", ".join(sorted(missing))
        )

    working["_previous_month"] = (
        working["previous_datetime"].dt.to_period("M")
    )
    working["_current_month"] = (
        working["reading_datetime"].dt.to_period("M")
    )

    previous_index = (
        working["previous_datetime"].dt.year * 12
        + working["previous_datetime"].dt.month
    )

    current_index = (
        working["reading_datetime"].dt.year * 12
        + working["reading_datetime"].dt.month
    )

    same_month = (
        working["_previous_month"]
        == working["_current_month"]
    )

    adjacent_month = (
        current_index - previous_index
    ).eq(1)

    valid_base = (
        working["previous_datetime"].notna()
        & working["reading_datetime"].notna()
        & working["gap_hours"].gt(0)
        & working["total_import_valid_delta_kwh"].notna()
    )

    same = working.loc[
        valid_base & same_month,
        [
            "household_ID",
            "previous_datetime",
            "reading_datetime",
            "gap_hours",
            "total_import_valid_delta_kwh",
            "_interval_id",
            "_current_month",
        ],
    ].copy()

    same["month_period"] = same["_current_month"]
    same["allocated_import_kwh"] = (
        same["total_import_valid_delta_kwh"]
    )
    same["allocated_hours"] = same["gap_hours"]
    same["segment_start"] = same["previous_datetime"]
    same["segment_end"] = same["reading_datetime"]
    same["actual_reading_time_1"] = (
        same["previous_datetime"]
    )
    same["actual_reading_time_2"] = (
        same["reading_datetime"]
    )
    same["allocation_type"] = "same_month"

    short_adjacent_cross = (
        valid_base
        & ~same_month
        & adjacent_month
        & working["gap_hours"].le(
            cross_month_max_gap_hours
        )
    )

    cross = working.loc[
        short_adjacent_cross,
        [
            "household_ID",
            "previous_datetime",
            "reading_datetime",
            "gap_hours",
            "total_import_valid_delta_kwh",
            "_interval_id",
            "_previous_month",
            "_current_month",
        ],
    ].copy()

    boundary = (
        cross["reading_datetime"]
        .dt.to_period("M")
        .dt.start_time
    )

    cross["hours_previous_month"] = (
        boundary
        - cross["previous_datetime"]
    ).dt.total_seconds() / 3600.0

    cross["hours_current_month"] = (
        cross["reading_datetime"]
        - boundary
    ).dt.total_seconds() / 3600.0

    cross["hours_total"] = (
        cross["hours_previous_month"]
        + cross["hours_current_month"]
    )

    cross = cross.loc[
        cross["hours_total"].gt(0)
        & cross["hours_previous_month"].ge(0)
        & cross["hours_current_month"].ge(0)
    ].copy()

    previous_rows = cross.copy()
    previous_rows["month_period"] = (
        previous_rows["_previous_month"]
    )
    previous_rows["allocated_hours"] = (
        previous_rows["hours_previous_month"]
    )
    previous_rows["allocation_weight"] = (
        previous_rows["hours_previous_month"]
        / previous_rows["hours_total"]
    )
    previous_rows["allocated_import_kwh"] = (
        previous_rows["total_import_valid_delta_kwh"]
        * previous_rows["allocation_weight"]
    )
    previous_rows["segment_start"] = (
        previous_rows["previous_datetime"]
    )
    previous_rows["segment_end"] = boundary.loc[
        previous_rows.index
    ]
    previous_rows["actual_reading_time_1"] = (
        previous_rows["previous_datetime"]
    )
    previous_rows["actual_reading_time_2"] = (
        previous_rows["reading_datetime"]
    )
    previous_rows["allocation_type"] = (
        "cross_month_previous"
    )

    current_rows = cross.copy()
    current_rows["month_period"] = (
        current_rows["_current_month"]
    )
    current_rows["allocated_hours"] = (
        current_rows["hours_current_month"]
    )
    current_rows["allocation_weight"] = (
        current_rows["hours_current_month"]
        / current_rows["hours_total"]
    )
    current_rows["allocated_import_kwh"] = (
        current_rows["total_import_valid_delta_kwh"]
        * current_rows["allocation_weight"]
    )
    current_rows["segment_start"] = boundary.loc[
        current_rows.index
    ]
    current_rows["segment_end"] = (
        current_rows["reading_datetime"]
    )
    current_rows["actual_reading_time_1"] = (
        current_rows["previous_datetime"]
    )
    current_rows["actual_reading_time_2"] = (
        current_rows["reading_datetime"]
    )
    current_rows["allocation_type"] = (
        "cross_month_current"
    )

    segment_columns = [
        "household_ID",
        "month_period",
        "allocated_import_kwh",
        "allocated_hours",
        "segment_start",
        "segment_end",
        "actual_reading_time_1",
        "actual_reading_time_2",
        "_interval_id",
        "allocation_type",
    ]

    segments = pd.concat(
        [
            same[segment_columns],
            previous_rows[segment_columns],
            current_rows[segment_columns],
        ],
        ignore_index=True,
    )

    segments = segments.loc[
        segments["allocated_hours"].gt(0)
        & segments["allocated_import_kwh"].ge(0)
        & segments["segment_start"].notna()
        & segments["segment_end"].notna()
        & segments["actual_reading_time_1"].notna()
        & segments["actual_reading_time_2"].notna()
    ].copy()

    monthly = (
        segments.groupby(
            ["household_ID", "month_period"],
            as_index=False,
            observed=True,
        )
        .agg(
            reconstructed_kwh=(
                "allocated_import_kwh",
                "sum",
            ),
            covered_hours=(
                "allocated_hours",
                "sum",
            ),
            first_covered_time=(
                "segment_start",
                "min",
            ),
            last_covered_time=(
                "segment_end",
                "max",
            ),
            contributing_intervals=(
                "_interval_id",
                "nunique",
            ),
        )
    )

    monthly["month_start_time"] = (
        monthly["month_period"].dt.start_time
    )

    monthly["month_end_time"] = (
        monthly["month_period"] + 1
    ).dt.start_time

    monthly["calendar_hours"] = (
        monthly["month_period"].dt.days_in_month
        * 24.0
    )

    monthly["coverage_fraction"] = (
        monthly["covered_hours"]
        / monthly["calendar_hours"]
    )

    if monthly["coverage_fraction"].gt(1.001).any():
        problem = monthly.loc[
            monthly["coverage_fraction"].gt(1.001),
            [
                "household_ID",
                "month_period",
                "coverage_fraction",
            ],
        ].head()

        display(problem)

        raise RuntimeError(
            "Monthly represented coverage exceeded 100%. "
            "Duplicate or overlapping intervals remain."
        )

    monthly["coverage_fraction"] = (
        monthly["coverage_fraction"].clip(
            lower=0.0,
            upper=1.0,
        )
    )

    # Every real endpoint contributing to a month is a candidate anchor.
    reading_candidates = pd.concat(
        [
            segments[
                [
                    "household_ID",
                    "month_period",
                    "actual_reading_time_1",
                ]
            ].rename(
                columns={
                    "actual_reading_time_1": (
                        "actual_reading_time"
                    )
                }
            ),
            segments[
                [
                    "household_ID",
                    "month_period",
                    "actual_reading_time_2",
                ]
            ].rename(
                columns={
                    "actual_reading_time_2": (
                        "actual_reading_time"
                    )
                }
            ),
        ],
        ignore_index=True,
    ).drop_duplicates()

    reading_candidates = reading_candidates.merge(
        monthly[
            [
                "household_ID",
                "month_period",
                "month_start_time",
                "month_end_time",
            ]
        ],
        on=["household_ID", "month_period"],
        how="inner",
        validate="many_to_one",
    )

    reading_candidates[
        "start_boundary_gap_hours"
    ] = (
        (
            reading_candidates["actual_reading_time"]
            - reading_candidates["month_start_time"]
        )
        .abs()
        .dt.total_seconds()
        .div(3600.0)
    )

    reading_candidates[
        "end_boundary_gap_hours"
    ] = (
        (
            reading_candidates["actual_reading_time"]
            - reading_candidates["month_end_time"]
        )
        .abs()
        .dt.total_seconds()
        .div(3600.0)
    )

    group_keys = [
        "household_ID",
        "month_period",
    ]

    nearest_start_indices = (
        reading_candidates.groupby(
            group_keys,
            observed=True,
        )["start_boundary_gap_hours"].idxmin()
    )

    nearest_end_indices = (
        reading_candidates.groupby(
            group_keys,
            observed=True,
        )["end_boundary_gap_hours"].idxmin()
    )

    nearest_start = (
        reading_candidates.loc[
            nearest_start_indices,
            group_keys
            + [
                "actual_reading_time",
                "start_boundary_gap_hours",
            ],
        ]
        .rename(
            columns={
                "actual_reading_time": (
                    "nearest_start_actual_reading_time"
                )
            }
        )
    )

    nearest_end = (
        reading_candidates.loc[
            nearest_end_indices,
            group_keys
            + [
                "actual_reading_time",
                "end_boundary_gap_hours",
            ],
        ]
        .rename(
            columns={
                "actual_reading_time": (
                    "nearest_end_actual_reading_time"
                )
            }
        )
    )

    monthly = (
        monthly.merge(
            nearest_start,
            on=group_keys,
            how="left",
            validate="one_to_one",
        )
        .merge(
            nearest_end,
            on=group_keys,
            how="left",
            validate="one_to_one",
        )
    )

    monthly["coverage_pass"] = (
        monthly["coverage_fraction"].ge(
            MIN_MONTH_COVERAGE
        )
    )

    monthly["start_boundary_pass"] = (
        monthly["start_boundary_gap_hours"].le(
            MAX_BOUNDARY_GAP_HOURS
        )
    )

    monthly["end_boundary_pass"] = (
        monthly["end_boundary_gap_hours"].le(
            MAX_BOUNDARY_GAP_HOURS
        )
    )

    monthly["month_quality_pass"] = (
        monthly["coverage_pass"]
        & monthly["start_boundary_pass"]
        & monthly["end_boundary_pass"]
    )

    monthly["cross_month_max_gap_hours"] = float(
        cross_month_max_gap_hours
    )

    source_same_energy = working.loc[
        valid_base & same_month,
        "total_import_valid_delta_kwh",
    ].sum()

    source_cross_energy = working.loc[
        short_adjacent_cross,
        "total_import_valid_delta_kwh",
    ].sum()

    allocated_energy = (
        segments["allocated_import_kwh"].sum()
    )

    conservation_difference = (
        allocated_energy
        - source_same_energy
        - source_cross_energy
    )

    excluded_cross_mask = (
        valid_base
        & ~same_month
        & ~short_adjacent_cross
    )

    audit = {
        "cross_month_max_gap_hours": float(
            cross_month_max_gap_hours
        ),
        "same_month_intervals": int(
            (valid_base & same_month).sum()
        ),
        "allocated_adjacent_cross_month_intervals": int(
            short_adjacent_cross.sum()
        ),
        "excluded_cross_month_intervals": int(
            excluded_cross_mask.sum()
        ),
        "same_month_source_energy_kwh": float(
            source_same_energy
        ),
        "cross_month_source_energy_kwh": float(
            source_cross_energy
        ),
        "allocated_energy_kwh": float(
            allocated_energy
        ),
        "conservation_difference_kwh": float(
            conservation_difference
        ),
    }

    if not np.isclose(
        conservation_difference,
        0.0,
        atol=1e-6,
    ):
        raise RuntimeError(
            "Calendar-month allocation did not conserve energy."
        )

    return monthly, segments, audit


# Test 1: a complete May at 1 kWh/hour must contain 744 kWh and pass.
synthetic_times = pd.date_range(
    "2024-04-30 18:00:00",
    "2024-06-01 00:00:00",
    freq="6h",
)

synthetic = pd.DataFrame({
    "household_ID": "H_FULL",
    "previous_datetime": synthetic_times[:-1],
    "reading_datetime": synthetic_times[1:],
})

synthetic["gap_hours"] = (
    synthetic["reading_datetime"]
    - synthetic["previous_datetime"]
).dt.total_seconds() / 3600.0

synthetic["total_import_valid_delta_kwh"] = (
    synthetic["gap_hours"]
)

synthetic["_interval_id"] = np.arange(
    len(synthetic),
    dtype=np.int64,
)

(
    synthetic_monthly,
    synthetic_segments,
    synthetic_audit,
) = reconstruct_sixhour_months(
    synthetic,
    cross_month_max_gap_hours=24.0,
)

synthetic_may = synthetic_monthly.loc[
    synthetic_monthly["month_period"].eq(
        pd.Period("2024-05", freq="M")
    )
].iloc[0]

assert np.isclose(
    synthetic_may["reconstructed_kwh"],
    744.0,
)
assert np.isclose(
    synthetic_may["coverage_fraction"],
    1.0,
)
assert np.isclose(
    synthetic_may["start_boundary_gap_hours"],
    0.0,
)
assert np.isclose(
    synthetic_may["end_boundary_gap_hours"],
    0.0,
)
assert bool(
    synthetic_may["month_quality_pass"]
)
assert np.isclose(
    synthetic_audit["conservation_difference_kwh"],
    0.0,
)

# Test 2: a 24-hour cross-month gap with readings 12 hours on either
# side of midnight may conserve and allocate energy, but it must fail the
# 6.5-hour actual-reading boundary rule.
long_gap_times = list(
    pd.date_range(
        "2024-04-30 18:00:00",
        "2024-05-31 12:00:00",
        freq="6h",
    )
) + [pd.Timestamp("2024-06-01 12:00:00")]

long_gap_times = pd.DatetimeIndex(
    sorted(set(long_gap_times))
)

long_gap_test = pd.DataFrame({
    "household_ID": "H_LONG_BOUNDARY",
    "previous_datetime": long_gap_times[:-1],
    "reading_datetime": long_gap_times[1:],
})

long_gap_test["gap_hours"] = (
    long_gap_test["reading_datetime"]
    - long_gap_test["previous_datetime"]
).dt.total_seconds() / 3600.0

long_gap_test["total_import_valid_delta_kwh"] = (
    long_gap_test["gap_hours"]
)

long_gap_test["_interval_id"] = np.arange(
    len(long_gap_test),
    dtype=np.int64,
)

long_gap_monthly, _, _ = reconstruct_sixhour_months(
    long_gap_test,
    cross_month_max_gap_hours=24.0,
)

long_gap_may = long_gap_monthly.loc[
    long_gap_monthly["month_period"].eq(
        pd.Period("2024-05", freq="M")
    )
].iloc[0]

long_cross_boundary_test_passed = bool(
    np.isclose(long_gap_may["coverage_fraction"], 1.0)
    and np.isclose(
        long_gap_may["end_boundary_gap_hours"],
        12.0,
    )
    and not bool(long_gap_may["month_quality_pass"])
)

assert long_cross_boundary_test_passed

print("Full-calendar-month reconstruction tests: PASS")
display(
    synthetic_monthly.loc[
        synthetic_monthly["month_period"].eq(
            pd.Period("2024-05", freq="M")
        )
    ]
)


In [ ]:
# ============================================================
# STEP 5 — LOAD THE FIVE SIX-HOUR FILES SAFELY
# ============================================================

locked_households = set(cluster_df["household_ID"].dropna())

sixhour_parts = []
file_rows = []

for source_order, file_path in enumerate(sixhour_files):
    header = pd.read_csv(file_path, nrows=0)
    header_map = {
        str(column).strip(): column
        for column in header.columns
    }

    required_stripped = [
        "household_ID",
        "DATE",
        "TIME",
        "TOTAL_IMPORT (kWh)",
    ]

    missing = [
        column
        for column in required_stripped
        if column not in header_map
    ]

    if missing:
        raise ValueError(
            f"{file_path.name} is missing: {missing}"
        )

    actual_columns = [
        header_map[column]
        for column in required_stripped
    ]

    loaded_rows = 0
    retained_rows = 0

    for chunk in pd.read_csv(
        file_path,
        usecols=actual_columns,
        chunksize=250_000,
        low_memory=False,
    ):
        chunk.columns = chunk.columns.astype(str).str.strip()
        loaded_rows += len(chunk)

        chunk["household_ID"] = normalise_household_id(
            chunk["household_ID"]
        )

        chunk = chunk.loc[
            chunk["household_ID"].isin(locked_households)
        ].copy()

        retained_rows += len(chunk)

        datetime_text = (
            chunk["DATE"].astype("string").str.strip()
            + " "
            + chunk["TIME"].astype("string").str.strip()
        )

        # The audited source uses ISO-style dates. Parse explicit
        # representations first, then use pandas' mixed parser only for
        # genuinely mixed residual values. This avoids silent day/month
        # swapping and the per-element dateutil warning.
        chunk["reading_datetime"] = pd.NaT

        explicit_datetime_formats = [
            "%Y-%m-%d %H:%M:%S",
            "%Y-%m-%d %H:%M:%S.%f",
            "%Y-%m-%d %H:%M",
        ]

        for datetime_format in explicit_datetime_formats:
            unresolved_datetime = chunk[
                "reading_datetime"
            ].isna()

            if not unresolved_datetime.any():
                break

            chunk.loc[
                unresolved_datetime,
                "reading_datetime",
            ] = pd.to_datetime(
                datetime_text.loc[unresolved_datetime],
                format=datetime_format,
                errors="coerce",
            )

        unresolved_datetime = chunk["reading_datetime"].isna()

        if unresolved_datetime.any():
            chunk.loc[
                unresolved_datetime,
                "reading_datetime",
            ] = pd.to_datetime(
                datetime_text.loc[unresolved_datetime],
                format="mixed",
                errors="coerce",
                dayfirst=False,
            )

        chunk["total_import_register_kwh"] = pd.to_numeric(
            chunk["TOTAL_IMPORT (kWh)"]
            .astype("string")
            .str.replace(",", "", regex=False)
            .str.strip(),
            errors="coerce",
        )

        chunk["_source_order"] = source_order
        chunk["_source_file"] = file_path.name

        chunk = chunk[
            [
                "household_ID",
                "reading_datetime",
                "total_import_register_kwh",
                "_source_order",
                "_source_file",
            ]
        ]

        chunk = chunk.dropna(
            subset=[
                "household_ID",
                "reading_datetime",
            ]
        )

        sixhour_parts.append(chunk)

    file_rows.append({
        "source_file": file_path.name,
        "raw_rows": loaded_rows,
        "locked_household_rows": retained_rows,
    })

sixhour = pd.concat(
    sixhour_parts,
    ignore_index=True,
)

del sixhour_parts

sixhour_file_audit = pd.DataFrame(file_rows)
display(sixhour_file_audit)

duplicate_mask = sixhour.duplicated(
    ["household_ID", "reading_datetime"],
    keep=False,
)

duplicate_rows = sixhour.loc[duplicate_mask].copy()

if len(duplicate_rows):
    conflicting_duplicate_keys = (
        duplicate_rows.groupby(
            ["household_ID", "reading_datetime"],
            observed=True,
        )["total_import_register_kwh"]
        .nunique(dropna=True)
        .gt(1)
    )
else:
    conflicting_duplicate_keys = pd.Series(dtype=bool)

conflicting_duplicate_count = int(
    conflicting_duplicate_keys.sum()
)

if conflicting_duplicate_count > 0:
    raise RuntimeError(
        f"Found {conflicting_duplicate_count} household-timestamp keys "
        "with conflicting cumulative import values."
    )

sixhour["_register_present"] = sixhour[
    "total_import_register_kwh"
].notna().astype(int)

rows_before_deduplication = len(sixhour)

sixhour = (
    sixhour.sort_values(
        [
            "household_ID",
            "reading_datetime",
            "_register_present",
            "_source_order",
        ]
    )
    .drop_duplicates(
        ["household_ID", "reading_datetime"],
        keep="last",
    )
    .drop(columns="_register_present")
    .sort_values(
        ["household_ID", "reading_datetime"]
    )
    .reset_index(drop=True)
)

sixhour["previous_datetime"] = (
    sixhour.groupby("household_ID", observed=True)[
        "reading_datetime"
    ].shift(1)
)

sixhour["previous_import_register_kwh"] = (
    sixhour.groupby("household_ID", observed=True)[
        "total_import_register_kwh"
    ].shift(1)
)

sixhour["gap_hours"] = (
    sixhour["reading_datetime"]
    - sixhour["previous_datetime"]
).dt.total_seconds() / 3600.0

sixhour["total_import_raw_delta_kwh"] = (
    sixhour["total_import_register_kwh"]
    - sixhour["previous_import_register_kwh"]
)

sixhour["import_rate_kwh_per_hour"] = (
    sixhour["total_import_raw_delta_kwh"]
    / sixhour["gap_hours"]
)

valid_delta = (
    sixhour["gap_hours"].gt(0)
    & sixhour["total_import_raw_delta_kwh"].ge(0)
    & sixhour["import_rate_kwh_per_hour"].le(
        MAX_RATE_KWH_PER_HOUR
    )
)

sixhour["total_import_valid_delta_kwh"] = sixhour[
    "total_import_raw_delta_kwh"
].where(valid_delta)

sixhour["_interval_id"] = np.arange(
    len(sixhour),
    dtype=np.int64,
)

sixhour_quality_audit = pd.DataFrame([{
    "rows_before_deduplication": rows_before_deduplication,
    "rows_after_deduplication": len(sixhour),
    "duplicate_rows_detected": int(duplicate_mask.sum()),
    "duplicate_rows_removed": (
        rows_before_deduplication - len(sixhour)
    ),
    "conflicting_duplicate_keys": conflicting_duplicate_count,
    "households": sixhour["household_ID"].nunique(),
    "date_start": sixhour["reading_datetime"].min(),
    "date_end": sixhour["reading_datetime"].max(),
    "median_gap_hours": sixhour["gap_hours"].median(),
    "p95_gap_hours": sixhour["gap_hours"].quantile(0.95),
    "negative_or_reset_deltas": (
        sixhour["total_import_raw_delta_kwh"] < 0
    ).sum(),
    "above_rate_cap_deltas": (
        sixhour["import_rate_kwh_per_hour"]
        > MAX_RATE_KWH_PER_HOUR
    ).sum(),
    "valid_import_intervals": (
        sixhour["total_import_valid_delta_kwh"]
        .notna()
        .sum()
    ),
}])

display(sixhour_quality_audit)

if sixhour["household_ID"].nunique() < 0.95 * EXPECTED_LOCKED_HOUSEHOLDS:
    raise RuntimeError(
        "Fewer than 95% of locked households appear in the "
        "six-hour register files."
    )

print("Six-hour ingestion and differencing: PASS")


In [ ]:
# ============================================================
# STEP 6 — RECONSTRUCT MONTHLY KWH AND BUILD THE PRIMARY COHORT
# ============================================================

monthly_reconstruction_frames = []
allocation_audit_rows = []

for cross_month_maximum_gap in SENSITIVITY_MAX_GAP_HOURS:
    (
        monthly_reconstruction,
        _,
        allocation_audit,
    ) = reconstruct_sixhour_months(
        sixhour,
        cross_month_max_gap_hours=cross_month_maximum_gap,
    )

    monthly_reconstruction = (
        monthly_reconstruction.loc[
            monthly_reconstruction["month_period"].between(
                TARGET_START,
                TARGET_END,
            )
        ]
        .copy()
    )

    monthly_reconstruction_frames.append(monthly_reconstruction)
    allocation_audit_rows.append(allocation_audit)

sixhour_monthly_all = pd.concat(
    monthly_reconstruction_frames,
    ignore_index=True,
)

allocation_audit_table = pd.DataFrame(
    allocation_audit_rows
)

display(allocation_audit_table)

coverage_summary = (
    sixhour_monthly_all.groupby(
        [
            "cross_month_max_gap_hours",
            "month_period",
        ],
        as_index=False,
        observed=True,
    )
    .agg(
        household_months=("household_ID", "size"),
        households=("household_ID", "nunique"),
        quality_pass_household_months=(
            "month_quality_pass",
            "sum",
        ),
        median_coverage=("coverage_fraction", "median"),
        p10_coverage=(
            "coverage_fraction",
            lambda values: values.quantile(0.10),
        ),
        median_start_boundary_gap_hours=(
            "start_boundary_gap_hours",
            "median",
        ),
        median_end_boundary_gap_hours=(
            "end_boundary_gap_hours",
            "median",
        ),
    )
)

display(coverage_summary)


def build_cohort_eligibility_audit(
    eligible_month_frame,
    minimum_valid_months,
):
    """
    Build one row per household using plain month-label sets.

    Using strings here deliberately avoids pandas PeriodArray aggregation
    behaviour that can make boolean reductions fail on some pandas versions.
    """

    target_labels = {
        str(period)
        for period in TARGET_MONTHS
    }
    early_labels = {
        str(period)
        for period in EARLY_TARGET_MONTHS
    }
    late_labels = {
        str(period)
        for period in LATE_TARGET_MONTHS
    }

    rows = []

    for household_id, group in eligible_month_frame.groupby(
        "household_ID",
        observed=True,
        sort=False,
    ):
        month_labels = sorted({
            str(period)
            for period in group["month_period"].dropna()
        })
        month_set = set(month_labels)

        rows.append({
            "household_ID": household_id,
            "valid_months": len(month_set),
            "valid_month_labels": ",".join(month_labels),
            "all_months_within_target_window": (
                month_set.issubset(target_labels)
            ),
            "has_early_window_month": bool(
                month_set.intersection(early_labels)
            ),
            "has_late_window_month": bool(
                month_set.intersection(late_labels)
            ),
            "minimum_valid_months_required": int(
                minimum_valid_months
            ),
        })

    audit = pd.DataFrame(rows)

    if audit.empty:
        return pd.DataFrame(
            columns=[
                "household_ID",
                "valid_months",
                "valid_month_labels",
                "all_months_within_target_window",
                "has_early_window_month",
                "has_late_window_month",
                "minimum_valid_months_required",
                "eligible_primary_cohort",
            ]
        )

    audit["eligible_primary_cohort"] = (
        audit["valid_months"].ge(minimum_valid_months)
        & audit["all_months_within_target_window"]
        & audit["has_early_window_month"]
        & audit["has_late_window_month"]
    )

    return audit


primary_monthly_all = (
    sixhour_monthly_all.loc[
        sixhour_monthly_all[
            "cross_month_max_gap_hours"
        ].eq(float(PRIMARY_MAX_GAP_HOURS))
    ]
    .copy()
)

primary_monthly_eligible = (
    primary_monthly_all.loc[
        primary_monthly_all["month_quality_pass"]
    ]
    .copy()
)

if primary_monthly_eligible.duplicated(
    ["household_ID", "month_period"]
).any():
    raise RuntimeError(
        "The primary monthly reconstruction contains duplicate "
        "household-month rows."
    )

primary_cohort_eligibility_audit = (
    build_cohort_eligibility_audit(
        primary_monthly_eligible,
        minimum_valid_months=PRIMARY_MIN_VALID_MONTHS,
    )
)

primary_household_ids = set(
    primary_cohort_eligibility_audit.loc[
        primary_cohort_eligibility_audit[
            "eligible_primary_cohort"
        ],
        "household_ID",
    ]
)

primary_analysis_panel = (
    primary_monthly_eligible.loc[
        primary_monthly_eligible[
            "household_ID"
        ].isin(primary_household_ids)
    ]
    .copy()
    .sort_values(
        ["household_ID", "month_period"]
    )
    .reset_index(drop=True)
)

expected_primary_households = (
    primary_analysis_panel["household_ID"].nunique()
)

if expected_primary_households == 0:
    raise RuntimeError(
        "No household satisfied the primary minimum-month "
        "quality rule."
    )

primary_household_month_counts = (
    primary_analysis_panel.groupby(
        "household_ID",
        observed=True,
    )["month_period"]
    .nunique()
)

if not bool(
    np.all(
        primary_household_month_counts.to_numpy()
        >= PRIMARY_MIN_VALID_MONTHS
    )
):
    raise RuntimeError(
        "At least one selected household has fewer than the "
        "required number of valid full months."
    )

if not bool(
    np.all(
        primary_household_month_counts.to_numpy()
        <= len(TARGET_MONTHS)
    )
):
    raise RuntimeError(
        "A selected household contains more months than the "
        "target window permits."
    )

selected_eligibility = (
    primary_cohort_eligibility_audit.loc[
        primary_cohort_eligibility_audit[
            "eligible_primary_cohort"
        ]
    ]
    .copy()
)

if not bool(
    np.all(
        selected_eligibility[
            "all_months_within_target_window"
        ].to_numpy(dtype=bool)
    )
):
    raise RuntimeError(
        "The primary cohort contains a month outside "
        "May-November 2024."
    )

if not bool(
    np.all(
        selected_eligibility[
            "has_early_window_month"
        ].to_numpy(dtype=bool)
    )
    and np.all(
        selected_eligibility[
            "has_late_window_month"
        ].to_numpy(dtype=bool)
    )
):
    raise RuntimeError(
        "At least one primary household does not span both "
        "the early and later parts of the target window."
    )

if not bool(
    np.all(
        primary_analysis_panel[
            "month_quality_pass"
        ].to_numpy(dtype=bool)
    )
):
    raise RuntimeError(
        "At least one primary household-month failed the "
        "coverage or actual-boundary-reading rule."
    )

primary_month_availability_summary = (
    pd.DataFrame({
        "month_period": TARGET_MONTHS,
    })
    .merge(
        primary_monthly_eligible.groupby(
            "month_period",
            as_index=False,
            observed=True,
        )["household_ID"]
        .nunique()
        .rename(
            columns={
                "household_ID": (
                    "all_quality_pass_households"
                )
            }
        ),
        on="month_period",
        how="left",
    )
    .merge(
        primary_analysis_panel.groupby(
            "month_period",
            as_index=False,
            observed=True,
        )["household_ID"]
        .nunique()
        .rename(
            columns={
                "household_ID": (
                    "selected_cohort_households_with_month"
                )
            }
        ),
        on="month_period",
        how="left",
    )
    .fillna(0)
)

for count_column in [
    "all_quality_pass_households",
    "selected_cohort_households_with_month",
]:
    primary_month_availability_summary[count_column] = (
        primary_month_availability_summary[
            count_column
        ].astype(int)
    )

november_primary = (
    primary_analysis_panel.loc[
        primary_analysis_panel[
            "month_period"
        ].eq(TARGET_END)
    ]
    .copy()
)

if len(november_primary):
    november_availability_audit = pd.DataFrame([{
        "month": str(TARGET_END),
        "selected_cohort_households_with_valid_november": (
            november_primary["household_ID"].nunique()
        ),
        "selected_primary_households": (
            expected_primary_households
        ),
        "availability_rate": (
            november_primary["household_ID"].nunique()
            / expected_primary_households
        ),
        "calendar_hours": int(
            november_primary["calendar_hours"].iloc[0]
        ),
        "minimum_coverage_fraction": (
            november_primary["coverage_fraction"].min()
        ),
        "median_coverage_fraction": (
            november_primary["coverage_fraction"].median()
        ),
        "maximum_start_boundary_gap_hours": (
            november_primary[
                "start_boundary_gap_hours"
            ].max()
        ),
        "maximum_end_boundary_gap_hours": (
            november_primary[
                "end_boundary_gap_hours"
            ].max()
        ),
        "all_available_november_rows_pass_quality": bool(
            november_primary[
                "month_quality_pass"
            ].all()
        ),
    }])
else:
    november_availability_audit = pd.DataFrame([{
        "month": str(TARGET_END),
        "selected_cohort_households_with_valid_november": 0,
        "selected_primary_households": (
            expected_primary_households
        ),
        "availability_rate": 0.0,
        "calendar_hours": 30 * 24,
        "minimum_coverage_fraction": np.nan,
        "median_coverage_fraction": np.nan,
        "maximum_start_boundary_gap_hours": np.nan,
        "maximum_end_boundary_gap_hours": np.nan,
        "all_available_november_rows_pass_quality": False,
    }])

primary_panel_audit = pd.DataFrame([{
    "adjacent_cross_month_max_gap_hours": (
        PRIMARY_MAX_GAP_HOURS
    ),
    "minimum_month_coverage": (
        MIN_MONTH_COVERAGE
    ),
    "maximum_start_end_boundary_gap_hours": (
        MAX_BOUNDARY_GAP_HOURS
    ),
    "target_window_months": len(TARGET_MONTHS),
    "minimum_valid_months_required": (
        PRIMARY_MIN_VALID_MONTHS
    ),
    "requires_early_and_late_window_representation": True,
    "eligible_households": expected_primary_households,
    "household_month_rows": len(
        primary_analysis_panel
    ),
    "minimum_observed_months": int(
        primary_household_month_counts.min()
    ),
    "median_observed_months": float(
        primary_household_month_counts.median()
    ),
    "maximum_observed_months": int(
        primary_household_month_counts.max()
    ),
    "minimum_observed_coverage": (
        primary_analysis_panel[
            "coverage_fraction"
        ].min()
    ),
    "maximum_observed_start_boundary_gap_hours": (
        primary_analysis_panel[
            "start_boundary_gap_hours"
        ].max()
    ),
    "maximum_observed_end_boundary_gap_hours": (
        primary_analysis_panel[
            "end_boundary_gap_hours"
        ].max()
    ),
}])

display(primary_panel_audit)
display(primary_month_availability_summary)
display(november_availability_audit)

if expected_primary_households < MIN_PRIMARY_HOUSEHOLDS:
    raise RuntimeError(
        "The minimum-five-valid-month carbon cohort contains "
        f"{expected_primary_households} households, below the "
        f"prespecified minimum of {MIN_PRIMARY_HOUSEHOLDS}. "
        "Do not weaken the quality rules solely to obtain a "
        "larger modelling sample."
    )

print(
    "May-November minimum-five-valid-full-month "
    "reconstruction cohort: PASS"
)


In [ ]:
# ============================================================
# STEP 7 — VALIDATE SIX-HOUR RECONSTRUCTION AGAINST BILLING
# ============================================================

monthly_billing_raw = pd.read_csv(
    monthly_billing_path,
    low_memory=False,
)

monthly_billing_raw.columns = (
    monthly_billing_raw.columns.astype(str).str.strip()
)

required_billing_columns = {
    "household_ID",
    "month",
    "consumption",
}

missing_billing_columns = required_billing_columns.difference(
    monthly_billing_raw.columns
)

if missing_billing_columns:
    raise ValueError(
        "Monthly billing file is missing: "
        + ", ".join(sorted(missing_billing_columns))
    )

monthly_billing = monthly_billing_raw.copy()

monthly_billing["household_ID"] = normalise_household_id(
    monthly_billing["household_ID"]
)

monthly_billing["month_period"] = pd.to_datetime(
    monthly_billing["month"],
    errors="coerce",
).dt.to_period("M")

monthly_billing["consumption_kwh"] = pd.to_numeric(
    monthly_billing["consumption"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce",
)

billing_duplicates = monthly_billing.loc[
    monthly_billing.duplicated(
        ["household_ID", "month_period"],
        keep=False,
    )
].copy()

if len(billing_duplicates):
    billing_conflicts = (
        billing_duplicates.groupby(
            ["household_ID", "month_period"],
            observed=True,
        )["consumption_kwh"]
        .nunique(dropna=True)
        .gt(1)
    )
else:
    billing_conflicts = pd.Series(dtype=bool)

if int(billing_conflicts.sum()) > 0:
    raise RuntimeError(
        "Conflicting duplicate monthly billing values were found."
    )

monthly_billing = (
    monthly_billing.drop_duplicates(
        [
            "household_ID",
            "month_period",
            "consumption_kwh",
        ]
    )
    .dropna(
        subset=[
            "household_ID",
            "month_period",
            "consumption_kwh",
        ]
    )
    .loc[
        lambda frame: frame["consumption_kwh"].ge(0)
    ]
    .copy()
)

billing_validation_base = monthly_billing.loc[
    monthly_billing["month_period"].between(
        BILLING_VALIDATION_START,
        BILLING_VALIDATION_END,
    ),
    [
        "household_ID",
        "month_period",
        "consumption_kwh",
    ],
].copy()


def validate_reconstruction(monthly_reconstruction):
    eligible = monthly_reconstruction.loc[
        monthly_reconstruction["month_quality_pass"]
    ].copy()

    validation = billing_validation_base.merge(
        eligible,
        on=["household_ID", "month_period"],
        how="inner",
        validate="one_to_one",
    )

    validation["signed_error_kwh"] = (
        validation["reconstructed_kwh"]
        - validation["consumption_kwh"]
    )

    validation["absolute_error_kwh"] = (
        validation["signed_error_kwh"].abs()
    )

    validation["ape"] = np.where(
        validation["consumption_kwh"] > 0,
        validation["absolute_error_kwh"]
        / validation["consumption_kwh"],
        np.nan,
    )

    if (
        len(validation) >= 10
        and validation["consumption_kwh"].nunique() > 1
        and validation["reconstructed_kwh"].nunique() > 1
    ):
        pearson_r, pearson_p = pearsonr(
            validation["consumption_kwh"],
            validation["reconstructed_kwh"],
        )

        spearman_r, spearman_p = spearmanr(
            validation["consumption_kwh"],
            validation["reconstructed_kwh"],
        )
    else:
        pearson_r = pearson_p = np.nan
        spearman_r = spearman_p = np.nan

    summary = {
        "cross_month_max_gap_hours": float(
            monthly_reconstruction[
                "cross_month_max_gap_hours"
            ].iloc[0]
        ),
        "minimum_coverage": MIN_MONTH_COVERAGE,
        "matched_household_months": len(validation),
        "matched_households": validation[
            "household_ID"
        ].nunique(),
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_r": spearman_r,
        "spearman_p": spearman_p,
        "mae_kwh": validation[
            "absolute_error_kwh"
        ].mean(),
        "median_absolute_error_kwh": validation[
            "absolute_error_kwh"
        ].median(),
        "median_ape": validation["ape"].median(),
        "mean_ape": validation["ape"].mean(),
        "mean_bias_kwh": validation[
            "signed_error_kwh"
        ].mean(),
        "median_coverage": validation[
            "coverage_fraction"
        ].median(),
    }

    return validation, summary


required_reconstruction_columns = {
    "household_ID",
    "month_period",
    "reconstructed_kwh",
    "coverage_fraction",
    "month_quality_pass",
    "cross_month_max_gap_hours",
}

missing_reconstruction_columns = (
    required_reconstruction_columns.difference(
        sixhour_monthly_all.columns
    )
)

if missing_reconstruction_columns:
    raise RuntimeError(
        "Six-hour monthly reconstruction is missing: "
        + ", ".join(sorted(missing_reconstruction_columns))
    )

available_gap_rules = set(
    pd.to_numeric(
        sixhour_monthly_all[
            "cross_month_max_gap_hours"
        ],
        errors="coerce",
    ).dropna()
)

missing_gap_rules = (
    set(float(value) for value in SENSITIVITY_MAX_GAP_HOURS)
    - available_gap_rules
)

if missing_gap_rules:
    raise RuntimeError(
        "Monthly reconstruction is unavailable for gap rule(s): "
        + ", ".join(
            str(value)
            for value in sorted(missing_gap_rules)
        )
    )

validation_frames = []
validation_summary_rows = []

for maximum_gap in SENSITIVITY_MAX_GAP_HOURS:
    reconstruction = sixhour_monthly_all.loc[
        sixhour_monthly_all[
            "cross_month_max_gap_hours"
        ].eq(
            maximum_gap
        )
    ].copy()

    validation_rows, validation_summary = (
        validate_reconstruction(reconstruction)
    )

    validation_frames.append(validation_rows)
    validation_summary_rows.append(validation_summary)

billing_validation_all = pd.concat(
    validation_frames,
    ignore_index=True,
)

billing_validation_summary = pd.DataFrame(
    validation_summary_rows
)

display(billing_validation_summary.round(4))

primary_validation_summary = billing_validation_summary.loc[
    billing_validation_summary[
        "cross_month_max_gap_hours"
    ].eq(
        PRIMARY_MAX_GAP_HOURS
    )
].iloc[0]

primary_validation_pass = (
    primary_validation_summary["matched_household_months"]
    >= MIN_VALIDATION_HOUSEHOLD_MONTHS
    and primary_validation_summary["spearman_r"]
    >= MIN_VALIDATION_SPEARMAN
    and primary_validation_summary["median_ape"]
    <= MAX_VALIDATION_MEDIAN_APE
)

validation_gate = pd.DataFrame([
    {
        "check": "matched household-months",
        "required": MIN_VALIDATION_HOUSEHOLD_MONTHS,
        "observed": primary_validation_summary[
            "matched_household_months"
        ],
        "passed": (
            primary_validation_summary[
                "matched_household_months"
            ]
            >= MIN_VALIDATION_HOUSEHOLD_MONTHS
        ),
    },
    {
        "check": "Spearman correlation",
        "required": MIN_VALIDATION_SPEARMAN,
        "observed": primary_validation_summary["spearman_r"],
        "passed": (
            primary_validation_summary["spearman_r"]
            >= MIN_VALIDATION_SPEARMAN
        ),
    },
    {
        "check": "median absolute percentage error",
        "required": MAX_VALIDATION_MEDIAN_APE,
        "observed": primary_validation_summary["median_ape"],
        "passed": (
            primary_validation_summary["median_ape"]
            <= MAX_VALIDATION_MEDIAN_APE
        ),
    },
])

display(validation_gate)

if not primary_validation_pass:
    raise RuntimeError(
        "The prespecified six-hour reconstruction failed billing "
        "validation. It must not be used as the primary carbon target."
    )

print("Six-hour reconstruction versus billing validation: PASS")

In [ ]:
# ============================================================
# EXPORT STRICT SIX-HOUR MONTHLY RECONSTRUCTION
# FOR THE SEPARATE MONTHLY-VALIDATION NOTEBOOK
# ============================================================

from pathlib import Path
import numpy as np

required_columns = {
    "household_ID",
    "month_period",
    "reconstructed_kwh",
    "coverage_fraction",
    "start_boundary_gap_hours",
    "end_boundary_gap_hours",
    "month_quality_pass",
    "cross_month_max_gap_hours",
}

missing_columns = required_columns.difference(
    primary_monthly_all.columns
)

if missing_columns:
    raise RuntimeError(
        "primary_monthly_all is missing required columns: "
        + ", ".join(sorted(missing_columns))
    )

monthly_reconstruction_export = primary_monthly_all.copy()

# Confirm that this is the locked 48-hour reconstruction.
if not np.isclose(
    monthly_reconstruction_export[
        "cross_month_max_gap_hours"
    ],
    float(PRIMARY_MAX_GAP_HOURS),
    rtol=0.0,
    atol=1e-9,
).all():
    raise RuntimeError(
        "The export contains a reconstruction rule other than "
        f"{PRIMARY_MAX_GAP_HOURS} hours."
    )

if monthly_reconstruction_export.duplicated(
    ["household_ID", "month_period"]
).any():
    raise RuntimeError(
        "Duplicate household-month rows exist in the "
        "reconstruction export."
    )

export_path = (
    PROJECT_ROOT
    / "sixhour_monthly_reconstruction_for_monthly_validation.csv"
)

monthly_reconstruction_export_for_csv = (
    monthly_reconstruction_export.copy()
)

monthly_reconstruction_export_for_csv["month_period"] = (
    monthly_reconstruction_export_for_csv[
        "month_period"
    ].astype(str)
)

monthly_reconstruction_export_for_csv.to_csv(
    export_path,
    index=False,
)

print("Saved:", export_path.resolve())
print(
    "Rows:",
    f"{len(monthly_reconstruction_export_for_csv):,}",
)
print(
    "Households:",
    f"{monthly_reconstruction_export_for_csv['household_ID'].nunique():,}",
)
print(
    "Quality-passing rows:",
    f"{monthly_reconstruction_export_for_csv['month_quality_pass'].sum():,}",
)

In [ ]:
# ============================================================
# STEP 8 — CALCULATE MAY–NOVEMBER ELECTRICITY-RELATED CO2
# ============================================================

factor_audit = pd.DataFrame([
    {
        "factor_name": "SLSEA published electricity factor",
        "factor_kg_co2_per_kwh": GRID_FACTOR_KG_CO2_PER_KWH,
        "source_status": (
            "Official Sri Lankan source; reference year not stated"
        ),
        "used_for_primary_results": True,
    }
])

display(factor_audit)

carbon_monthly = primary_analysis_panel.copy()

carbon_monthly["emissions_kg_co2"] = (
    carbon_monthly["reconstructed_kwh"]
    * GRID_FACTOR_KG_CO2_PER_KWH
)

household_carbon = (
    carbon_monthly.groupby(
        "household_ID",
        as_index=False,
        observed=True,
    )
    .agg(
        observed_months=("month_period", "nunique"),
        first_valid_month=("month_period", "min"),
        last_valid_month=("month_period", "max"),
        valid_month_labels=(
            "month_period",
            lambda values: ",".join(
                sorted({
                    str(value)
                    for value in values.dropna()
                })
            ),
        ),
        mean_monthly_kwh=("reconstructed_kwh", "mean"),
        median_monthly_kwh=("reconstructed_kwh", "median"),
        total_observed_kwh=("reconstructed_kwh", "sum"),
        mean_monthly_emissions_kg=(
            "emissions_kg_co2",
            "mean",
        ),
        median_monthly_emissions_kg=(
            "emissions_kg_co2",
            "median",
        ),
        total_observed_emissions_kg=(
            "emissions_kg_co2",
            "sum",
        ),
        minimum_month_coverage=(
            "coverage_fraction",
            "min",
        ),
        median_month_coverage=(
            "coverage_fraction",
            "median",
        ),
    )
)

if not bool(
    np.all(
        household_carbon[
            "observed_months"
        ].to_numpy()
        >= PRIMARY_MIN_VALID_MONTHS
    )
):
    raise RuntimeError(
        "The carbon table contains a household with fewer "
        "than the required valid months."
    )

if not bool(
    np.all(
        household_carbon[
            "observed_months"
        ].to_numpy()
        <= len(TARGET_MONTHS)
    )
):
    raise RuntimeError(
        "The carbon table contains an impossible month count."
    )

if not np.allclose(
    household_carbon["mean_monthly_emissions_kg"],
    household_carbon["mean_monthly_kwh"]
    * GRID_FACTOR_KG_CO2_PER_KWH,
):
    raise RuntimeError("The carbon calculation is inconsistent.")

kwh_rank = household_carbon[
    "mean_monthly_kwh"
].rank(method="average")

emissions_rank = household_carbon[
    "mean_monthly_emissions_kg"
].rank(method="average")

factor_rank_invariance = bool(
    kwh_rank.equals(emissions_rank)
)

if not factor_rank_invariance:
    raise RuntimeError(
        "A positive common factor unexpectedly changed "
        "household ranks."
    )

if (
    household_carbon["observed_months"].nunique() > 1
    and household_carbon["mean_monthly_kwh"].nunique() > 1
):
    month_count_spearman_r, month_count_spearman_p = (
        spearmanr(
            household_carbon["observed_months"],
            household_carbon["mean_monthly_kwh"],
        )
    )
else:
    month_count_spearman_r = np.nan
    month_count_spearman_p = np.nan

household_month_count_target_audit = pd.DataFrame([{
    "households": len(household_carbon),
    "minimum_valid_months_required": (
        PRIMARY_MIN_VALID_MONTHS
    ),
    "minimum_observed_months": int(
        household_carbon["observed_months"].min()
    ),
    "median_observed_months": float(
        household_carbon["observed_months"].median()
    ),
    "maximum_observed_months": int(
        household_carbon["observed_months"].max()
    ),
    "spearman_observed_months_vs_mean_monthly_kwh": (
        month_count_spearman_r
    ),
    "spearman_p_value": month_count_spearman_p,
    "interpretation": (
        "Observed-month count is audited as a possible "
        "availability-related influence and is not used as a "
        "model predictor."
    ),
}])

display(household_carbon.head())
display(household_month_count_target_audit.round(4))

print(
    "Direct carbon calculation using 0.71 kg CO2/kWh: PASS"
)


In [ ]:
# ============================================================
# STEP 9 — ARCHETYPE COMPARISON AND COHORT-SELECTION AUDIT
# ============================================================

carbon_with_archetype = household_carbon.merge(
    cluster_df[
        [
            "household_ID",
            "final_cluster",
            "cluster_name",
        ]
    ],
    on="household_ID",
    how="left",
    validate="one_to_one",
)

if carbon_with_archetype["cluster_name"].isna().any():
    raise RuntimeError(
        "Some carbon households do not have a locked archetype."
    )

archetype_carbon_summary = (
    carbon_with_archetype.groupby(
        "cluster_name",
        observed=True,
    )
    .agg(
        households=("household_ID", "count"),
        median_monthly_kwh=("mean_monthly_kwh", "median"),
        mean_monthly_kwh=("mean_monthly_kwh", "mean"),
        median_monthly_emissions_kg=(
            "mean_monthly_emissions_kg",
            "median",
        ),
        mean_monthly_emissions_kg=(
            "mean_monthly_emissions_kg",
            "mean",
        ),
        median_observed_months=(
            "observed_months",
            "median",
        ),
        median_total_observed_emissions_kg=(
            "total_observed_emissions_kg",
            "median",
        ),
    )
    .reindex(ARCHETYPE_ORDER)
    .reset_index()
)

archetype_groups = [
    carbon_with_archetype.loc[
        carbon_with_archetype["cluster_name"].eq(archetype),
        "mean_monthly_kwh",
    ].dropna()
    for archetype in ARCHETYPE_ORDER
]

if any(len(group) < 5 for group in archetype_groups):
    raise RuntimeError(
        "At least one aligned archetype has fewer than five households."
    )

kw_statistic, kw_p_value = kruskal(*archetype_groups)

n_total = sum(len(group) for group in archetype_groups)
n_groups = len(archetype_groups)

epsilon_squared = max(
    0.0,
    (kw_statistic - n_groups + 1)
    / (n_total - n_groups),
)

archetype_global_test = pd.DataFrame([{
    "outcome_tested": "mean monthly reconstructed kWh",
    "H_statistic": kw_statistic,
    "p_value": kw_p_value,
    "epsilon_squared": epsilon_squared,
    "note": (
        "Testing calculated CO2 would be numerically redundant "
        "because it is a positive linear rescaling of kWh."
    ),
}])

pairwise_rows = []

for first_index, first_name in enumerate(ARCHETYPE_ORDER):
    for second_name in ARCHETYPE_ORDER[first_index + 1:]:
        first_values = carbon_with_archetype.loc[
            carbon_with_archetype["cluster_name"].eq(first_name),
            "mean_monthly_kwh",
        ].dropna()

        second_values = carbon_with_archetype.loc[
            carbon_with_archetype["cluster_name"].eq(second_name),
            "mean_monthly_kwh",
        ].dropna()

        u_statistic, p_value = mannwhitneyu(
            first_values,
            second_values,
            alternative="two-sided",
        )

        rank_biserial = (
            2.0 * u_statistic
            / (len(first_values) * len(second_values))
            - 1.0
        )

        pairwise_rows.append({
            "group_1": first_name,
            "group_2": second_name,
            "n_group_1": len(first_values),
            "n_group_2": len(second_values),
            "median_group_1_kwh": first_values.median(),
            "median_group_2_kwh": second_values.median(),
            "median_difference_group_1_minus_2_kwh": (
                first_values.median()
                - second_values.median()
            ),
            "U_statistic": u_statistic,
            "rank_biserial_group_1_vs_2": rank_biserial,
            "p_raw": p_value,
        })

pairwise_tests = (
    pd.DataFrame(pairwise_rows)
    .sort_values("p_raw")
    .reset_index(drop=True)
)

number_of_tests = len(pairwise_tests)
running_maximum = 0.0
holm_values = []

for rank, raw_p in enumerate(
    pairwise_tests["p_raw"],
    start=1,
):
    adjusted = min(
        (number_of_tests - rank + 1) * raw_p,
        1.0,
    )
    running_maximum = max(running_maximum, adjusted)
    holm_values.append(running_maximum)

pairwise_tests["p_holm"] = holm_values
pairwise_tests["significant_holm_0_05"] = (
    pairwise_tests["p_holm"] < 0.05
)

locked_counts = (
    cluster_df["cluster_name"]
    .value_counts()
    .reindex(ARCHETYPE_ORDER, fill_value=0)
)

matched_counts = (
    carbon_with_archetype["cluster_name"]
    .value_counts()
    .reindex(ARCHETYPE_ORDER, fill_value=0)
)

unmatched_counts = locked_counts - matched_counts

match_table = pd.DataFrame(
    [matched_counts.values, unmatched_counts.values],
    index=["included_carbon_cohort", "not_included"],
    columns=ARCHETYPE_ORDER,
)

zero_margin = (
    match_table.sum(axis=0).eq(0).any()
    or match_table.sum(axis=1).eq(0).any()
)

if zero_margin:
    match_chi2 = np.nan
    match_p = np.nan
    match_dof = 0
    cramers_v = np.nan
    match_test_status = (
        "Chi-square not estimable because one inclusion-status "
        "row or archetype column has a zero total."
    )
else:
    match_chi2, match_p, match_dof, _ = chi2_contingency(
        match_table
    )

    match_n = match_table.to_numpy().sum()

    cramers_v = math.sqrt(
        match_chi2
        / (
            match_n
            * min(
                match_table.shape[0] - 1,
                match_table.shape[1] - 1,
            )
        )
    )

    match_test_status = "Chi-square test completed."

match_bias_result = pd.DataFrame([{
    "locked_households": len(cluster_df),
    "included_households": len(carbon_with_archetype),
    "inclusion_rate": (
        len(carbon_with_archetype) / len(cluster_df)
    ),
    "chi_square": match_chi2,
    "degrees_of_freedom": match_dof,
    "p_value": match_p,
    "cramers_v": cramers_v,
    "test_status": match_test_status,
    "interpretation": (
        "Carbon results apply to households with at least five "
        "quality-passing full months, not automatically to all "
        "locked households."
    ),
}])

# Feature-level included-versus-excluded audit.
inclusion_audit_source = cluster_df[
    ["household_ID", "cluster_name"]
    + TEMPORAL_SHAPE_FEATURES
    + ["log_daily_kwh"]
].copy()

included_ids = set(
    carbon_with_archetype["household_ID"]
)

inclusion_audit_source["included_in_carbon_cohort"] = (
    inclusion_audit_source["household_ID"].isin(
        included_ids
    )
)

inclusion_feature_rows = []

for feature in TEMPORAL_SHAPE_FEATURES + ["log_daily_kwh"]:
    included_values = inclusion_audit_source.loc[
        inclusion_audit_source[
            "included_in_carbon_cohort"
        ],
        feature,
    ].dropna()

    excluded_values = inclusion_audit_source.loc[
        ~inclusion_audit_source[
            "included_in_carbon_cohort"
        ],
        feature,
    ].dropna()

    if len(included_values) and len(excluded_values):
        u_statistic, p_value = mannwhitneyu(
            included_values,
            excluded_values,
            alternative="two-sided",
        )

        rank_biserial = (
            2.0 * u_statistic
            / (len(included_values) * len(excluded_values))
            - 1.0
        )
    else:
        u_statistic = np.nan
        p_value = np.nan
        rank_biserial = np.nan

    inclusion_feature_rows.append({
        "feature": feature,
        "included_n": len(included_values),
        "excluded_n": len(excluded_values),
        "included_median": included_values.median(),
        "excluded_median": excluded_values.median(),
        "median_difference_included_minus_excluded": (
            included_values.median()
            - excluded_values.median()
        ),
        "U_statistic": u_statistic,
        "rank_biserial_included_vs_excluded": rank_biserial,
        "p_raw": p_value,
    })

inclusion_feature_audit = pd.DataFrame(
    inclusion_feature_rows
)

valid_inclusion_p = (
    inclusion_feature_audit["p_raw"].notna()
)

if valid_inclusion_p.any():
    ordered_index = (
        inclusion_feature_audit.loc[
            valid_inclusion_p,
            "p_raw",
        ]
        .sort_values()
        .index
    )

    running_maximum = 0.0
    adjusted_map = {}
    total_tests = len(ordered_index)

    for rank, row_index in enumerate(
        ordered_index,
        start=1,
    ):
        raw_p = inclusion_feature_audit.loc[
            row_index,
            "p_raw",
        ]
        adjusted = min(
            (total_tests - rank + 1) * raw_p,
            1.0,
        )
        running_maximum = max(
            running_maximum,
            adjusted,
        )
        adjusted_map[row_index] = running_maximum

    inclusion_feature_audit["p_holm"] = (
        inclusion_feature_audit.index.map(
            adjusted_map
        )
    )
else:
    inclusion_feature_audit["p_holm"] = np.nan

inclusion_feature_audit[
    "significant_holm_0_05"
] = (
    inclusion_feature_audit["p_holm"] < 0.05
)

display(archetype_carbon_summary.round(3))
display(archetype_global_test.round(4))
display(pairwise_tests.round(4))
display(match_table)
display(match_bias_result.round(4))
display(inclusion_feature_audit.round(4))

plt.figure(figsize=(9, 5))
plt.boxplot(
    [
        carbon_with_archetype.loc[
            carbon_with_archetype["cluster_name"].eq(archetype),
            "mean_monthly_emissions_kg",
        ].dropna()
        for archetype in ARCHETYPE_ORDER
    ],
    showfliers=False,
)
plt.xticks(
    range(1, len(ARCHETYPE_ORDER) + 1),
    ARCHETYPE_ORDER,
    rotation=20,
)
plt.ylabel("Mean monthly electricity-related CO2 (kg)")
plt.title("Eligible-month mean emissions by locked archetype")
plt.tight_layout()
plt.show()
plt.close()


In [ ]:
# ============================================================
# STEP 10 — BUILD DIRECT-TARGET-EXCLUDED PRIMARY AND SURVEY TABLES
# ============================================================

primary_model_table = carbon_with_archetype.merge(
    raw_feature_df[
        ["household_ID"] + TEMPORAL_SHAPE_FEATURES
    ],
    on="household_ID",
    how="inner",
    validate="one_to_one",
)

primary_feature_columns = TEMPORAL_SHAPE_FEATURES.copy()

FORBIDDEN_PREDICTOR_TERMS = [
    "consumption",
    "kwh",
    "emission",
    "carbon",
    "log_daily",
    "final_cluster",
    "cluster_name",
    "archetype",
    "coverage",
    "target",
    "high_emission",
]


def audit_feature_set(frame, features, name):
    leakage_features = [
        feature
        for feature in features
        if any(
            forbidden in feature.lower()
            for forbidden in FORBIDDEN_PREDICTOR_TERMS
        )
    ]

    missing_features = [
        feature
        for feature in features
        if feature not in frame.columns
    ]

    duplicate_features = sorted({
        feature
        for feature in features
        if features.count(feature) > 1
    })

    constant_features = [
        feature
        for feature in features
        if feature in frame.columns
        and frame[feature].nunique(dropna=True) <= 1
    ]

    audit = pd.DataFrame([{
        "feature_set": name,
        "households": len(frame),
        "features": len(features),
        "leakage_features": leakage_features,
        "missing_features": missing_features,
        "duplicate_features": duplicate_features,
        "constant_features": constant_features,
    }])

    failures = (
        leakage_features
        + missing_features
        + duplicate_features
        + constant_features
    )

    if failures:
        display(audit)
        raise RuntimeError(
            f"Feature audit failed for {name}: "
            + ", ".join(sorted(set(failures)))
        )

    missingness = pd.DataFrame({
        "feature": features,
        "missing_count": [
            frame[feature].isna().sum()
            for feature in features
        ],
        "missing_percent": [
            frame[feature].isna().mean() * 100
            for feature in features
        ],
    })

    return audit, missingness


primary_feature_audit, primary_feature_missingness = (
    audit_feature_set(
        primary_model_table,
        primary_feature_columns,
        "temporal_shape_only",
    )
)

display(primary_feature_audit)
display(primary_feature_missingness)

# Prespecified Wave-1 household-context variables.
#
# The original teacher code also used current-period avg_monthly_kwh,
# avg_log_daily_kwh and final_cluster. They are deliberately excluded:
# - avg_monthly_kwh and avg_log_daily_kwh directly construct or closely
#   encode the carbon target;
# - final_cluster is a derived label built from the temporal predictors
#   and is not an independent household characteristic.
TEACHER_SOCIO_FEATURES = [
    "n_members",
    "mean_age",
    "floor_area",
    "total_monthly_expenditure_of_last_month",
    "n_acs",
    "n_appliances",
    "avg_hours_home",
    "occupant_density",
]

TEACHER_VARIABLE_UNITS = {
    "n_members": "household members",
    "mean_age": "years",
    "floor_area": "square feet as recorded in Wave 1",
    "total_monthly_expenditure_of_last_month": "survey-reported monthly expenditure",
    "n_acs": "number of AC units in the Wave 1 AC roster",
    "n_appliances": "number of appliance records in the Wave 1 appliance roster",
    "avg_hours_home": "mean member hours at home during the previous week",
    "occupant_density": "household members per 1,000 square feet",
}

SURVEY_ALIAS_GROUPS = {
    "n_members": [
        "no_of_household_members",
        "household_size",
        "w1_hh_no_of_household_members",
        "w1_demo_member_count",
        "n_members",
    ],
    "mean_age": [
        "avg_member_age",
        "w1_demo_mean_age",
        "mean_age",
    ],
    "floor_area": [
        "floor_area",
        "w1_hh_floor_area",
    ],
    "total_monthly_expenditure_of_last_month": [
        "total_monthly_expenditure_of_last_month",
        "w1_hh_total_monthly_expenditure_of_last_month",
        "w1_hh_total_monthly_expenditure",
    ],
    "n_acs": [
        "ac_total_count",
        "w1_ac_total_count",
        "n_acs",
    ],
    "n_appliances": [
        "appliance_roster_count",
        "appliance_total_count",
        "w1_appliance_total_count",
        "n_appliances",
    ],
    "avg_hours_home": [
        "avg_member_hours_at_home_week",
        "w1_demo_mean_home_hours_last_week",
        "avg_hours_home",
    ],
    "occupant_density": [
        "occupant_density_per_1000_sqft",
        "occupant_density",
    ],
}


def first_present(frame, aliases):
    for alias in aliases:
        if alias in frame.columns:
            return alias
    return None


def numeric_or_binary(series):
    numeric = pd.to_numeric(series, errors="coerce")

    text = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    binary_map = {
        "yes": 1.0,
        "y": 1.0,
        "true": 1.0,
        "owned": 1.0,
        "1": 1.0,
        "no": 0.0,
        "n": 0.0,
        "false": 0.0,
        "not owned": 0.0,
        "0": 0.0,
    }

    return numeric.fillna(text.map(binary_map))


def build_teacher_features_from_raw(raw_paths):
    if not all(path is not None for path in raw_paths.values()):
        return None, pd.DataFrame()

    household = ensure_household_id(
        pd.read_csv(raw_paths["household"], low_memory=False)
    )
    demographics = ensure_household_id(
        pd.read_csv(raw_paths["demographics"], low_memory=False)
    )
    appliances = ensure_household_id(
        pd.read_csv(raw_paths["appliances"], low_memory=False)
    )
    ac_roster = ensure_household_id(
        pd.read_csv(raw_paths["ac_roster"], low_memory=False)
    )

    required_household_columns = [
        "household_ID",
        "no_of_household_members",
        "floor_area",
        "total_monthly_expenditure_of_last_month",
    ]
    missing_household_columns = [
        column
        for column in required_household_columns
        if column not in household.columns
    ]

    if missing_household_columns:
        raise KeyError(
            "Raw Wave 1 household file is missing: "
            + ", ".join(missing_household_columns)
        )

    household_features = (
        household[required_household_columns]
        .drop_duplicates("household_ID")
        .rename(columns={
            "no_of_household_members": "n_members",
        })
        .copy()
    )

    for column in [
        "n_members",
        "floor_area",
        "total_monthly_expenditure_of_last_month",
    ]:
        household_features[column] = pd.to_numeric(
            household_features[column],
            errors="coerce",
        )

    demographic_aggregations = {}

    if "age" in demographics.columns:
        demographics["age"] = pd.to_numeric(
            demographics["age"],
            errors="coerce",
        )
        demographic_aggregations["mean_age"] = ("age", "mean")

    home_hours_column = (
        "no_of_hours_stayed_at_home_during_last_week"
    )

    if home_hours_column in demographics.columns:
        demographics[home_hours_column] = pd.to_numeric(
            demographics[home_hours_column],
            errors="coerce",
        )
        demographic_aggregations["avg_hours_home"] = (
            home_hours_column,
            "mean",
        )

    if demographic_aggregations:
        demographic_features = (
            demographics.groupby("household_ID")
            .agg(**demographic_aggregations)
            .reset_index()
        )
    else:
        demographic_features = (
            demographics[["household_ID"]]
            .drop_duplicates()
        )

    appliance_count_column = (
        "appliance_ID"
        if "appliance_ID" in appliances.columns
        else None
    )
    if appliance_count_column is None:
        appliance_features = (
            appliances.groupby("household_ID")
            .size()
            .rename("n_appliances")
            .reset_index()
        )
    else:
        appliance_features = (
            appliances.groupby("household_ID")[
                appliance_count_column
            ]
            .nunique()
            .rename("n_appliances")
            .reset_index()
        )

    ac_count_column = (
        "ac_ID"
        if "ac_ID" in ac_roster.columns
        else None
    )
    if ac_count_column is None:
        ac_features = (
            ac_roster.groupby("household_ID")
            .size()
            .rename("n_acs")
            .reset_index()
        )
    else:
        ac_features = (
            ac_roster.groupby("household_ID")[
                ac_count_column
            ]
            .nunique()
            .rename("n_acs")
            .reset_index()
        )

    raw_teacher = (
        household_features
        .merge(
            demographic_features,
            on="household_ID",
            how="left",
            validate="one_to_one",
        )
        .merge(
            appliance_features,
            on="household_ID",
            how="left",
            validate="one_to_one",
        )
        .merge(
            ac_features,
            on="household_ID",
            how="left",
            validate="one_to_one",
        )
    )

    raw_teacher["n_appliances"] = (
        raw_teacher["n_appliances"].fillna(0.0)
    )
    raw_teacher["n_acs"] = (
        raw_teacher["n_acs"].fillna(0.0)
    )
    raw_teacher["occupant_density"] = np.where(
        raw_teacher["floor_area"] > 0,
        raw_teacher["n_members"]
        / raw_teacher["floor_area"]
        * 1000.0,
        np.nan,
    )

    source_audit = pd.DataFrame([
        {
            "source_type": "raw_wave1",
            "source_path": str(raw_paths["household"]),
            "mapped_variables": (
                "n_members; floor_area; "
                "total_monthly_expenditure_of_last_month; "
                "occupant_density"
            ),
            "households": len(household_features),
        },
        {
            "source_type": "raw_wave1",
            "source_path": str(raw_paths["demographics"]),
            "mapped_variables": "mean_age; avg_hours_home",
            "households": (
                demographic_features["household_ID"].nunique()
            ),
        },
        {
            "source_type": "raw_wave1",
            "source_path": str(raw_paths["appliances"]),
            "mapped_variables": "n_appliances",
            "households": (
                appliance_features["household_ID"].nunique()
            ),
        },
        {
            "source_type": "raw_wave1",
            "source_path": str(raw_paths["ac_roster"]),
            "mapped_variables": "n_acs",
            "households": (
                ac_features["household_ID"].nunique()
            ),
        },
    ])

    return raw_teacher, source_audit


teacher_source_frames = []
teacher_source_audit_rows = []
teacher_variable_source_rows = []

raw_teacher_features, raw_teacher_source_audit = (
    build_teacher_features_from_raw(wave1_raw_paths)
)

if raw_teacher_features is not None:
    teacher_source_frames.append(
        (
            "raw_wave1_reconstruction",
            raw_teacher_features[
                ["household_ID"] + TEACHER_SOCIO_FEATURES
            ].copy(),
        )
    )

    teacher_source_audit_rows.extend(
        raw_teacher_source_audit.to_dict(orient="records")
    )

for source_priority, source_path in enumerate(
    survey_paths,
    start=1,
):
    source_frame = ensure_household_id(
        load_table(source_path)
    )

    duplicate_count = int(
        source_frame["household_ID"].duplicated().sum()
    )

    if duplicate_count > 0:
        raise RuntimeError(
            f"{source_path} is not one row per household "
            f"({duplicate_count} duplicate rows)."
        )

    selected = source_frame[["household_ID"]].copy()
    mapped_variables = []

    for canonical_name, aliases in SURVEY_ALIAS_GROUPS.items():
        source_column = first_present(
            source_frame,
            aliases,
        )

        if source_column is None:
            continue

        selected[canonical_name] = numeric_or_binary(
            source_frame[source_column]
        )
        mapped_variables.append(canonical_name)
        teacher_variable_source_rows.append({
            "canonical_variable": canonical_name,
            "source_path": str(source_path),
            "source_column": source_column,
            "source_priority": source_priority,
            "non_missing_values": int(
                selected[canonical_name].notna().sum()
            ),
        })

    teacher_source_audit_rows.append({
        "source_type": "validated_output",
        "source_path": str(source_path),
        "mapped_variables": "; ".join(mapped_variables),
        "households": source_frame[
            "household_ID"
        ].nunique(),
    })

    if mapped_variables:
        teacher_source_frames.append(
            (
                str(source_path),
                selected[
                    ["household_ID"] + mapped_variables
                ].copy(),
            )
        )

if not teacher_source_frames:
    raise FileNotFoundError(
        "Wave-1 household-context household variables could not be built. "
        "Run survey_validation(3).ipynb to create "
        "survey_outputs/socio_demographic_household_dataset_FINAL.csv "
        "and run surveyanalysis_main.ipynb to create "
        "ANALYSIS2_consumption_group_profile_dataset.csv, or keep the "
        "four raw Wave 1 files under the project folder."
    )

combined_teacher = None

for source_name, source_frame in teacher_source_frames:
    indexed = source_frame.set_index("household_ID")

    if combined_teacher is None:
        combined_teacher = indexed
    else:
        combined_teacher = combined_teacher.combine_first(
            indexed
        )

survey_selected = (
    combined_teacher
    .reset_index()
)

for feature in TEACHER_SOCIO_FEATURES:
    if feature not in survey_selected.columns:
        survey_selected[feature] = np.nan

    survey_selected[feature] = pd.to_numeric(
        survey_selected[feature],
        errors="coerce",
    )

derived_density = np.where(
    survey_selected["floor_area"] > 0,
    survey_selected["n_members"]
    / survey_selected["floor_area"]
    * 1000.0,
    np.nan,
)
survey_selected["occupant_density"] = (
    survey_selected["occupant_density"]
    .fillna(
        pd.Series(
            derived_density,
            index=survey_selected.index,
        )
    )
)

if survey_selected["household_ID"].duplicated().any():
    raise RuntimeError(
        "Combined Wave-1 household-context feature table is not one row "
        "per household."
    )

teacher_source_audit = pd.DataFrame(
    teacher_source_audit_rows
)
teacher_variable_source_audit = pd.DataFrame(
    teacher_variable_source_rows
)

teacher_variable_availability = pd.DataFrame({
    "feature": TEACHER_SOCIO_FEATURES,
    "unit_or_definition": [
        TEACHER_VARIABLE_UNITS[feature]
        for feature in TEACHER_SOCIO_FEATURES
    ],
    "available_in_combined_source": [
        feature in survey_selected.columns
        and survey_selected[feature].notna().any()
        for feature in TEACHER_SOCIO_FEATURES
    ],
    "non_missing_households_before_carbon_match": [
        int(survey_selected[feature].notna().sum())
        for feature in TEACHER_SOCIO_FEATURES
    ],
})

missing_teacher_sources = teacher_variable_availability.loc[
    ~teacher_variable_availability[
        "available_in_combined_source"
    ],
    "feature",
].tolist()

if missing_teacher_sources:
    raise RuntimeError(
        "The following teacher variables are unavailable: "
        + ", ".join(missing_teacher_sources)
    )

survey_candidate_feature_columns = (
    TEACHER_SOCIO_FEATURES.copy()
)
survey_feature_columns = []
survey_model_table = primary_model_table.merge(
    survey_selected[
        ["household_ID"] + TEACHER_SOCIO_FEATURES
    ],
    on="household_ID",
    how="inner",
    validate="one_to_one",
)

if len(survey_model_table) < MIN_PRIMARY_HOUSEHOLDS:
    raise RuntimeError(
        "The Wave-1 household-context survey-matched carbon cohort contains "
        f"only {len(survey_model_table)} households, below the "
        f"prespecified minimum of {MIN_PRIMARY_HOUSEHOLDS}."
    )

print(
    "Wave-1 household-context household predictors were assembled from "
    "validated outputs and/or raw Wave 1 files."
)
display(teacher_source_audit)
display(teacher_variable_availability)
print(
    "Wave-1 household-context carbon cohort households:",
    len(survey_model_table),
)
print(
    "Direct target variables, log_daily_kwh and numeric cluster "
    "labels remain excluded."
)
print("Direct-target-excluded modelling-table construction: PASS")


In [ ]:
# ============================================================
# STEP 11 — MODEL, TUNING, METRIC AND BOOTSTRAP HELPERS
# ============================================================

def expected_calibration_error(
    y_true,
    probabilities,
    number_of_bins=10,
):
    y_array = np.asarray(y_true, dtype=int)
    p_array = np.asarray(probabilities, dtype=float)

    bin_edges = np.linspace(0.0, 1.0, number_of_bins + 1)
    bin_ids = np.digitize(
        p_array,
        bin_edges[1:-1],
        right=True,
    )

    ece = 0.0

    for bin_id in range(number_of_bins):
        mask = bin_ids == bin_id

        if not mask.any():
            continue

        observed = y_array[mask].mean()
        predicted = p_array[mask].mean()
        ece += mask.mean() * abs(observed - predicted)

    return float(ece)


def classification_metrics(
    y_true,
    probabilities,
    threshold=0.50,
):
    y_array = np.asarray(y_true, dtype=int)
    p_array = np.asarray(probabilities, dtype=float)
    predictions = (p_array >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_array,
        predictions,
        labels=[0, 1],
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    negative_predictive_value = (
        tn / (tn + fn) if (tn + fn) else np.nan
    )

    metrics = {
        "decision_threshold": float(threshold),
        "accuracy": accuracy_score(y_array, predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y_array,
            predictions,
        ),
        "macro_f1": f1_score(
            y_array,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "mcc": matthews_corrcoef(y_array, predictions),
        "high_emission_precision": precision_score(
            y_array,
            predictions,
            pos_label=1,
            zero_division=0,
        ),
        "high_emission_recall": recall_score(
            y_array,
            predictions,
            pos_label=1,
            zero_division=0,
        ),
        "high_emission_f1": f1_score(
            y_array,
            predictions,
            pos_label=1,
            zero_division=0,
        ),
        "specificity": specificity,
        "negative_predictive_value": negative_predictive_value,
        "pr_auc": average_precision_score(y_array, p_array),
        "brier_score": brier_score_loss(y_array, p_array),
        "log_loss": log_loss(
            y_array,
            np.column_stack([1.0 - p_array, p_array]),
            labels=[0, 1],
        ),
        "ece_10_bins": expected_calibration_error(
            y_array,
            p_array,
            number_of_bins=10,
        ),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }

    metrics["roc_auc"] = (
        roc_auc_score(y_array, p_array)
        if np.unique(y_array).size == 2
        else np.nan
    )

    return metrics


def regression_metrics(y_true, predictions):
    """Calculate physically valid original- and log-scale regression metrics."""
    y_array = np.asarray(y_true, dtype=float)
    prediction_array = np.asarray(predictions, dtype=float)

    if y_array.shape != prediction_array.shape:
        raise ValueError("Regression targets and predictions have unequal shapes.")
    if not np.isfinite(y_array).all() or not np.isfinite(prediction_array).all():
        raise ValueError("Regression metrics received non-finite values.")
    if (y_array < 0).any():
        raise ValueError("Electricity-related emissions cannot be negative.")

    # Negative CO2 predictions are physically impossible. This clipping rule is
    # prespecified and is applied consistently to OOF and held-out predictions.
    prediction_array = np.clip(prediction_array, 0.0, None)

    mae = mean_absolute_error(y_array, prediction_array)
    rmse = math.sqrt(mean_squared_error(y_array, prediction_array))
    medae = median_absolute_error(y_array, prediction_array)
    mean_target = np.mean(y_array)
    original_r2 = r2_score(y_array, prediction_array)
    log_r2 = r2_score(
        np.log1p(y_array),
        np.log1p(prediction_array),
    )

    return {
        "mae_kg_co2": mae,
        "rmse_kg_co2": rmse,
        "median_absolute_error_kg_co2": medae,
        # Kept for compatibility; this is always the principal original-scale R2.
        "r2": original_r2,
        "r2_original_kg_co2_scale": original_r2,
        "r2_log1p_scale": log_r2,
        "normalized_mae_percent_of_test_mean": (
            100.0 * mae / mean_target
            if mean_target > 0
            else np.nan
        ),
        "spearman_r": (
            spearmanr(y_array, prediction_array)[0]
            if (
                len(y_array) >= 3
                and np.unique(y_array).size > 1
                and np.unique(prediction_array).size > 1
            )
            else np.nan
        ),
    }


REGRESSION_SCORING_RULE = (
    "clip_predictions_at_zero_before_all_regression_scoring"
)


def clipped_regression_predictions(estimator, X):
    """Apply the notebook's prespecified physical non-negativity rule."""
    predictions = np.asarray(estimator.predict(X), dtype=float)
    if not np.isfinite(predictions).all():
        raise ValueError(
            "Regression scorer received non-finite model predictions."
        )
    return np.clip(predictions, 0.0, None)


def clipped_neg_mae_scorer(estimator, X, y_true):
    predictions = clipped_regression_predictions(estimator, X)
    return -mean_absolute_error(
        np.asarray(y_true, dtype=float),
        predictions,
    )


def clipped_neg_mse_scorer(estimator, X, y_true):
    predictions = clipped_regression_predictions(estimator, X)
    return -mean_squared_error(
        np.asarray(y_true, dtype=float),
        predictions,
    )


def clipped_r2_scorer(estimator, X, y_true):
    predictions = clipped_regression_predictions(estimator, X)
    return r2_score(
        np.asarray(y_true, dtype=float),
        predictions,
    )


def select_development_threshold(
    y_true,
    probabilities,
):
    """
    Select a decision threshold using threshold-dependent metrics only.

    PR-AUC, ROC-AUC, Brier score, log loss and calibration do not change
    when only the decision threshold changes. Recomputing them for every
    candidate is unnecessary and can make the notebook extremely slow.
    """

    y_array = np.asarray(y_true, dtype=int)
    p_array = np.asarray(probabilities, dtype=float)

    if y_array.shape[0] != p_array.shape[0]:
        raise ValueError(
            "Threshold selection received unequal target and "
            "probability lengths."
        )

    if not np.isfinite(p_array).all():
        raise ValueError(
            "Threshold selection received non-finite probabilities."
        )

    p_array = np.clip(p_array, 0.0, 1.0)

    candidate_thresholds = np.unique(
        np.concatenate([
            np.linspace(0.05, 0.95, 181),
            np.quantile(
                p_array,
                np.linspace(0.02, 0.98, 97),
            ),
            np.array([0.50]),
        ])
    )

    rows = []

    for threshold in candidate_thresholds:
        predictions = (
            p_array >= float(threshold)
        ).astype(int)

        true_negative = int(
            np.sum((y_array == 0) & (predictions == 0))
        )
        false_positive = int(
            np.sum((y_array == 0) & (predictions == 1))
        )
        false_negative = int(
            np.sum((y_array == 1) & (predictions == 0))
        )
        true_positive = int(
            np.sum((y_array == 1) & (predictions == 1))
        )

        recall = (
            true_positive
            / (true_positive + false_negative)
            if (true_positive + false_negative)
            else 0.0
        )
        specificity = (
            true_negative
            / (true_negative + false_positive)
            if (true_negative + false_positive)
            else 0.0
        )
        precision = (
            true_positive
            / (true_positive + false_positive)
            if (true_positive + false_positive)
            else 0.0
        )
        f1_value = (
            2.0 * precision * recall
            / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        rows.append({
            "threshold": float(threshold),
            "balanced_accuracy": (
                0.5 * (recall + specificity)
            ),
            "high_emission_f1": f1_value,
            "high_emission_recall": recall,
            "specificity": specificity,
        })

    threshold_table = pd.DataFrame(rows)

    selected = (
        threshold_table.sort_values(
            [
                "balanced_accuracy",
                "high_emission_f1",
                "high_emission_recall",
                "threshold",
            ],
            ascending=[False, False, False, True],
        )
        .iloc[0]
    )

    return float(selected["threshold"]), threshold_table


def build_classification_search_spaces(
    random_state=RANDOM_STATE,
):
    """Return prespecified pipelines and compact, defensible grids."""
    return {
        "Dummy prior baseline": {
            "pipeline": Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                (
                    "model",
                    DummyClassifier(strategy="prior"),
                ),
            ]),
            "param_grid": {},
        },
        "Logistic Regression": {
            "pipeline": Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                (
                    "scaler",
                    RobustScaler(
                        quantile_range=(25.0, 75.0)
                    ),
                ),
                (
                    "model",
                    LogisticRegression(
                        max_iter=5000,
                        random_state=random_state,
                    ),
                ),
            ]),
            "param_grid": {
                "model__C": [0.1, 1.0, 10.0],
                "model__class_weight": [None, "balanced"],
            },
        },
        "Random Forest": {
            "pipeline": Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=400,
                        random_state=random_state,
                        n_jobs=1,
                    ),
                ),
            ]),
            "param_grid": {
                "model__max_depth": [6, None],
                "model__min_samples_leaf": [2, 8],
                "model__max_features": ["sqrt", 1.0],
                "model__class_weight": [None, "balanced"],
            },
        },
        "Histogram Gradient Boosting": {
            "pipeline": Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                (
                    "model",
                    HistGradientBoostingClassifier(
                        max_iter=300,
                        random_state=random_state,
                    ),
                ),
            ]),
            "param_grid": {
                "model__learning_rate": [0.05, 0.10],
                "model__max_leaf_nodes": [15, 31],
                "model__min_samples_leaf": [10, 20],
                "model__l2_regularization": [1.0],
            },
        },
    }


def build_regression_search_spaces(
    random_state=RANDOM_STATE,
    add_missing_indicators=False,
    include_log_target=False,
):
    """
    Return prespecified regression pipelines.

    The Wave-1 household-context matched-cohort analysis can add missingness indicators
    and compare original versus log1p target formulations. GridSearchCV scores
    every TransformedTargetRegressor on the original kg-CO2 scale because its
    predict method automatically back-transforms predictions.
    """

    def make_pipeline(model, scale=False):
        steps = [
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=bool(add_missing_indicators),
                ),
            )
        ]
        if scale:
            steps.append((
                "scaler",
                RobustScaler(quantile_range=(25.0, 75.0)),
            ))
        steps.append(("model", model))
        return Pipeline(steps)

    base_spaces = {
        "Dummy mean baseline": {
            "pipeline": make_pipeline(DummyRegressor(strategy="mean")),
            "param_grid": {},
            "target_transform": "original",
        },
        "Ridge Regression": {
            "pipeline": make_pipeline(Ridge(), scale=True),
            "param_grid": {
                "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
            },
            "target_transform": "original",
        },
        "Random Forest Regressor": {
            "pipeline": make_pipeline(
                RandomForestRegressor(
                    n_estimators=400,
                    random_state=random_state,
                    n_jobs=1,
                )
            ),
            "param_grid": {
                "model__max_depth": [6, None],
                "model__min_samples_leaf": [2, 8],
                "model__max_features": ["sqrt", 1.0],
            },
            "target_transform": "original",
        },
        "Histogram Gradient Boosting Regressor": {
            "pipeline": make_pipeline(
                HistGradientBoostingRegressor(
                    max_iter=300,
                    random_state=random_state,
                )
            ),
            "param_grid": {
                "model__learning_rate": [0.05, 0.10],
                "model__max_leaf_nodes": [15, 31],
                "model__min_samples_leaf": [10, 20],
                "model__l2_regularization": [1.0],
                "model__loss": ["squared_error", "absolute_error"],
            },
            "target_transform": "original",
        },
    }

    if not include_log_target:
        return base_spaces

    log_spaces = {}
    for model_name, specification in base_spaces.items():
        if model_name == "Dummy mean baseline":
            continue
        log_name = model_name + " [log1p target]"
        log_spaces[log_name] = {
            "pipeline": TransformedTargetRegressor(
                regressor=specification["pipeline"],
                func=np.log1p,
                inverse_func=np.expm1,
                check_inverse=False,
            ),
            "param_grid": {
                "regressor__" + parameter: values
                for parameter, values in specification["param_grid"].items()
            },
            "target_transform": "log1p",
        }

    return {**base_spaces, **log_spaces}


def bootstrap_classification_intervals(
    y_true,
    probabilities,
    threshold,
    iterations=BOOTSTRAP_ITERATIONS,
    random_state=RANDOM_STATE,
):
    y_array = np.asarray(y_true, dtype=int)
    p_array = np.asarray(probabilities, dtype=float)
    rng = np.random.default_rng(random_state)
    rows = []

    for iteration in range(iterations):
        indices = rng.integers(
            0,
            len(y_array),
            size=len(y_array),
        )

        sampled_y = y_array[indices]
        sampled_p = p_array[indices]

        if np.unique(sampled_y).size < 2:
            continue

        rows.append({
            "iteration": iteration,
            **classification_metrics(
                sampled_y,
                sampled_p,
                threshold=threshold,
            ),
        })

    bootstrap_long = pd.DataFrame(rows)

    metric_names = [
        "accuracy",
        "balanced_accuracy",
        "macro_f1",
        "mcc",
        "high_emission_precision",
        "high_emission_recall",
        "high_emission_f1",
        "specificity",
        "pr_auc",
        "roc_auc",
        "brier_score",
        "log_loss",
        "ece_10_bins",
    ]

    interval_rows = []

    for metric in metric_names:
        values = bootstrap_long[metric].dropna()

        interval_rows.append({
            "metric": metric,
            "valid_bootstrap_samples": len(values),
            "estimate": classification_metrics(
                y_array,
                p_array,
                threshold=threshold,
            )[metric],
            "ci_2_5_percent": values.quantile(0.025),
            "ci_97_5_percent": values.quantile(0.975),
        })

    return bootstrap_long, pd.DataFrame(interval_rows)


def bootstrap_regression_intervals(
    y_true,
    predictions,
    iterations=BOOTSTRAP_ITERATIONS,
    random_state=RANDOM_STATE,
):
    y_array = np.asarray(y_true, dtype=float)
    prediction_array = np.asarray(predictions, dtype=float)
    rng = np.random.default_rng(random_state)
    rows = []

    for iteration in range(iterations):
        indices = rng.integers(
            0,
            len(y_array),
            size=len(y_array),
        )

        sampled_y = y_array[indices]
        sampled_predictions = prediction_array[indices]

        rows.append({
            "iteration": iteration,
            **regression_metrics(
                sampled_y,
                sampled_predictions,
            ),
        })

    bootstrap_long = pd.DataFrame(rows)
    point_metrics = regression_metrics(
        y_array,
        prediction_array,
    )
    interval_rows = []

    for metric in point_metrics:
        values = bootstrap_long[metric].dropna()

        interval_rows.append({
            "metric": metric,
            "valid_bootstrap_samples": len(values),
            "estimate": point_metrics[metric],
            "ci_2_5_percent": values.quantile(0.025),
            "ci_97_5_percent": values.quantile(0.975),
        })

    return bootstrap_long, pd.DataFrame(interval_rows)


print(
    "Model spaces include baselines, linear models, Random Forest "
    "and Histogram Gradient Boosting with development-only grids."
)


In [ ]:
# ============================================================
# STEP 12 — NESTED DEVELOPMENT SELECTION AND HELD-OUT EVALUATION
# ============================================================

def create_split_plan(
    modelling_table,
    cohort_name,
    random_state=RANDOM_STATE,
):
    split_source = modelling_table[
        [
            "household_ID",
            "mean_monthly_kwh",
            "mean_monthly_emissions_kg",
        ]
    ].copy()

    if split_source["household_ID"].duplicated().any():
        raise RuntimeError(
            f"{cohort_name} contains duplicate household rows."
        )

    # The final holdout is created without outcome stratification.
    development_ids, test_ids = train_test_split(
        split_source["household_ID"],
        test_size=TEST_SIZE,
        random_state=random_state,
        shuffle=True,
    )

    development_ids = development_ids.reset_index(drop=True)
    test_ids = test_ids.reset_index(drop=True)

    indexed = split_source.set_index("household_ID")
    development_emissions = indexed.loc[
        development_ids,
        "mean_monthly_emissions_kg",
    ].reset_index(drop=True)

    development_threshold_emissions = float(
        development_emissions.quantile(
            HIGH_EMISSION_QUANTILE
        )
    )

    return {
        "name": cohort_name,
        "random_state": int(random_state),
        "development_ids": development_ids,
        "test_ids": test_ids,
        "development_threshold_emissions": (
            development_threshold_emissions
        ),
        "development_threshold_kwh": (
            development_threshold_emissions
            / GRID_FACTOR_KG_CO2_PER_KWH
        ),
        "threshold_quantile": HIGH_EMISSION_QUANTILE,
        "threshold_source": "development outcomes only",
    }


def subset_split_plan(
    parent_plan,
    eligible_household_ids,
    cohort_name,
):
    eligible_ids = set(eligible_household_ids)

    development_ids = parent_plan[
        "development_ids"
    ].loc[
        parent_plan["development_ids"].isin(eligible_ids)
    ].reset_index(drop=True)

    test_ids = parent_plan[
        "test_ids"
    ].loc[
        parent_plan["test_ids"].isin(eligible_ids)
    ].reset_index(drop=True)

    return {
        "name": cohort_name,
        "random_state": parent_plan["random_state"],
        "development_ids": development_ids,
        "test_ids": test_ids,
        "development_threshold_emissions": parent_plan[
            "development_threshold_emissions"
        ],
        "development_threshold_kwh": parent_plan[
            "development_threshold_kwh"
        ],
        "threshold_quantile": parent_plan[
            "threshold_quantile"
        ],
        "threshold_source": (
            "shared primary-development threshold"
        ),
    }


def prepare_split_data(
    modelling_table,
    feature_columns,
    split_plan,
    cohort_name,
):
    selected = modelling_table[
        [
            "household_ID",
            "mean_monthly_kwh",
            "mean_monthly_emissions_kg",
        ]
        + feature_columns
    ].copy()

    if selected["household_ID"].duplicated().any():
        raise RuntimeError(
            f"{cohort_name} contains duplicate household rows."
        )

    selected = selected.set_index(
        "household_ID",
        drop=False,
    )

    development_ids = split_plan["development_ids"]
    test_ids = split_plan["test_ids"]

    missing_plan_ids = (
        set(development_ids)
        .union(set(test_ids))
        .difference(set(selected.index))
    )

    if missing_plan_ids:
        raise RuntimeError(
            f"{cohort_name} is missing split-plan households: "
            + ", ".join(sorted(missing_plan_ids)[:10])
        )

    return {
        "selected": selected,
        "X_development": selected.loc[
            development_ids,
            feature_columns,
        ].reset_index(drop=True),
        "X_test": selected.loc[
            test_ids,
            feature_columns,
        ].reset_index(drop=True),
        "kwh_development": selected.loc[
            development_ids,
            "mean_monthly_kwh",
        ].reset_index(drop=True),
        "kwh_test": selected.loc[
            test_ids,
            "mean_monthly_kwh",
        ].reset_index(drop=True),
        "emissions_development": selected.loc[
            development_ids,
            "mean_monthly_emissions_kg",
        ].reset_index(drop=True),
        "emissions_test": selected.loc[
            test_ids,
            "mean_monthly_emissions_kg",
        ].reset_index(drop=True),
        "id_development": development_ids.reset_index(
            drop=True
        ),
        "id_test": test_ids.reset_index(drop=True),
    }


def safe_classification_splits(y, requested_splits):
    minimum_class_count = int(pd.Series(y).value_counts().min())
    usable_splits = min(int(requested_splits), minimum_class_count)

    if usable_splits < 2:
        raise RuntimeError(
            "At least two observations per class are required for CV."
        )

    return usable_splits


def safe_regression_splits(y, requested_splits):
    usable_splits = min(int(requested_splits), len(y) - 1)

    if usable_splits < 2:
        raise RuntimeError(
            "At least three observations are required for regression CV."
        )

    return usable_splits


def dominant_parameter_summary(parameter_json_values):
    parameter_series = pd.Series(parameter_json_values, dtype="string")
    counts = parameter_series.value_counts(dropna=False)

    if counts.empty:
        return "{}", np.nan

    return str(counts.index[0]), float(counts.iloc[0] / counts.sum())


def classification_class_weight_summary(tuning_results):
    required_columns = {
        "model",
        "selection_stage",
        "outer_fold",
        "mean_test_pr_auc",
        "param_model__class_weight",
    }

    if not required_columns.issubset(tuning_results.columns):
        return pd.DataFrame(
            columns=[
                "model",
                "class_weight",
                "mean_outer_fold_best_pr_auc",
                "std_outer_fold_best_pr_auc",
                "outer_folds",
            ]
        )

    rows = tuning_results.loc[
        tuning_results["selection_stage"].eq(
            "nested_outer_inner_search"
        )
        & tuning_results["model"].isin(
            ["Logistic Regression", "Random Forest"]
        )
    ].copy()

    if rows.empty:
        return pd.DataFrame()

    rows["class_weight"] = rows[
        "param_model__class_weight"
    ].astype("string").fillna("None")

    per_fold_best = (
        rows.groupby(
            ["model", "class_weight", "outer_fold"],
            observed=True,
        )["mean_test_pr_auc"]
        .max()
        .reset_index()
    )

    return (
        per_fold_best.groupby(
            ["model", "class_weight"],
            observed=True,
        )["mean_test_pr_auc"]
        .agg(["mean", "std", "count"])
        .reset_index()
        .rename(columns={
            "mean": "mean_outer_fold_best_pr_auc",
            "std": "std_outer_fold_best_pr_auc",
            "count": "outer_folds",
        })
    )



class ManualSearchResult:
    """Small GridSearchCV-compatible record for fully nested target tuning."""

    def __init__(self, best_params, best_score, best_estimator, cv_results):
        self.best_params_ = dict(best_params)
        self.best_score_ = float(best_score)
        self.best_estimator_ = best_estimator
        self.cv_results_ = cv_results.to_dict(orient="list")


def nested_target_classification_search(
    X,
    emissions,
    specification,
    random_state,
    requested_splits,
    adaptive_target=True,
    fixed_target_threshold=None,
):
    """
    Tune one classifier without allowing validation outcomes to define
    their own target threshold.

    With adaptive_target=True, each inner-training fold derives its own Q75
    emissions threshold. That threshold is then applied unchanged to the
    corresponding inner-validation fold. Splits use KFold rather than
    stratification on a target derived from the full candidate dataset.
    """
    X = X.reset_index(drop=True)
    emissions = pd.Series(emissions, dtype=float).reset_index(drop=True)

    split_count = safe_regression_splits(
        emissions,
        requested_splits,
    )
    inner_cv = KFold(
        n_splits=split_count,
        shuffle=True,
        random_state=random_state,
    )

    parameter_candidates = list(
        ParameterGrid(specification["param_grid"])
    )
    if not parameter_candidates:
        parameter_candidates = [{}]

    all_parameter_names = sorted({
        parameter_name
        for candidate in parameter_candidates
        for parameter_name in candidate
    })

    candidate_rows = []

    for candidate_index, parameters in enumerate(
        parameter_candidates,
        start=1,
    ):
        pr_auc_scores = []
        roc_auc_scores = []
        balanced_accuracy_scores = []
        f1_scores = []
        target_thresholds = []

        for inner_fold, (
            inner_train_index,
            inner_valid_index,
        ) in enumerate(
            inner_cv.split(X),
            start=1,
        ):
            X_inner_train = X.iloc[inner_train_index]
            X_inner_valid = X.iloc[inner_valid_index]
            emissions_inner_train = emissions.iloc[
                inner_train_index
            ]
            emissions_inner_valid = emissions.iloc[
                inner_valid_index
            ]

            if adaptive_target:
                target_threshold = float(
                    emissions_inner_train.quantile(
                        HIGH_EMISSION_QUANTILE
                    )
                )
            else:
                if fixed_target_threshold is None:
                    raise ValueError(
                        "fixed_target_threshold is required when "
                        "adaptive_target=False."
                    )
                target_threshold = float(fixed_target_threshold)

            y_inner_train = (
                emissions_inner_train >= target_threshold
            ).astype(int)
            y_inner_valid = (
                emissions_inner_valid >= target_threshold
            ).astype(int)

            if (
                y_inner_train.nunique() != 2
                or y_inner_valid.nunique() != 2
            ):
                raise RuntimeError(
                    "An inner classification fold contains only one "
                    "class after the training-fold target threshold."
                )

            estimator = clone(
                specification["pipeline"]
            ).set_params(**parameters)
            estimator.fit(X_inner_train, y_inner_train)

            probabilities = estimator.predict_proba(
                X_inner_valid
            )[:, 1]
            predictions_at_half = (
                probabilities >= 0.50
            ).astype(int)

            pr_auc_scores.append(
                average_precision_score(
                    y_inner_valid,
                    probabilities,
                )
            )
            roc_auc_scores.append(
                roc_auc_score(
                    y_inner_valid,
                    probabilities,
                )
            )
            balanced_accuracy_scores.append(
                balanced_accuracy_score(
                    y_inner_valid,
                    predictions_at_half,
                )
            )
            f1_scores.append(
                f1_score(
                    y_inner_valid,
                    predictions_at_half,
                    zero_division=0,
                )
            )
            target_thresholds.append(target_threshold)

        row = {
            "candidate_index": candidate_index,
            "params": json.dumps(
                parameters,
                sort_keys=True,
                default=str,
            ),
            "mean_test_pr_auc": float(
                np.mean(pr_auc_scores)
            ),
            "std_test_pr_auc": float(
                np.std(pr_auc_scores, ddof=1)
                if len(pr_auc_scores) > 1
                else 0.0
            ),
            "mean_test_roc_auc": float(
                np.mean(roc_auc_scores)
            ),
            "mean_test_balanced_accuracy": float(
                np.mean(balanced_accuracy_scores)
            ),
            "mean_test_f1": float(np.mean(f1_scores)),
            "valid_inner_folds": len(pr_auc_scores),
            "inner_target_source": (
                "inner-training Q75"
                if adaptive_target
                else "fixed development threshold"
            ),
            "mean_inner_target_threshold_kg_co2": float(
                np.mean(target_thresholds)
            ),
            "minimum_inner_target_threshold_kg_co2": float(
                np.min(target_thresholds)
            ),
            "maximum_inner_target_threshold_kg_co2": float(
                np.max(target_thresholds)
            ),
        }

        for parameter_name in all_parameter_names:
            row[f"param_{parameter_name}"] = parameters.get(
                parameter_name,
                None,
            )

        candidate_rows.append(row)

    candidate_results = pd.DataFrame(candidate_rows)

    best_row = (
        candidate_results.sort_values(
            [
                "mean_test_pr_auc",
                "mean_test_balanced_accuracy",
                "mean_test_f1",
                "candidate_index",
            ],
            ascending=[False, False, False, True],
        )
        .iloc[0]
    )

    best_params = json.loads(best_row["params"])
    return (
        best_params,
        float(best_row["mean_test_pr_auc"]),
        candidate_results,
        split_count,
    )


def select_classification_on_development(
    X_development,
    emissions_development,
    final_target_threshold,
    random_state=RANDOM_STATE,
    outer_splits=OUTER_CV_SPLITS,
    inner_splits=INNER_CV_SPLITS,
    adaptive_outer_target=True,
    refit_all_models=True,
    collect_grid_results=True,
):
    """
    Select a classifier and probability threshold using development data only.

    The final holdout is never passed here. With adaptive_outer_target=True:
    - outer folds are ordinary shuffled KFold splits, so the full-development
      target is not used to stratify model-selection folds;
    - each outer-training fold derives its own Q75 target threshold;
    - each inner-training fold independently derives its own Q75 threshold
      for hyperparameter tuning;
    - the corresponding validation fold is labelled using the threshold
      learned from its training fold only.

    This closes target-construction leakage while preserving the prespecified
    relative high-emission definition.
    """
    X_development = X_development.reset_index(drop=True)
    emissions_development = pd.Series(
        emissions_development,
        dtype=float,
    ).reset_index(drop=True)

    final_y_development = (
        emissions_development >= float(final_target_threshold)
    ).astype(int)

    outer_split_count = safe_regression_splits(
        emissions_development,
        outer_splits,
    )
    outer_cv = KFold(
        n_splits=outer_split_count,
        shuffle=True,
        random_state=random_state,
    )

    search_spaces = build_classification_search_spaces(
        random_state=random_state
    )

    oof_probabilities = {
        name: np.full(
            len(X_development),
            np.nan,
            dtype=float,
        )
        for name in search_spaces
    }
    oof_targets = {
        name: np.full(
            len(X_development),
            -1,
            dtype=int,
        )
        for name in search_spaces
    }

    outer_fold_rows = []
    target_threshold_rows = []
    grid_result_frames = []

    for outer_fold, (
        outer_train_index,
        outer_valid_index,
    ) in enumerate(
        outer_cv.split(X_development),
        start=1,
    ):
        X_outer_train = X_development.iloc[
            outer_train_index
        ]
        X_outer_valid = X_development.iloc[
            outer_valid_index
        ]
        emissions_outer_train = emissions_development.iloc[
            outer_train_index
        ]
        emissions_outer_valid = emissions_development.iloc[
            outer_valid_index
        ]

        if adaptive_outer_target:
            outer_target_threshold = float(
                emissions_outer_train.quantile(
                    HIGH_EMISSION_QUANTILE
                )
            )
            outer_target_source = "outer-training Q75"
        else:
            outer_target_threshold = float(
                final_target_threshold
            )
            outer_target_source = (
                "fixed development threshold"
            )

        y_outer_train = (
            emissions_outer_train >= outer_target_threshold
        ).astype(int)
        y_outer_valid = (
            emissions_outer_valid >= outer_target_threshold
        ).astype(int)

        if (
            y_outer_train.nunique() != 2
            or y_outer_valid.nunique() != 2
        ):
            raise RuntimeError(
                "An outer classification fold contains only one "
                "class after applying the training-defined target."
            )

        target_threshold_rows.append({
            "outer_fold": outer_fold,
            "outer_training_households": len(
                outer_train_index
            ),
            "outer_validation_households": len(
                outer_valid_index
            ),
            "target_threshold_kg_co2": (
                outer_target_threshold
            ),
            "target_threshold_kwh": (
                outer_target_threshold
                / GRID_FACTOR_KG_CO2_PER_KWH
            ),
            "target_threshold_source": (
                outer_target_source
            ),
            "outer_training_positive_rate": (
                y_outer_train.mean()
            ),
            "outer_validation_positive_rate": (
                y_outer_valid.mean()
            ),
        })

        for model_name, specification in (
            search_spaces.items()
        ):
            (
                best_parameters,
                inner_best_pr_auc,
                candidate_cv,
                inner_split_count,
            ) = nested_target_classification_search(
                X=X_outer_train,
                emissions=emissions_outer_train,
                specification=specification,
                random_state=(
                    random_state
                    + outer_fold * 100
                ),
                requested_splits=inner_splits,
                adaptive_target=adaptive_outer_target,
                fixed_target_threshold=(
                    outer_target_threshold
                    if not adaptive_outer_target
                    else None
                ),
            )

            estimator = clone(
                specification["pipeline"]
            ).set_params(**best_parameters)
            estimator.fit(
                X_outer_train,
                y_outer_train,
            )

            probabilities = estimator.predict_proba(
                X_outer_valid
            )[:, 1]

            oof_probabilities[model_name][
                outer_valid_index
            ] = probabilities
            oof_targets[model_name][
                outer_valid_index
            ] = y_outer_valid.to_numpy()

            outer_metrics_at_half = (
                classification_metrics(
                    y_outer_valid,
                    probabilities,
                    threshold=0.50,
                )
            )

            outer_fold_rows.append({
                "model": model_name,
                "outer_fold": outer_fold,
                "outer_training_households": len(
                    outer_train_index
                ),
                "outer_validation_households": len(
                    outer_valid_index
                ),
                "inner_cv_splits": inner_split_count,
                "outer_target_threshold_kg_co2": (
                    outer_target_threshold
                ),
                "inner_target_definition": (
                    "inner-training Q75"
                    if adaptive_outer_target
                    else "fixed development threshold"
                ),
                "best_parameters": json.dumps(
                    best_parameters,
                    sort_keys=True,
                    default=str,
                ),
                "inner_cv_best_pr_auc": (
                    inner_best_pr_auc
                ),
                "outer_pr_auc": (
                    outer_metrics_at_half["pr_auc"]
                ),
                "outer_roc_auc": (
                    outer_metrics_at_half["roc_auc"]
                ),
                "outer_balanced_accuracy_at_0_50": (
                    outer_metrics_at_half[
                        "balanced_accuracy"
                    ]
                ),
            })

            if collect_grid_results:
                candidate_cv = candidate_cv.copy()
                candidate_cv.insert(
                    0,
                    "model",
                    model_name,
                )
                candidate_cv.insert(
                    1,
                    "selection_stage",
                    "nested_outer_inner_search",
                )
                candidate_cv.insert(
                    2,
                    "outer_fold",
                    outer_fold,
                )
                candidate_cv.insert(
                    3,
                    "outer_target_threshold_kg_co2",
                    outer_target_threshold,
                )
                grid_result_frames.append(
                    candidate_cv
                )

    selection_rows = []
    threshold_tables = {}
    oof_prediction_frames = []

    outer_fold_results = pd.DataFrame(
        outer_fold_rows
    )
    outer_target_thresholds = pd.DataFrame(
        target_threshold_rows
    )

    for model_name in search_spaces:
        probabilities = oof_probabilities[
            model_name
        ]
        targets = oof_targets[model_name]

        if (
            np.isnan(probabilities).any()
            or (targets < 0).any()
        ):
            raise RuntimeError(
                "Nested OOF predictions are incomplete "
                f"for {model_name}."
            )

        selected_threshold, threshold_table = (
            select_development_threshold(
                targets,
                probabilities,
            )
        )
        threshold_tables[
            model_name
        ] = threshold_table

        nested_metrics = classification_metrics(
            targets,
            probabilities,
            threshold=selected_threshold,
        )

        model_fold_rows = outer_fold_results.loc[
            outer_fold_results[
                "model"
            ].eq(model_name)
        ]
        (
            dominant_parameters,
            parameter_agreement,
        ) = dominant_parameter_summary(
            model_fold_rows["best_parameters"]
        )

        selection_rows.append({
            "model": model_name,
            "nested_oof_pr_auc": (
                nested_metrics["pr_auc"]
            ),
            "nested_oof_roc_auc": (
                nested_metrics["roc_auc"]
            ),
            "nested_oof_selected_threshold": (
                selected_threshold
            ),
            "nested_oof_balanced_accuracy": (
                nested_metrics[
                    "balanced_accuracy"
                ]
            ),
            "nested_oof_high_emission_precision": (
                nested_metrics[
                    "high_emission_precision"
                ]
            ),
            "nested_oof_high_emission_recall": (
                nested_metrics[
                    "high_emission_recall"
                ]
            ),
            "nested_oof_high_emission_f1": (
                nested_metrics[
                    "high_emission_f1"
                ]
            ),
            "nested_oof_mcc": (
                nested_metrics["mcc"]
            ),
            "outer_fold_pr_auc_mean": (
                model_fold_rows[
                    "outer_pr_auc"
                ].mean()
            ),
            "outer_fold_pr_auc_std": (
                model_fold_rows[
                    "outer_pr_auc"
                ].std(ddof=1)
            ),
            "dominant_outer_best_parameters": (
                dominant_parameters
            ),
            "outer_parameter_agreement_fraction": (
                parameter_agreement
            ),
        })

        oof_prediction_frames.append(
            pd.DataFrame({
                "development_row": np.arange(
                    len(X_development)
                ),
                "model": model_name,
                "nested_oof_target": targets,
                "nested_oof_probability": (
                    probabilities
                ),
                "selected_probability_threshold": (
                    selected_threshold
                ),
                "nested_oof_prediction": (
                    probabilities
                    >= selected_threshold
                ).astype(int),
            })
        )

    model_selection = pd.DataFrame(
        selection_rows
    )

    candidate_selection = (
        model_selection.sort_values(
            [
                "outer_fold_pr_auc_mean",
                "outer_fold_pr_auc_std",
                "model",
            ],
            ascending=[False, True, True],
        )
    )

    selected_model_name = (
        candidate_selection.iloc[0]["model"]
    )
    selected_threshold = float(
        candidate_selection.iloc[0][
            "nested_oof_selected_threshold"
        ]
    )

    model_selection[
        "selected_by_development_cv"
    ] = model_selection["model"].eq(
        selected_model_name
    )
    model_selection["selection_metric"] = (
        "highest mean outer-fold PR-AUC; lower outer-fold "
        "PR-AUC SD and model name deterministic tie-break"
    )

    final_refit_names = (
        list(search_spaces)
        if refit_all_models
        else [
            "Dummy prior baseline",
            selected_model_name,
        ]
    )

    fitted_searches = {}
    fitted_estimators = {}
    final_best_parameter_map = {}

    final_split_count = safe_regression_splits(
        emissions_development,
        FINAL_TUNING_CV_SPLITS,
    )

    for model_name in final_refit_names:
        specification = search_spaces[
            model_name
        ]

        (
            best_parameters,
            best_score,
            final_grid,
            _,
        ) = nested_target_classification_search(
            X=X_development,
            emissions=emissions_development,
            specification=specification,
            random_state=random_state + 9000,
            requested_splits=(
                FINAL_TUNING_CV_SPLITS
            ),
            adaptive_target=adaptive_outer_target,
            fixed_target_threshold=(
                final_target_threshold
                if not adaptive_outer_target
                else None
            ),
        )

        final_estimator = clone(
            specification["pipeline"]
        ).set_params(**best_parameters)
        final_estimator.fit(
            X_development,
            final_y_development,
        )

        fitted_searches[
            model_name
        ] = ManualSearchResult(
            best_params=best_parameters,
            best_score=best_score,
            best_estimator=final_estimator,
            cv_results=final_grid,
        )
        fitted_estimators[
            model_name
        ] = final_estimator
        final_best_parameter_map[
            model_name
        ] = json.dumps(
            best_parameters,
            sort_keys=True,
            default=str,
        )

        if collect_grid_results:
            final_grid = final_grid.copy()
            final_grid.insert(
                0,
                "model",
                model_name,
            )
            final_grid.insert(
                1,
                "selection_stage",
                "full_development_final_tuning",
            )
            final_grid.insert(
                2,
                "outer_fold",
                np.nan,
            )
            final_grid.insert(
                3,
                "outer_target_threshold_kg_co2",
                float(final_target_threshold),
            )
            grid_result_frames.append(
                final_grid
            )

    model_selection[
        "full_development_best_parameters"
    ] = model_selection["model"].map(
        final_best_parameter_map
    )

    tuning_results = (
        pd.concat(
            grid_result_frames,
            ignore_index=True,
        )
        if grid_result_frames
        else pd.DataFrame()
    )

    return {
        "selection_procedure": (
            "nested_outer_cv_with_inner_hyperparameter_tuning"
        ),
        "model_family_selection_statistic": (
            "mean_outer_fold_pr_auc"
        ),
        "adaptive_outer_target": bool(
            adaptive_outer_target
        ),
        "outer_split_strategy": (
            "shuffled KFold independent of global target labels"
        ),
        "inner_target_definition": (
            "inner-training Q75"
            if adaptive_outer_target
            else "fixed development threshold"
        ),
        "outer_cv_splits": outer_split_count,
        "inner_cv_splits_requested": (
            inner_splits
        ),
        "final_tuning_cv_splits": (
            final_split_count
        ),
        "final_y_development": (
            final_y_development
        ),
        "model_selection": model_selection,
        "outer_fold_results": (
            outer_fold_results
        ),
        "outer_target_thresholds": (
            outer_target_thresholds
        ),
        "oof_probabilities": (
            oof_probabilities
        ),
        "oof_targets": oof_targets,
        "oof_predictions": pd.concat(
            oof_prediction_frames,
            ignore_index=True,
        ),
        "threshold_tables": (
            threshold_tables
        ),
        "selected_model_name": (
            selected_model_name
        ),
        "selected_threshold": (
            selected_threshold
        ),
        "fitted_searches": (
            fitted_searches
        ),
        "fitted_models": (
            fitted_estimators
        ),
        "tuning_results": (
            tuning_results
        ),
        "class_weight_ablation": (
            classification_class_weight_summary(
                tuning_results
            )
        ),
    }



def select_regression_on_development(
    X_development,
    y_development,
    random_state=RANDOM_STATE,
    outer_splits=OUTER_CV_SPLITS,
    inner_splits=INNER_CV_SPLITS,
    refit_all_models=True,
    collect_grid_results=True,
    selection_objective="mae",
    add_missing_indicators=False,
    include_log_target=False,
):
    """Select a regression family with honest nested OOF predictions."""
    X_development = X_development.reset_index(drop=True)
    y_development = pd.Series(y_development).reset_index(drop=True)

    outer_split_count = safe_regression_splits(
        y_development,
        outer_splits,
    )
    outer_cv = KFold(
        n_splits=outer_split_count,
        shuffle=True,
        random_state=random_state,
    )

    if selection_objective not in {"mae", "r2"}:
        raise ValueError("selection_objective must be 'mae' or 'r2'.")

    search_spaces = build_regression_search_spaces(
        random_state=random_state,
        add_missing_indicators=add_missing_indicators,
        include_log_target=include_log_target,
    )
    scoring = {
        "neg_mae": clipped_neg_mae_scorer,
        "neg_mse": clipped_neg_mse_scorer,
        "r2": clipped_r2_scorer,
    }
    refit_metric = "r2" if selection_objective == "r2" else "neg_mae"

    oof_predictions = {
        name: np.full(len(X_development), np.nan, dtype=float)
        for name in search_spaces
    }
    outer_fold_rows = []
    grid_result_frames = []

    for outer_fold, (outer_train_index, outer_valid_index) in enumerate(
        outer_cv.split(X_development),
        start=1,
    ):
        X_outer_train = X_development.iloc[outer_train_index]
        X_outer_valid = X_development.iloc[outer_valid_index]
        y_outer_train = y_development.iloc[outer_train_index]
        y_outer_valid = y_development.iloc[outer_valid_index]

        inner_split_count = safe_regression_splits(
            y_outer_train,
            inner_splits,
        )
        inner_cv = KFold(
            n_splits=inner_split_count,
            shuffle=True,
            random_state=random_state + outer_fold * 100,
        )

        for model_name, specification in search_spaces.items():
            search = GridSearchCV(
                estimator=specification["pipeline"],
                param_grid=specification["param_grid"],
                scoring=scoring,
                refit=refit_metric,
                cv=inner_cv,
                n_jobs=SEARCH_N_JOBS,
                return_train_score=False,
                error_score="raise",
            )
            search.fit(X_outer_train, y_outer_train)
            predictions = np.clip(
                np.asarray(
                    search.best_estimator_.predict(X_outer_valid),
                    dtype=float,
                ),
                0.0,
                None,
            )
            oof_predictions[model_name][outer_valid_index] = (
                predictions
            )
            outer_metrics = regression_metrics(
                y_outer_valid,
                predictions,
            )

            outer_fold_rows.append({
                "model": model_name,
                "outer_fold": outer_fold,
                "outer_training_households": len(outer_train_index),
                "outer_validation_households": len(outer_valid_index),
                "inner_cv_splits": inner_split_count,
                "best_parameters": json.dumps(
                    search.best_params_,
                    sort_keys=True,
                    default=str,
                ),
                "inner_cv_refit_metric": refit_metric,
                "inner_cv_best_score": search.best_score_,
                "target_transform": specification[
                    "target_transform"
                ],
                "outer_mae_kg_co2": outer_metrics["mae_kg_co2"],
                "outer_rmse_kg_co2": outer_metrics["rmse_kg_co2"],
                "outer_r2": outer_metrics["r2"],
            })

            if collect_grid_results:
                candidate_cv = pd.DataFrame(search.cv_results_)
                candidate_cv.insert(0, "model", model_name)
                candidate_cv.insert(1, "selection_stage", (
                    "nested_outer_inner_search"
                ))
                candidate_cv.insert(2, "outer_fold", outer_fold)
                grid_result_frames.append(candidate_cv)

    outer_fold_results = pd.DataFrame(outer_fold_rows)
    selection_rows = []
    oof_prediction_frames = []

    for model_name in search_spaces:
        predictions = oof_predictions[model_name]

        if np.isnan(predictions).any():
            raise RuntimeError(
                f"Nested OOF predictions are incomplete for {model_name}."
            )

        nested_metrics = regression_metrics(
            y_development,
            predictions,
        )
        model_fold_rows = outer_fold_results.loc[
            outer_fold_results["model"].eq(model_name)
        ]
        dominant_parameters, parameter_agreement = (
            dominant_parameter_summary(
                model_fold_rows["best_parameters"]
            )
        )

        selection_rows.append({
            "model": model_name,
            "target_transform": search_spaces[model_name][
                "target_transform"
            ],
            "nested_oof_mae_kg_co2": nested_metrics["mae_kg_co2"],
            "nested_oof_rmse_kg_co2": nested_metrics["rmse_kg_co2"],
            "nested_oof_median_absolute_error_kg_co2": (
                nested_metrics[
                    "median_absolute_error_kg_co2"
                ]
            ),
            "nested_oof_r2": nested_metrics["r2"],
            "nested_oof_spearman_r": nested_metrics["spearman_r"],
            "outer_fold_mae_mean": model_fold_rows[
                "outer_mae_kg_co2"
            ].mean(),
            "outer_fold_mae_std": model_fold_rows[
                "outer_mae_kg_co2"
            ].std(ddof=1),
            "dominant_outer_best_parameters": dominant_parameters,
            "outer_parameter_agreement_fraction": parameter_agreement,
        })

        oof_prediction_frames.append(pd.DataFrame({
            "development_row": np.arange(len(X_development)),
            "model": model_name,
            "actual_emissions_kg": y_development,
            "nested_oof_predicted_emissions_kg": predictions,
            "nested_oof_residual_kg": (
                y_development.to_numpy() - predictions
            ),
        }))

    model_selection = pd.DataFrame(selection_rows)
    # The dummy baseline remains eligible for final selection.
    candidate_selection = model_selection.copy()

    if selection_objective == "r2":
        candidate_selection = candidate_selection.sort_values(
            [
                "nested_oof_r2",
                "nested_oof_rmse_kg_co2",
                "nested_oof_mae_kg_co2",
            ],
            ascending=[False, True, True],
        )
        selection_metric_text = (
            "highest honest nested OOF original-scale R2; "
            "RMSE and MAE tie-break"
        )
    else:
        candidate_selection = candidate_selection.sort_values(
            [
                "nested_oof_mae_kg_co2",
                "nested_oof_rmse_kg_co2",
                "nested_oof_r2",
            ],
            ascending=[True, True, False],
        )
        selection_metric_text = (
            "lowest honest nested OOF MAE; RMSE and R2 tie-break"
        )

    selected_model_name = candidate_selection.iloc[0]["model"]
    model_selection["selected_by_development_cv"] = (
        model_selection["model"].eq(selected_model_name)
    )
    model_selection["selection_metric"] = selection_metric_text

    final_refit_names = (
        list(search_spaces)
        if refit_all_models
        else ["Dummy mean baseline", selected_model_name]
    )
    fitted_searches = {}
    fitted_estimators = {}
    final_best_parameter_map = {}

    final_split_count = safe_regression_splits(
        y_development,
        FINAL_TUNING_CV_SPLITS,
    )
    final_cv = KFold(
        n_splits=final_split_count,
        shuffle=True,
        random_state=random_state + 9000,
    )

    for model_name in final_refit_names:
        specification = search_spaces[model_name]
        final_search = GridSearchCV(
            estimator=specification["pipeline"],
            param_grid=specification["param_grid"],
            scoring=scoring,
            refit=refit_metric,
            cv=final_cv,
            n_jobs=SEARCH_N_JOBS,
            return_train_score=False,
            error_score="raise",
        )
        final_search.fit(X_development, y_development)
        fitted_searches[model_name] = final_search
        fitted_estimators[model_name] = final_search.best_estimator_
        final_best_parameter_map[model_name] = json.dumps(
            final_search.best_params_,
            sort_keys=True,
            default=str,
        )

        if collect_grid_results:
            final_grid = pd.DataFrame(final_search.cv_results_)
            final_grid.insert(0, "model", model_name)
            final_grid.insert(1, "selection_stage", (
                "full_development_final_tuning"
            ))
            final_grid.insert(2, "outer_fold", np.nan)
            grid_result_frames.append(final_grid)

    model_selection["full_development_best_parameters"] = (
        model_selection["model"].map(final_best_parameter_map)
    )
    tuning_results = (
        pd.concat(grid_result_frames, ignore_index=True)
        if grid_result_frames
        else pd.DataFrame()
    )

    return {
        "selection_procedure": (
            "nested_outer_cv_with_inner_hyperparameter_tuning"
        ),
        "selection_objective": selection_objective,
        "regression_scoring_rule": REGRESSION_SCORING_RULE,
        "add_missing_indicators": bool(add_missing_indicators),
        "include_log_target": bool(include_log_target),
        "outer_cv_splits": outer_split_count,
        "inner_cv_splits_requested": inner_splits,
        "final_tuning_cv_splits": final_split_count,
        "model_selection": model_selection,
        "outer_fold_results": outer_fold_results,
        "oof_predictions": pd.concat(
            oof_prediction_frames,
            ignore_index=True,
        ),
        "selected_model_name": selected_model_name,
        "fitted_searches": fitted_searches,
        "fitted_models": fitted_estimators,
        "tuning_results": tuning_results,
    }


def run_classification_workflow(
    modelling_table,
    feature_columns,
    cohort_name,
    split_plan,
    outer_splits=OUTER_CV_SPLITS,
    inner_splits=INNER_CV_SPLITS,
    adaptive_outer_target=True,
):
    data = prepare_split_data(
        modelling_table,
        feature_columns,
        split_plan,
        cohort_name,
    )

    X_development = data["X_development"]
    X_test = data["X_test"]
    threshold_emissions = float(
        split_plan["development_threshold_emissions"]
    )
    y_development = (
        data["emissions_development"] >= threshold_emissions
    ).astype(int)
    y_test = (
        data["emissions_test"] >= threshold_emissions
    ).astype(int)

    if y_development.nunique() != 2 or y_test.nunique() != 2:
        raise RuntimeError(
            f"{cohort_name} split does not contain both classes."
        )

    selection = select_classification_on_development(
        X_development=X_development,
        emissions_development=data["emissions_development"],
        final_target_threshold=threshold_emissions,
        random_state=split_plan["random_state"],
        outer_splits=outer_splits,
        inner_splits=inner_splits,
        adaptive_outer_target=adaptive_outer_target,
        refit_all_models=True,
        collect_grid_results=True,
    )

    selected_model_name = selection["selected_model_name"]
    selected_model = selection["fitted_models"][
        selected_model_name
    ]
    selected_threshold = selection["selected_threshold"]

    dummy_outer_mean_pr_auc = float(
        selection["model_selection"].loc[
            selection["model_selection"]["model"].eq(
                "Dummy prior baseline"
            ),
            "outer_fold_pr_auc_mean",
        ].iloc[0]
    )
    selected_outer_mean_pr_auc = float(
        selection["model_selection"].loc[
            selection["model_selection"]["model"].eq(
                selected_model_name
            ),
            "outer_fold_pr_auc_mean",
        ].iloc[0]
    )

    evaluation_names = list(dict.fromkeys([
        "Dummy prior baseline",
        selected_model_name,
    ]))
    test_metric_rows = []
    test_prediction_frames = []

    for model_name in evaluation_names:
        fitted_model = selection["fitted_models"][model_name]
        probabilities = fitted_model.predict_proba(X_test)[:, 1]
        threshold = (
            selected_threshold
            if model_name == selected_model_name
            else 0.50
        )
        metrics = classification_metrics(
            y_test,
            probabilities,
            threshold=threshold,
        )
        test_metric_rows.append({
            "cohort": cohort_name,
            "model": model_name,
            "selected_model": model_name == selected_model_name,
            **metrics,
        })
        test_prediction_frames.append(pd.DataFrame({
            "household_ID": data["id_test"],
            "cohort": cohort_name,
            "model": model_name,
            "actual_mean_monthly_kwh": data["kwh_test"],
            "actual_mean_monthly_emissions_kg": data[
                "emissions_test"
            ],
            "actual_high_emission": y_test,
            "uncalibrated_model_probability": probabilities,
            "locked_decision_threshold": threshold,
            "predicted_high_emission": (
                probabilities >= threshold
            ).astype(int),
        }))

    test_metrics = pd.DataFrame(test_metric_rows)
    test_predictions = pd.concat(
        test_prediction_frames,
        ignore_index=True,
    )
    selected_test_predictions = test_predictions.loc[
        test_predictions["model"].eq(selected_model_name)
    ].reset_index(drop=True)
    selected_test_probabilities = selected_test_predictions[
        "uncalibrated_model_probability"
    ].to_numpy()

    bootstrap_long, bootstrap_intervals = (
        bootstrap_classification_intervals(
            y_test,
            selected_test_probabilities,
            threshold=selected_threshold,
        )
    )
    selected_confusion = confusion_matrix(
        y_test,
        selected_test_probabilities >= selected_threshold,
        labels=[0, 1],
    )

    calibration_bins = min(10, max(3, len(y_test) // 20))
    observed_fraction, mean_probability = calibration_curve(
        y_test,
        selected_test_probabilities,
        n_bins=calibration_bins,
        strategy="quantile",
    )
    calibration_table = pd.DataFrame({
        "mean_uncalibrated_probability": mean_probability,
        "observed_positive_fraction": observed_fraction,
    })

    development_correlation = X_development.corr(method="spearman")
    correlated_pairs = []
    for first_index, first_feature in enumerate(feature_columns):
        for second_feature in feature_columns[first_index + 1:]:
            correlation_value = development_correlation.loc[
                first_feature,
                second_feature,
            ]
            if pd.notna(correlation_value) and abs(correlation_value) >= 0.80:
                correlated_pairs.append({
                    "feature_1": first_feature,
                    "feature_2": second_feature,
                    "spearman_rho": correlation_value,
                })
    correlated_pairs = pd.DataFrame(
        correlated_pairs,
        columns=["feature_1", "feature_2", "spearman_rho"],
    )

    split_audit = pd.DataFrame([{
        "cohort": cohort_name,
        "available_households": len(data["selected"]),
        "development_households": len(X_development),
        "heldout_test_households": len(X_test),
        "development_positive_count": int(y_development.sum()),
        "development_negative_count": int((1 - y_development).sum()),
        "heldout_test_positive_count": int(y_test.sum()),
        "heldout_test_negative_count": int((1 - y_test).sum()),
        "heldout_test_positive_count_at_least_20": bool(
            y_test.sum()
            >= MIN_TEST_POSITIVES_FOR_STABLE_POINT_ESTIMATES
        ),
        "features": len(feature_columns),
        "threshold_quantile": split_plan["threshold_quantile"],
        "threshold_kg_co2_from_development_only": threshold_emissions,
        "threshold_kwh_from_development_only": split_plan[
            "development_threshold_kwh"
        ],
        "development_positive_rate": y_development.mean(),
        "test_positive_rate": y_test.mean(),
        "test_used_for_hyperparameter_selection": False,
        "test_used_for_model_family_selection": False,
        "test_used_for_decision_threshold_selection": False,
        "test_outcomes_used_to_stratify_split": False,
        "split_strategy": (
            "Random household holdout independent of outcome values"
        ),
        "development_selection_procedure": selection[
            "selection_procedure"
        ],
        "outer_target_definition": (
            "outer-training Q75"
            if adaptive_outer_target
            else "fixed development threshold"
        ),
        "inner_target_definition": selection[
            "inner_target_definition"
        ],
        "development_cv_split_strategy": selection[
            "outer_split_strategy"
        ],
        "selected_model": selected_model_name,
        "selected_decision_threshold": selected_threshold,
        "selected_model_is_dummy_or_beats_dummy_mean_outer_fold_pr_auc": (
            selected_model_name == "Dummy prior baseline"
            or selected_outer_mean_pr_auc > dummy_outer_mean_pr_auc
        ),
    }])

    return {
        "task": "classification",
        "cohort_name": cohort_name,
        "feature_columns": feature_columns,
        "split_plan": split_plan,
        **data,
        "y_development": y_development,
        "y_test": y_test,
        "split_audit": split_audit,
        **selection,
        "selected_model": selected_model,
        "selected_confusion": selected_confusion,
        "test_metrics": test_metrics,
        "test_predictions": test_predictions,
        "calibration_table": calibration_table,
        "bootstrap_long": bootstrap_long,
        "bootstrap_intervals": bootstrap_intervals,
        "development_correlation": development_correlation,
        "correlated_pairs": correlated_pairs,
    }


def run_regression_workflow(
    modelling_table,
    feature_columns,
    cohort_name,
    split_plan,
    outer_splits=OUTER_CV_SPLITS,
    inner_splits=INNER_CV_SPLITS,
    selection_objective="mae",
    add_missing_indicators=False,
    include_log_target=False,
):
    data = prepare_split_data(
        modelling_table,
        feature_columns,
        split_plan,
        cohort_name,
    )
    X_development = data["X_development"]
    X_test = data["X_test"]
    y_development = data["emissions_development"]
    y_test = data["emissions_test"]

    selection = select_regression_on_development(
        X_development=X_development,
        y_development=y_development,
        random_state=split_plan["random_state"],
        outer_splits=outer_splits,
        inner_splits=inner_splits,
        refit_all_models=True,
        collect_grid_results=True,
        selection_objective=selection_objective,
        add_missing_indicators=add_missing_indicators,
        include_log_target=include_log_target,
    )

    selected_model_name = selection["selected_model_name"]
    selected_model = selection["fitted_models"][selected_model_name]
    dummy_nested_mae = float(
        selection["model_selection"].loc[
            selection["model_selection"]["model"].eq(
                "Dummy mean baseline"
            ),
            "nested_oof_mae_kg_co2",
        ].iloc[0]
    )
    selected_nested_mae = float(
        selection["model_selection"].loc[
            selection["model_selection"]["model"].eq(
                selected_model_name
            ),
            "nested_oof_mae_kg_co2",
        ].iloc[0]
    )

    evaluation_names = list(dict.fromkeys([
        "Dummy mean baseline",
        selected_model_name,
    ]))
    test_metric_rows = []
    test_prediction_frames = []

    for model_name in evaluation_names:
        fitted_model = selection["fitted_models"][model_name]
        predictions = np.clip(
            np.asarray(fitted_model.predict(X_test), dtype=float),
            0.0,
            None,
        )
        metrics = regression_metrics(y_test, predictions)
        test_metric_rows.append({
            "cohort": cohort_name,
            "model": model_name,
            "selected_model": model_name == selected_model_name,
            **metrics,
        })
        test_prediction_frames.append(pd.DataFrame({
            "household_ID": data["id_test"],
            "cohort": cohort_name,
            "model": model_name,
            "actual_mean_monthly_kwh": data["kwh_test"],
            "actual_mean_monthly_emissions_kg": y_test,
            "predicted_mean_monthly_emissions_kg": predictions,
            "residual_actual_minus_predicted_kg": (
                y_test.to_numpy() - predictions
            ),
        }))

    test_metrics = pd.DataFrame(test_metric_rows)
    test_predictions = pd.concat(
        test_prediction_frames,
        ignore_index=True,
    )
    selected_test_predictions = test_predictions.loc[
        test_predictions["model"].eq(selected_model_name)
    ].reset_index(drop=True)
    selected_predictions = selected_test_predictions[
        "predicted_mean_monthly_emissions_kg"
    ].to_numpy()
    bootstrap_long, bootstrap_intervals = (
        bootstrap_regression_intervals(
            y_test,
            selected_predictions,
        )
    )

    selected_test_metrics = test_metrics.loc[
        test_metrics["model"].eq(selected_model_name)
    ].iloc[0]
    baseline_test_metrics = test_metrics.loc[
        test_metrics["model"].eq("Dummy mean baseline")
    ].iloc[0]
    relative_improvement = pd.DataFrame([{
        "selected_model": selected_model_name,
        "baseline_model": "Dummy mean baseline",
        "test_mae_reduction_percent": (
            100.0
            * (
                baseline_test_metrics["mae_kg_co2"]
                - selected_test_metrics["mae_kg_co2"]
            )
            / baseline_test_metrics["mae_kg_co2"]
        ),
        "test_rmse_reduction_percent": (
            100.0
            * (
                baseline_test_metrics["rmse_kg_co2"]
                - selected_test_metrics["rmse_kg_co2"]
            )
            / baseline_test_metrics["rmse_kg_co2"]
        ),
    }])

    split_audit = pd.DataFrame([{
        "cohort": cohort_name,
        "available_households": len(data["selected"]),
        "development_households": len(X_development),
        "heldout_test_households": len(X_test),
        "features": len(feature_columns),
        "continuous_target": (
            "mean monthly electricity-related emissions kg CO2"
        ),
        "emission_factor_kg_co2_per_kwh": (
            GRID_FACTOR_KG_CO2_PER_KWH
        ),
        "test_used_for_hyperparameter_selection": False,
        "test_used_for_model_family_selection": False,
        "test_outcomes_used_to_stratify_split": False,
        "split_strategy": (
            "Same random household holdout used by classification"
        ),
        "development_selection_procedure": selection[
            "selection_procedure"
        ],
        "selected_model": selected_model_name,
        "selection_objective": selection_objective,
        "missing_value_strategy": (
            "training-fold median imputation with missingness indicators"
            if add_missing_indicators
            else "training-fold median imputation"
        ),
        "log_target_candidates_included": bool(include_log_target),
        "selected_target_transform": selection[
            "model_selection"
        ].loc[
            selection["model_selection"]["model"].eq(
                selected_model_name
            ),
            "target_transform",
        ].iloc[0],
        "selected_model_is_dummy_or_beats_dummy_nested_oof_mae": (
            selected_model_name == "Dummy mean baseline"
            or selected_nested_mae < dummy_nested_mae
        ),
    }])

    return {
        "task": "regression",
        "cohort_name": cohort_name,
        "feature_columns": feature_columns,
        "split_plan": split_plan,
        **data,
        "y_development": y_development,
        "y_test": y_test,
        "split_audit": split_audit,
        **selection,
        "selected_model": selected_model,
        "test_metrics": test_metrics,
        "test_predictions": test_predictions,
        "bootstrap_long": bootstrap_long,
        "bootstrap_intervals": bootstrap_intervals,
        "relative_improvement": relative_improvement,
    }


def run_primary_split_stability(
    modelling_table,
    feature_columns,
    primary_split_plan,
    primary_regression_results,
    primary_classification_results,
):
    """
    Re-run development-only nested selection under two alternative holdouts.

    Alternative holdout outcomes are used only to count class prevalence; no
    alternative holdout performance is evaluated or used to alter the locked
    primary model selected under RANDOM_STATE=42.
    """
    rows = []

    def append_primary_row():
        classification_params = primary_classification_results[
            "fitted_searches"
        ][primary_classification_results["selected_model_name"]].best_params_
        regression_params = primary_regression_results[
            "fitted_searches"
        ][primary_regression_results["selected_model_name"]].best_params_
        y_test = primary_classification_results["y_test"]
        rows.append({
            "split_seed": primary_split_plan["random_state"],
            "is_locked_primary_split": True,
            "development_households": len(
                primary_split_plan["development_ids"]
            ),
            "holdout_households": len(primary_split_plan["test_ids"]),
            "holdout_positive_count": int(y_test.sum()),
            "holdout_positive_rate": float(y_test.mean()),
            "selected_regression_model": (
                primary_regression_results["selected_model_name"]
            ),
            "selected_regression_parameters": json.dumps(
                regression_params,
                sort_keys=True,
                default=str,
            ),
            "selected_classification_model": (
                primary_classification_results[
                    "selected_model_name"
                ]
            ),
            "selected_classification_parameters": json.dumps(
                classification_params,
                sort_keys=True,
                default=str,
            ),
            "selected_probability_threshold": float(
                primary_classification_results[
                    "selected_threshold"
                ]
            ),
        })

    append_primary_row()

    for seed in STABILITY_ALTERNATIVE_SEEDS:
        alternative_plan = create_split_plan(
            modelling_table,
            f"stability_split_seed_{seed}",
            random_state=seed,
        )
        alternative_data = prepare_split_data(
            modelling_table,
            feature_columns,
            alternative_plan,
            f"stability_split_seed_{seed}",
        )

        regression_selection = select_regression_on_development(
            X_development=alternative_data["X_development"],
            y_development=alternative_data[
                "emissions_development"
            ],
            random_state=seed,
            outer_splits=STABILITY_OUTER_CV_SPLITS,
            inner_splits=STABILITY_INNER_CV_SPLITS,
            refit_all_models=False,
            collect_grid_results=False,
        )
        classification_selection = (
            select_classification_on_development(
                X_development=alternative_data["X_development"],
                emissions_development=alternative_data[
                    "emissions_development"
                ],
                final_target_threshold=alternative_plan[
                    "development_threshold_emissions"
                ],
                random_state=seed,
                outer_splits=STABILITY_OUTER_CV_SPLITS,
                inner_splits=STABILITY_INNER_CV_SPLITS,
                adaptive_outer_target=True,
                refit_all_models=False,
                collect_grid_results=False,
            )
        )

        alt_threshold = alternative_plan[
            "development_threshold_emissions"
        ]
        alternative_y_test = (
            alternative_data["emissions_test"] >= alt_threshold
        ).astype(int)

        regression_name = regression_selection[
            "selected_model_name"
        ]
        classification_name = classification_selection[
            "selected_model_name"
        ]
        rows.append({
            "split_seed": seed,
            "is_locked_primary_split": False,
            "development_households": len(
                alternative_plan["development_ids"]
            ),
            "holdout_households": len(
                alternative_plan["test_ids"]
            ),
            "holdout_positive_count": int(
                alternative_y_test.sum()
            ),
            "holdout_positive_rate": float(
                alternative_y_test.mean()
            ),
            "selected_regression_model": regression_name,
            "selected_regression_parameters": json.dumps(
                regression_selection["fitted_searches"][
                    regression_name
                ].best_params_,
                sort_keys=True,
                default=str,
            ),
            "selected_classification_model": classification_name,
            "selected_classification_parameters": json.dumps(
                classification_selection["fitted_searches"][
                    classification_name
                ].best_params_,
                sort_keys=True,
                default=str,
            ),
            "selected_probability_threshold": float(
                classification_selection["selected_threshold"]
            ),
        })

    stability = pd.DataFrame(rows).sort_values(
        "split_seed"
    ).reset_index(drop=True)

    summary = pd.DataFrame([{
        "evaluated_split_seeds": stability["split_seed"].nunique(),
        "regression_selected_family_count": stability[
            "selected_regression_model"
        ].nunique(),
        "classification_selected_family_count": stability[
            "selected_classification_model"
        ].nunique(),
        "probability_threshold_min": stability[
            "selected_probability_threshold"
        ].min(),
        "probability_threshold_max": stability[
            "selected_probability_threshold"
        ].max(),
        "probability_threshold_range": stability[
            "selected_probability_threshold"
        ].max()
        - stability["selected_probability_threshold"].min(),
        "minimum_holdout_positive_count": stability[
            "holdout_positive_count"
        ].min(),
        "maximum_holdout_positive_count": stability[
            "holdout_positive_count"
        ].max(),
        "all_splits_selected_same_regression_family": (
            stability["selected_regression_model"].nunique() == 1
        ),
        "all_splits_selected_same_classification_family": (
            stability["selected_classification_model"].nunique() == 1
        ),
    }])

    return stability, summary


print(
    "Primary model-family and threshold selection now use honest nested "
    "development OOF predictions before the untouched held-out test."
)

In [ ]:
# ============================================================
# STEP 13 — RUN PRIMARY MODELS, SAMPLE AUDIT AND SPLIT STABILITY
# ============================================================

primary_split_plan = create_split_plan(
    primary_model_table,
    "primary_carbon_cohort_split",
    random_state=RANDOM_STATE,
)

primary_regression_results = run_regression_workflow(
    primary_model_table,
    primary_feature_columns,
    "primary_temporal_shape_regression",
    primary_split_plan,
    outer_splits=OUTER_CV_SPLITS,
    inner_splits=INNER_CV_SPLITS,
)

primary_classification_results = run_classification_workflow(
    primary_model_table,
    primary_feature_columns,
    "primary_temporal_shape_classification",
    primary_split_plan,
    outer_splits=OUTER_CV_SPLITS,
    inner_splits=INNER_CV_SPLITS,
    adaptive_outer_target=True,
)

classification_split_row = primary_classification_results[
    "split_audit"
].iloc[0]
regression_ci = primary_regression_results[
    "bootstrap_intervals"
]
classification_ci = primary_classification_results[
    "bootstrap_intervals"
]

sample_size_audit = pd.DataFrame([{
    "eligible_households": len(primary_model_table),
    "development_households": int(
        classification_split_row["development_households"]
    ),
    "heldout_test_households": int(
        classification_split_row["heldout_test_households"]
    ),
    "development_positive_count": int(
        classification_split_row["development_positive_count"]
    ),
    "heldout_test_positive_count": int(
        classification_split_row["heldout_test_positive_count"]
    ),
    "heldout_test_negative_count": int(
        classification_split_row["heldout_test_negative_count"]
    ),
    "heldout_test_positive_rate": float(
        classification_split_row["test_positive_rate"]
    ),
    "classification_positive_count_caution": (
        "Interpret point estimates cautiously"
        if classification_split_row[
            "heldout_test_positive_count"
        ] < MIN_TEST_POSITIVES_FOR_STABLE_POINT_ESTIMATES
        else "Positive count meets the prespecified caution threshold"
    ),
    "regression_bootstrap_max_ci_width": float(
        (
            regression_ci["ci_97_5_percent"]
            - regression_ci["ci_2_5_percent"]
        ).max()
    ),
    "classification_bootstrap_max_ci_width": float(
        (
            classification_ci["ci_97_5_percent"]
            - classification_ci["ci_2_5_percent"]
        ).max()
    ),
}])

print("\nPRIMARY SAMPLE-SIZE AND UNCERTAINTY AUDIT")
display(sample_size_audit.round(4))

print("\nPRIMARY CONTINUOUS-EMISSION REGRESSION")
display(primary_regression_results["split_audit"])
display(primary_regression_results["model_selection"].round(4))
display(primary_regression_results["test_metrics"].round(4))
display(primary_regression_results["relative_improvement"].round(4))
display(primary_regression_results["bootstrap_intervals"].round(4))

print("\nSECONDARY HIGH-EMISSION CLASSIFICATION")
display(primary_classification_results["split_audit"])
display(primary_classification_results["model_selection"].round(4))
display(primary_classification_results["class_weight_ablation"].round(4))
display(primary_classification_results["test_metrics"].round(4))
display(primary_classification_results["bootstrap_intervals"].round(4))
print("Selected-model held-out confusion matrix:")
print(primary_classification_results["selected_confusion"])

split_stability_results, split_stability_summary = (
    run_primary_split_stability(
        primary_model_table,
        primary_feature_columns,
        primary_split_plan,
        primary_regression_results,
        primary_classification_results,
    )
)

print("\nDEVELOPMENT-SELECTION SPLIT-STABILITY CHECK")
display(split_stability_results)
display(split_stability_summary.round(4))

if not bool(
    split_stability_summary.loc[
        0,
        "all_splits_selected_same_classification_family",
    ]
):
    print(
        "Caution: the selected classifier family changed across split "
        "seeds. Report this sensitivity and prioritize confidence "
        "intervals over a single point estimate."
    )

survey_temporal_regression_results = None
survey_augmented_regression_results = None
survey_temporal_classification_results = None
survey_augmented_classification_results = None
survey_paired_comparison = pd.DataFrame()
survey_feature_audit = None
survey_feature_missingness = None
teacher_development_availability = None
teacher_aligned_requirements_met = False
survey_augmented_features = []

if survey_model_table is None:
    raise RuntimeError(
        "Wave-1 household-context survey modelling table was not constructed."
    )

survey_split_plan = create_split_plan(
    survey_model_table,
    "teacher_aligned_survey_matched_fresh_split",
    random_state=TEACHER_MATCHED_RANDOM_STATE,
)

if (
    len(survey_split_plan["development_ids"])
    < MIN_PRIMARY_HOUSEHOLDS * 0.60
    or len(survey_split_plan["test_ids"]) < 50
):
    raise RuntimeError(
        "The Wave-1 household-context survey-matched subset is too small "
        "for the prespecified paired ML comparison."
    )

survey_development = survey_model_table.loc[
    survey_model_table["household_ID"].isin(
        survey_split_plan["development_ids"]
    )
].copy()

teacher_development_availability = pd.DataFrame({
    "feature": TEACHER_SOCIO_FEATURES,
    "development_non_missing_count": [
        int(survey_development[feature].notna().sum())
        for feature in TEACHER_SOCIO_FEATURES
    ],
    "development_non_missing_fraction": [
        float(survey_development[feature].notna().mean())
        for feature in TEACHER_SOCIO_FEATURES
    ],
    "development_unique_non_missing_values": [
        int(survey_development[feature].nunique(dropna=True))
        for feature in TEACHER_SOCIO_FEATURES
    ],
})

unusable_teacher_features = (
    teacher_development_availability.loc[
        (
            teacher_development_availability[
                "development_non_missing_fraction"
            ] < 0.60
        )
        | (
            teacher_development_availability[
                "development_unique_non_missing_values"
            ] <= 1
        ),
        "feature",
    ].tolist()
)

if unusable_teacher_features:
    display(teacher_development_availability)
    raise RuntimeError(
        "Teacher variables failed the development-only "
        "availability/variation rule: "
        + ", ".join(unusable_teacher_features)
    )

survey_feature_columns = TEACHER_SOCIO_FEATURES.copy()
survey_augmented_features = (
    primary_feature_columns + survey_feature_columns
)

(
    survey_feature_audit,
    survey_feature_missingness,
) = audit_feature_set(
    survey_model_table,
    survey_augmented_features,
    "temporal_plus_teacher_aligned_household_variables",
)

teacher_aligned_requirements_met = bool(
    survey_feature_columns == TEACHER_SOCIO_FEATURES
    and not any(
        forbidden in feature.lower()
        for feature in survey_augmented_features
        for forbidden in FORBIDDEN_PREDICTOR_TERMS
    )
)

display(teacher_development_availability)
display(survey_feature_audit)
display(survey_feature_missingness)

# Both arms use exactly the same survey-matched development/test IDs,
# the same carbon outcome and the same teacher-matched development Q75 target.
survey_temporal_regression_results = run_regression_workflow(
    survey_model_table,
    primary_feature_columns,
    "teacher_matched_temporal_only_regression",
    survey_split_plan,
    outer_splits=SURVEY_OUTER_CV_SPLITS,
    inner_splits=SURVEY_INNER_CV_SPLITS,
    selection_objective=TEACHER_REGRESSION_SELECTION_OBJECTIVE,
    add_missing_indicators=TEACHER_REGRESSION_ADD_MISSING_INDICATORS,
    include_log_target=TEACHER_REGRESSION_INCLUDE_LOG_TARGET,
)
survey_augmented_regression_results = run_regression_workflow(
    survey_model_table,
    survey_augmented_features,
    "teacher_matched_temporal_plus_household_regression",
    survey_split_plan,
    outer_splits=SURVEY_OUTER_CV_SPLITS,
    inner_splits=SURVEY_INNER_CV_SPLITS,
    selection_objective=TEACHER_REGRESSION_SELECTION_OBJECTIVE,
    add_missing_indicators=TEACHER_REGRESSION_ADD_MISSING_INDICATORS,
    include_log_target=TEACHER_REGRESSION_INCLUDE_LOG_TARGET,
)
survey_temporal_classification_results = (
    run_classification_workflow(
        survey_model_table,
        primary_feature_columns,
        "teacher_matched_temporal_only_classification",
        survey_split_plan,
        outer_splits=SURVEY_OUTER_CV_SPLITS,
        inner_splits=SURVEY_INNER_CV_SPLITS,
        adaptive_outer_target=True,
    )
)
survey_augmented_classification_results = (
    run_classification_workflow(
        survey_model_table,
        survey_augmented_features,
        "teacher_matched_temporal_plus_household_classification",
        survey_split_plan,
        outer_splits=SURVEY_OUTER_CV_SPLITS,
        inner_splits=SURVEY_INNER_CV_SPLITS,
        adaptive_outer_target=True,
    )
)

temporal_regression_test = survey_temporal_regression_results[
    "test_metrics"
].loc[
    survey_temporal_regression_results[
        "test_metrics"
    ]["selected_model"]
].iloc[0]
augmented_regression_test = survey_augmented_regression_results[
    "test_metrics"
].loc[
    survey_augmented_regression_results[
        "test_metrics"
    ]["selected_model"]
].iloc[0]
temporal_classification_test = (
    survey_temporal_classification_results[
        "test_metrics"
    ].loc[
        survey_temporal_classification_results[
            "test_metrics"
        ]["selected_model"]
    ].iloc[0]
)
augmented_classification_test = (
    survey_augmented_classification_results[
        "test_metrics"
    ].loc[
        survey_augmented_classification_results[
            "test_metrics"
        ]["selected_model"]
    ].iloc[0]
)

survey_paired_comparison = pd.DataFrame([
    {
        "task": "regression",
        "temporal_selected_model": (
            survey_temporal_regression_results[
                "selected_model_name"
            ]
        ),
        "teacher_augmented_selected_model": (
            survey_augmented_regression_results[
                "selected_model_name"
            ]
        ),
        "temporal_primary_metric": temporal_regression_test[
            "r2_original_kg_co2_scale"
        ],
        "teacher_augmented_primary_metric": (
            augmented_regression_test[
                "r2_original_kg_co2_scale"
            ]
        ),
        "primary_metric": "Original-scale R2 (higher is better)",
        "teacher_augmented_minus_temporal": (
            augmented_regression_test[
                "r2_original_kg_co2_scale"
            ]
            - temporal_regression_test[
                "r2_original_kg_co2_scale"
            ]
        ),
        "temporal_mae_kg_co2": temporal_regression_test[
            "mae_kg_co2"
        ],
        "teacher_augmented_mae_kg_co2": (
            augmented_regression_test["mae_kg_co2"]
        ),
        "temporal_log1p_r2": temporal_regression_test[
            "r2_log1p_scale"
        ],
        "teacher_augmented_log1p_r2": (
            augmented_regression_test["r2_log1p_scale"]
        ),
    },
    {
        "task": "classification",
        "temporal_selected_model": (
            survey_temporal_classification_results[
                "selected_model_name"
            ]
        ),
        "teacher_augmented_selected_model": (
            survey_augmented_classification_results[
                "selected_model_name"
            ]
        ),
        "temporal_primary_metric": temporal_classification_test[
            "pr_auc"
        ],
        "teacher_augmented_primary_metric": (
            augmented_classification_test["pr_auc"]
        ),
        "primary_metric": "PR-AUC (higher is better)",
        "teacher_augmented_minus_temporal": (
            augmented_classification_test["pr_auc"]
            - temporal_classification_test["pr_auc"]
        ),
    },
])

print("\nPAIRED WAVE-1 HOUSEHOLD-CONTEXT SURVEY-MATCHED COMPARISON")
display(survey_paired_comparison.round(4))

print(
    "\nRegression and classification model selection use nested "
    "development CV. The final held-out test was not used to choose "
    "the model family, hyperparameters, target threshold or probability "
    "decision threshold."
)


# Prespecified descriptive sensitivity of the locked teacher-augmented
# regression model to the number of valid target months. The model is not
# refitted and these results are not used for model selection.
teacher_regression_month_count_rows = []
teacher_regression_locked_model = (
    survey_augmented_regression_results["selected_model"]
)
teacher_regression_test_ids = set(
    survey_augmented_regression_results["id_test"]
)

for minimum_months in [5, 6, 7]:
    month_subset = survey_model_table.loc[
        survey_model_table["household_ID"].isin(
            teacher_regression_test_ids
        )
        & survey_model_table["observed_months"].ge(
            minimum_months
        )
    ].copy()

    if len(month_subset) < 30:
        teacher_regression_month_count_rows.append({
            "minimum_valid_months": minimum_months,
            "heldout_households": len(month_subset),
            "status": "Too few held-out households for stable reporting",
            "r2_original_kg_co2_scale": np.nan,
            "r2_log1p_scale": np.nan,
            "mae_kg_co2": np.nan,
            "rmse_kg_co2": np.nan,
        })
        continue

    month_predictions = np.clip(
        np.asarray(
            teacher_regression_locked_model.predict(
                month_subset[survey_augmented_features]
            ),
            dtype=float,
        ),
        0.0,
        None,
    )
    month_metrics = regression_metrics(
        month_subset["mean_monthly_emissions_kg"],
        month_predictions,
    )
    teacher_regression_month_count_rows.append({
        "minimum_valid_months": minimum_months,
        "heldout_households": len(month_subset),
        "status": "Locked-model descriptive sensitivity",
        "r2_original_kg_co2_scale": month_metrics[
            "r2_original_kg_co2_scale"
        ],
        "r2_log1p_scale": month_metrics["r2_log1p_scale"],
        "mae_kg_co2": month_metrics["mae_kg_co2"],
        "rmse_kg_co2": month_metrics["rmse_kg_co2"],
    })

teacher_regression_month_count_sensitivity = pd.DataFrame(
    teacher_regression_month_count_rows
)

print("\nTEACHER-AUGMENTED REGRESSION VALID-MONTH SENSITIVITY")
display(teacher_regression_month_count_sensitivity.round(4))

print(
    "The principal regression R2 is always computed on the original "
    "kg-CO2 scale. Log1p R2 is supplementary and cannot replace the "
    "original-scale result. A positive R2 is not forced or guaranteed."
)


In [ ]:
# ============================================================
# THESIS FIGURE — MATCHED-COHORT CARBON MODEL COMPARISON
# Temporal-only versus temporal + Wave 1 household variables
# ============================================================

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# Validate the existing comparison table
# ------------------------------------------------------------
required_columns = {
    "task",
    "temporal_primary_metric",
    "teacher_augmented_primary_metric",
    "temporal_mae_kg_co2",
    "teacher_augmented_mae_kg_co2",
}

missing_columns = required_columns.difference(
    survey_paired_comparison.columns
)

if missing_columns:
    raise RuntimeError(
        "survey_paired_comparison is missing required columns: "
        + ", ".join(sorted(missing_columns))
    )

if not {"regression", "classification"}.issubset(
    set(survey_paired_comparison["task"])
):
    raise RuntimeError(
        "Both regression and classification rows are required."
    )

regression_row = (
    survey_paired_comparison.loc[
        survey_paired_comparison["task"].eq("regression")
    ]
    .iloc[0]
)

classification_row = (
    survey_paired_comparison.loc[
        survey_paired_comparison["task"].eq("classification")
    ]
    .iloc[0]
)

matched_households = int(
    survey_model_table["household_ID"].nunique()
)

model_labels = [
    "Temporal only",
    "Temporal + Wave 1",
]

r2_values = [
    float(regression_row["temporal_primary_metric"]),
    float(regression_row["teacher_augmented_primary_metric"]),
]

mae_values = [
    float(regression_row["temporal_mae_kg_co2"]),
    float(regression_row["teacher_augmented_mae_kg_co2"]),
]

pr_auc_values = [
    float(classification_row["temporal_primary_metric"]),
    float(classification_row["teacher_augmented_primary_metric"]),
]

# ------------------------------------------------------------
# Create one composite thesis figure
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))

plot_information = [
    {
        "axis": axes[0],
        "values": r2_values,
        "title": "(a) Regression R²",
        "ylabel": "R²",
        "format": "{:.3f}",
        "direction": "Higher is better",
    },
    {
        "axis": axes[1],
        "values": mae_values,
        "title": "(b) Regression MAE",
        "ylabel": "MAE (kg CO₂/month)",
        "format": "{:.2f}",
        "direction": "Lower is better",
    },
    {
        "axis": axes[2],
        "values": pr_auc_values,
        "title": "(c) Classification PR-AUC",
        "ylabel": "PR-AUC",
        "format": "{:.3f}",
        "direction": "Higher is better",
    },
]

for information in plot_information:
    ax = information["axis"]
    values = information["values"]

    bars = ax.bar(model_labels, values)

    ax.set_title(
        information["title"] + "\n" + information["direction"],
        fontsize=10,
    )
    ax.set_ylabel(information["ylabel"])
    ax.grid(axis="y", linestyle="--", alpha=0.30)
    ax.tick_params(axis="x", rotation=15)

    upper_limit = max(values) * 1.25

    if information["ylabel"] in {"R²", "PR-AUC"}:
        upper_limit = min(1.0, max(upper_limit, 0.50))

    ax.set_ylim(0, upper_limit)

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + upper_limit * 0.025,
            information["format"].format(value),
            ha="center",
            va="bottom",
            fontsize=9,
        )

fig.suptitle(
    "Carbon-model performance in the matched Wave 1 cohort "
    f"(n={matched_households:,})",
    fontsize=12,
)

fig.tight_layout(rect=[0, 0, 1, 0.91])

figure_path = (
    Path(OUTPUT_DIR)
    / "figure_carbon_matched_cohort_model_comparison.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved figure:", figure_path)

# ------------------------------------------------------------
# Display exact values used in the figure
# ------------------------------------------------------------
comparison_values = {
    "Matched households": matched_households,
    "Temporal-only R2": r2_values[0],
    "Temporal + Wave 1 R2": r2_values[1],
    "Temporal-only MAE": mae_values[0],
    "Temporal + Wave 1 MAE": mae_values[1],
    "Temporal-only PR-AUC": pr_auc_values[0],
    "Temporal + Wave 1 PR-AUC": pr_auc_values[1],
}

for name, value in comparison_values.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

In [ ]:
# ============================================================
# THESIS FIGURE — MATCHED-COHORT CARBON MODEL COMPARISON
# Temporal-only versus temporal + Wave 1 household variables
# ============================================================

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np


# ------------------------------------------------------------
# Validate the existing comparison table
# ------------------------------------------------------------

required_columns = {
    "task",
    "temporal_primary_metric",
    "teacher_augmented_primary_metric",
    "temporal_mae_kg_co2",
    "teacher_augmented_mae_kg_co2",
}

missing_columns = required_columns.difference(
    survey_paired_comparison.columns
)

if missing_columns:
    raise RuntimeError(
        "survey_paired_comparison is missing required columns: "
        + ", ".join(sorted(missing_columns))
    )

if not {"regression", "classification"}.issubset(
    set(survey_paired_comparison["task"])
):
    raise RuntimeError(
        "Both regression and classification rows are required."
    )


# ------------------------------------------------------------
# Extract regression and classification rows
# ------------------------------------------------------------

regression_row = (
    survey_paired_comparison.loc[
        survey_paired_comparison["task"].eq("regression")
    ]
    .iloc[0]
)

classification_row = (
    survey_paired_comparison.loc[
        survey_paired_comparison["task"].eq("classification")
    ]
    .iloc[0]
)


# ------------------------------------------------------------
# Matched cohort size
# ------------------------------------------------------------

matched_households = int(
    survey_model_table["household_ID"].nunique()
)


# ------------------------------------------------------------
# Model labels
# ------------------------------------------------------------

model_labels = [
    "Temporal only",
    "Temporal + Wave 1",
]


# ------------------------------------------------------------
# Regression R² values
# ------------------------------------------------------------

r2_values = [
    float(regression_row["temporal_primary_metric"]),
    float(regression_row["teacher_augmented_primary_metric"]),
]


# ------------------------------------------------------------
# Regression MAE values
# ------------------------------------------------------------

mae_values = [
    float(regression_row["temporal_mae_kg_co2"]),
    float(regression_row["teacher_augmented_mae_kg_co2"]),
]


# ------------------------------------------------------------
# Classification Average Precision (AP) values
#
# IMPORTANT:
# These values originate from average_precision_score(),
# therefore they are reported as Average Precision (AP),
# not trapezoidal PR-AUC.
# ------------------------------------------------------------

ap_values = [
    float(classification_row["temporal_primary_metric"]),
    float(classification_row["teacher_augmented_primary_metric"]),
]


# ------------------------------------------------------------
# Create one composite thesis figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(13, 4.6)
)


plot_information = [
    {
        "axis": axes[0],
        "values": r2_values,
        "title": "(a) Regression R²",
        "ylabel": "R²",
        "format": "{:.3f}",
        "direction": "Higher is better",
        "bounded_metric": True,
    },
    {
        "axis": axes[1],
        "values": mae_values,
        "title": "(b) Regression MAE",
        "ylabel": "MAE (kg CO₂/month)",
        "format": "{:.2f}",
        "direction": "Lower is better",
        "bounded_metric": False,
    },
    {
        "axis": axes[2],
        "values": ap_values,
        "title": "(c) Classification Average Precision (AP)",
        "ylabel": "Average Precision (AP)",
        "format": "{:.3f}",
        "direction": "Higher is better",
        "bounded_metric": True,
    },
]


# ------------------------------------------------------------
# Draw panels
# ------------------------------------------------------------

for information in plot_information:

    ax = information["axis"]
    values = information["values"]

    bars = ax.bar(
        model_labels,
        values
    )

    ax.set_title(
        information["title"]
        + "\n"
        + information["direction"],
        fontsize=10,
    )

    ax.set_ylabel(
        information["ylabel"]
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.30
    )

    ax.tick_params(
        axis="x",
        rotation=15
    )

    # Provide enough space above bars for value labels.
    upper_limit = max(values) * 1.25

    # R² and AP are bounded performance metrics.
    if information["bounded_metric"]:
        upper_limit = min(
            1.0,
            max(
                upper_limit,
                0.50
            )
        )

    ax.set_ylim(
        0,
        upper_limit
    )

    # Add exact values above bars.
    for bar, value in zip(
        bars,
        values
    ):

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            bar.get_height()
            + upper_limit * 0.025,
            information["format"].format(
                value
            ),
            ha="center",
            va="bottom",
            fontsize=9,
        )


# ------------------------------------------------------------
# Overall figure title
# ------------------------------------------------------------

fig.suptitle(
    "Carbon-model performance in the matched Wave 1 cohort "
    f"(n={matched_households:,})",
    fontsize=12,
)


fig.tight_layout(
    rect=[0, 0, 1, 0.91]
)


# ------------------------------------------------------------
# Save figure
# ------------------------------------------------------------

figure_path = (
    Path(OUTPUT_DIR)
    / "figure_carbon_matched_cohort_model_comparison.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

plt.close(fig)

print(
    "Saved figure:",
    figure_path
)


# ------------------------------------------------------------
# Display exact values used in the figure
# ------------------------------------------------------------

comparison_values = {
    "Matched households":
        matched_households,

    "Temporal-only R2":
        r2_values[0],

    "Temporal + Wave 1 R2":
        r2_values[1],

    "Temporal-only MAE":
        mae_values[0],

    "Temporal + Wave 1 MAE":
        mae_values[1],

    "Temporal-only Average Precision (AP)":
        ap_values[0],

    "Temporal + Wave 1 Average Precision (AP)":
        ap_values[1],
}


print(
    "\nExact values used in figure:"
)

for name, value in comparison_values.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{name}: {value:.4f}"
        )

    else:
        print(
            f"{name}: {value}"
        )

In [ ]:
# ============================================================
# STEP 14 — HELD-OUT PERMUTATION IMPORTANCE, SHAP AND DIAGNOSTICS
# ============================================================

def transform_pipeline_features(
    fitted_pipeline,
    X,
):
    transformed = X.copy()

    for step_name, transformer in list(
        fitted_pipeline.named_steps.items()
    )[:-1]:
        transformed = transformer.transform(transformed)

    return np.asarray(transformed)


def tree_shap_outputs(
    fitted_pipeline,
    X,
    feature_columns,
    task,
    household_ids,
    actual_values,
):
    import shap

    model = fitted_pipeline.named_steps["model"]
    transformed = transform_pipeline_features(
        fitted_pipeline,
        X,
    )

    explainer = shap.TreeExplainer(model)
    raw_values = explainer.shap_values(transformed)

    if task == "classification":
        if isinstance(raw_values, list):
            values = raw_values[-1]
        elif (
            isinstance(raw_values, np.ndarray)
            and raw_values.ndim == 3
        ):
            values = raw_values[:, :, -1]
        else:
            values = raw_values
    else:
        values = raw_values

    values = np.asarray(values)

    expected_shape = (
        len(X),
        len(feature_columns),
    )

    if values.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected SHAP shape {values.shape}; "
            f"expected {expected_shape}."
        )

    global_importance = (
        pd.DataFrame({
            "feature": feature_columns,
            "mean_absolute_shap": np.abs(values).mean(axis=0),
        })
        .sort_values(
            "mean_absolute_shap",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    explanation_rows = []

    for row_index in range(len(X)):
        contributions = values[row_index]
        top_indices = np.argsort(
            np.abs(contributions)
        )[-3:][::-1]

        row = {
            "household_ID": household_ids.iloc[row_index],
            "actual_value": actual_values.iloc[row_index],
        }

        for rank, feature_index in enumerate(
            top_indices,
            start=1,
        ):
            row[f"top_feature_{rank}"] = (
                feature_columns[feature_index]
            )
            row[f"top_feature_{rank}_shap"] = (
                contributions[feature_index]
            )

        explanation_rows.append(row)

    return global_importance, pd.DataFrame(
        explanation_rows
    )


classification_model = primary_classification_results[
    "selected_model"
]
classification_X_test = primary_classification_results[
    "X_test"
]
classification_y_test = primary_classification_results[
    "y_test"
]

classification_permutation = permutation_importance(
    classification_model,
    classification_X_test,
    classification_y_test,
    scoring="average_precision",
    n_repeats=50,
    random_state=RANDOM_STATE,
    n_jobs=SEARCH_N_JOBS,
)

classification_permutation_importance = (
    pd.DataFrame({
        "feature": primary_feature_columns,
        "permutation_importance_mean": (
            classification_permutation.importances_mean
        ),
        "permutation_importance_std": (
            classification_permutation.importances_std
        ),
        "interpretation": (
            "Decrease in held-out PR-AUC after permutation"
        ),
    })
    .sort_values(
        "permutation_importance_mean",
        ascending=False,
    )
    .reset_index(drop=True)
)

regression_model = primary_regression_results[
    "selected_model"
]
regression_X_test = primary_regression_results["X_test"]
regression_y_test = primary_regression_results["y_test"]

regression_permutation = permutation_importance(
    regression_model,
    regression_X_test,
    regression_y_test,
    scoring=clipped_neg_mae_scorer,
    n_repeats=50,
    random_state=RANDOM_STATE,
    n_jobs=SEARCH_N_JOBS,
)

regression_permutation_importance = (
    pd.DataFrame({
        "feature": primary_feature_columns,
        "permutation_importance_mean": (
            regression_permutation.importances_mean
        ),
        "permutation_importance_std": (
            regression_permutation.importances_std
        ),
        "interpretation": (
            "Increase in held-out MAE after permutation"
        ),
    })
    .sort_values(
        "permutation_importance_mean",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    classification_permutation_importance.round(4)
)
display(regression_permutation_importance.round(4))

classification_shap_global = None
classification_shap_households = None
regression_shap_global = None
regression_shap_households = None
classification_linear_coefficients = None
regression_linear_coefficients = None

classification_model_step = classification_model.named_steps[
    "model"
]
regression_model_step = regression_model.named_steps["model"]

if hasattr(classification_model_step, "coef_"):
    classification_linear_coefficients = (
        pd.DataFrame({
            "feature": primary_feature_columns,
            "coefficient": (
                classification_model_step.coef_.ravel()
            ),
            "absolute_coefficient": np.abs(
                classification_model_step.coef_.ravel()
            ),
        })
        .sort_values(
            "absolute_coefficient",
            ascending=False,
        )
        .reset_index(drop=True)
    )
    display(classification_linear_coefficients.round(4))

if hasattr(regression_model_step, "coef_"):
    regression_linear_coefficients = (
        pd.DataFrame({
            "feature": primary_feature_columns,
            "coefficient": (
                regression_model_step.coef_.ravel()
            ),
            "absolute_coefficient": np.abs(
                regression_model_step.coef_.ravel()
            ),
        })
        .sort_values(
            "absolute_coefficient",
            ascending=False,
        )
        .reset_index(drop=True)
    )
    display(regression_linear_coefficients.round(4))

try:
    import shap

    if (
        hasattr(classification_model_step, "feature_importances_")
        or classification_model_step.__class__.__name__.startswith(
            "HistGradientBoosting"
        )
    ):
        (
            classification_shap_global,
            classification_shap_households,
        ) = tree_shap_outputs(
            classification_model,
            classification_X_test,
            primary_feature_columns,
            "classification",
            primary_classification_results["id_test"],
            classification_y_test,
        )
        display(classification_shap_global.round(4))

    if (
        hasattr(regression_model_step, "feature_importances_")
        or regression_model_step.__class__.__name__.startswith(
            "HistGradientBoosting"
        )
    ):
        (
            regression_shap_global,
            regression_shap_households,
        ) = tree_shap_outputs(
            regression_model,
            regression_X_test,
            primary_feature_columns,
            "regression",
            primary_regression_results["id_test"],
            regression_y_test,
        )
        display(regression_shap_global.round(4))

except ImportError:
    print(
        "SHAP is optional and is not installed. "
        "Permutation importance remains the primary model-agnostic XAI result."
    )

except Exception as error:
    print("SHAP could not be completed:", str(error))
    print(
        "Permutation importance and any linear coefficients remain valid."
    )

plt.figure(figsize=(8, 5))
classification_plot = (
    classification_permutation_importance.sort_values(
        "permutation_importance_mean"
    )
)
plt.barh(
    classification_plot["feature"],
    classification_plot["permutation_importance_mean"],
)
plt.xlabel("Decrease in held-out PR-AUC")
plt.title(
    "Classification held-out permutation importance"
)
plt.tight_layout()
plt.show()
plt.close()

plt.figure(figsize=(8, 5))
regression_plot = (
    regression_permutation_importance.sort_values(
        "permutation_importance_mean"
    )
)
plt.barh(
    regression_plot["feature"],
    regression_plot["permutation_importance_mean"],
)
plt.xlabel("Increase in held-out MAE (kg CO2)")
plt.title("Regression held-out permutation importance")
plt.tight_layout()
plt.show()
plt.close()

selected_regression_predictions = (
    primary_regression_results[
        "test_predictions"
    ].loc[
        primary_regression_results[
            "test_predictions"
        ]["model"].eq(
            primary_regression_results[
                "selected_model_name"
            ]
        )
    ].reset_index(drop=True)
)

plt.figure(figsize=(6, 5))
plt.scatter(
    selected_regression_predictions[
        "actual_mean_monthly_emissions_kg"
    ],
    selected_regression_predictions[
        "predicted_mean_monthly_emissions_kg"
    ],
    alpha=0.7,
)
axis_min = min(
    selected_regression_predictions[
        "actual_mean_monthly_emissions_kg"
    ].min(),
    selected_regression_predictions[
        "predicted_mean_monthly_emissions_kg"
    ].min(),
)
axis_max = max(
    selected_regression_predictions[
        "actual_mean_monthly_emissions_kg"
    ].max(),
    selected_regression_predictions[
        "predicted_mean_monthly_emissions_kg"
    ].max(),
)
plt.plot(
    [axis_min, axis_max],
    [axis_min, axis_max],
    linestyle="--",
)
plt.xlabel("Actual mean monthly emissions (kg CO2)")
plt.ylabel("Predicted mean monthly emissions (kg CO2)")
plt.title("Held-out regression: predicted versus actual")
plt.tight_layout()
plt.show()
plt.close()

classification_calibration = (
    primary_classification_results[
        "calibration_table"
    ]
)

plt.figure(figsize=(6, 5))
plt.plot(
    classification_calibration[
        "mean_uncalibrated_probability"
    ],
    classification_calibration[
        "observed_positive_fraction"
    ],
    marker="o",
)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Mean uncalibrated model probability")
plt.ylabel("Observed high-emission fraction")
plt.title("Selected classifier reliability diagram")
plt.tight_layout()
plt.show()
plt.close()

if not primary_classification_results[
    "correlated_pairs"
].empty:
    print(
        "Caution: strongly correlated predictors can divide or "
        "obscure permutation and SHAP importance."
    )
    display(
        primary_classification_results[
            "correlated_pairs"
        ].round(4)
    )

print(
    "Importance explains fitted-model behaviour, not causation. "
    "Classifier probabilities remain uncalibrated."
)


# Model-agnostic held-out importance for the final teacher-augmented
# regression model. Permutation occurs on the original input columns, so
# teacher-variable missingness indicators remain internal to the pipeline.
teacher_regression_model = survey_augmented_regression_results[
    "selected_model"
]
teacher_regression_X_test = survey_augmented_regression_results[
    "X_test"
]
teacher_regression_y_test = survey_augmented_regression_results[
    "y_test"
]

teacher_regression_permutation_r2 = permutation_importance(
    teacher_regression_model,
    teacher_regression_X_test,
    teacher_regression_y_test,
    scoring=clipped_r2_scorer,
    n_repeats=50,
    random_state=TEACHER_MATCHED_RANDOM_STATE,
    n_jobs=SEARCH_N_JOBS,
)
teacher_regression_permutation_mae = permutation_importance(
    teacher_regression_model,
    teacher_regression_X_test,
    teacher_regression_y_test,
    scoring=clipped_neg_mae_scorer,
    n_repeats=50,
    random_state=TEACHER_MATCHED_RANDOM_STATE,
    n_jobs=SEARCH_N_JOBS,
)

teacher_regression_permutation_importance = (
    pd.DataFrame({
        "feature": survey_augmented_features,
        "heldout_r2_decrease_mean": (
            teacher_regression_permutation_r2.importances_mean
        ),
        "heldout_r2_decrease_std": (
            teacher_regression_permutation_r2.importances_std
        ),
        "heldout_mae_increase_mean_kg_co2": (
            teacher_regression_permutation_mae.importances_mean
        ),
        "heldout_mae_increase_std_kg_co2": (
            teacher_regression_permutation_mae.importances_std
        ),
    })
    .sort_values("heldout_r2_decrease_mean", ascending=False)
    .reset_index(drop=True)
)

print("\nTEACHER-AUGMENTED HELD-OUT REGRESSION PERMUTATION IMPORTANCE")
display(teacher_regression_permutation_importance.round(4))


In [ ]:
# ============================================================
# STEP 15 — EXAMINER-CRITICAL ROBUSTNESS DIAGNOSTICS
# Feature ablation, regression residuals and paired Wave-1 uncertainty
# ============================================================

# -------------------------------------------------------------------------
# A. DEVELOPMENT-ONLY LEAVE-ONE-FEATURE-OUT ABLATION
# -------------------------------------------------------------------------
# Purpose:
#   Test whether predictive performance deteriorates when each behavioural
#   feature is removed. The untouched held-out test set is NOT used here.
#
# Design:
#   - same outer development folds as the primary nested-CV analysis;
#   - same selected model family;
#   - same fold-specific hyperparameters chosen for the full feature model;
#   - one feature removed at a time;
#   - no retuning after removal, so the ablation isolates feature removal
#     rather than allowing a new hyperparameter search to compensate.
#
# This is a diagnostic robustness analysis and is NOT used to alter the
# locked final model, feature set or held-out test conclusions.

ablation_variants = [
    ("all_features", None),
] + [
    (f"without_{feature}", feature)
    for feature in primary_feature_columns
]

feature_ablation_fold_rows = []

# Regression ablation ------------------------------------------------------
regression_selected_name = (
    primary_regression_results["selected_model_name"]
)
regression_outer_rows = (
    primary_regression_results["outer_fold_results"]
    .loc[
        lambda frame: frame["model"].eq(
            regression_selected_name
        )
    ]
    .set_index("outer_fold")
)

regression_ablation_cv = KFold(
    n_splits=primary_regression_results["outer_cv_splits"],
    shuffle=True,
    random_state=primary_split_plan["random_state"],
)

X_regression_development = (
    primary_regression_results["X_development"]
    .reset_index(drop=True)
)
y_regression_development = pd.Series(
    primary_regression_results["y_development"],
    dtype=float,
).reset_index(drop=True)

for outer_fold, (
    train_index,
    valid_index,
) in enumerate(
    regression_ablation_cv.split(
        X_regression_development
    ),
    start=1,
):
    if outer_fold not in regression_outer_rows.index:
        raise RuntimeError(
            "Regression ablation could not recover the "
            f"locked outer-fold parameters for fold {outer_fold}."
        )

    fold_parameters = json.loads(
        regression_outer_rows.loc[
            outer_fold,
            "best_parameters",
        ]
    )

    for variant_name, removed_feature in ablation_variants:
        features = [
            feature
            for feature in primary_feature_columns
            if feature != removed_feature
        ]

        estimator = clone(
            primary_regression_results["selected_model"]
        ).set_params(**fold_parameters)

        estimator.fit(
            X_regression_development.iloc[
                train_index
            ][features],
            y_regression_development.iloc[
                train_index
            ],
        )
        predictions = np.clip(
            np.asarray(
                estimator.predict(
                    X_regression_development.iloc[
                        valid_index
                    ][features]
                ),
                dtype=float,
            ),
            0.0,
            None,
        )

        fold_metrics = regression_metrics(
            y_regression_development.iloc[
                valid_index
            ],
            predictions,
        )

        feature_ablation_fold_rows.append({
            "task": "regression",
            "selected_model_family": regression_selected_name,
            "outer_fold": outer_fold,
            "variant": variant_name,
            "removed_feature": removed_feature,
            "features_retained": len(features),
            "mae_kg_co2": fold_metrics["mae_kg_co2"],
            "r2": fold_metrics["r2"],
            "pr_auc": np.nan,
            "roc_auc": np.nan,
            "target_threshold_kg_co2": np.nan,
        })


# Classification ablation --------------------------------------------------
classification_selected_name = (
    primary_classification_results[
        "selected_model_name"
    ]
)
classification_outer_rows = (
    primary_classification_results[
        "outer_fold_results"
    ]
    .loc[
        lambda frame: frame["model"].eq(
            classification_selected_name
        )
    ]
    .set_index("outer_fold")
)

classification_ablation_cv = KFold(
    n_splits=primary_classification_results[
        "outer_cv_splits"
    ],
    shuffle=True,
    random_state=primary_split_plan["random_state"],
)

X_classification_development = (
    primary_classification_results[
        "X_development"
    ].reset_index(drop=True)
)
emissions_classification_development = pd.Series(
    primary_classification_results[
        "emissions_development"
    ],
    dtype=float,
).reset_index(drop=True)

for outer_fold, (
    train_index,
    valid_index,
) in enumerate(
    classification_ablation_cv.split(
        X_classification_development
    ),
    start=1,
):
    if outer_fold not in classification_outer_rows.index:
        raise RuntimeError(
            "Classification ablation could not recover the "
            f"locked outer-fold parameters for fold {outer_fold}."
        )

    fold_parameters = json.loads(
        classification_outer_rows.loc[
            outer_fold,
            "best_parameters",
        ]
    )

    fold_threshold = float(
        emissions_classification_development.iloc[
            train_index
        ].quantile(HIGH_EMISSION_QUANTILE)
    )

    y_train_fold = (
        emissions_classification_development.iloc[
            train_index
        ] >= fold_threshold
    ).astype(int)
    y_valid_fold = (
        emissions_classification_development.iloc[
            valid_index
        ] >= fold_threshold
    ).astype(int)

    if (
        y_train_fold.nunique() != 2
        or y_valid_fold.nunique() != 2
    ):
        raise RuntimeError(
            "A classification ablation fold contains "
            "only one class."
        )

    for variant_name, removed_feature in ablation_variants:
        features = [
            feature
            for feature in primary_feature_columns
            if feature != removed_feature
        ]

        estimator = clone(
            primary_classification_results[
                "selected_model"
            ]
        ).set_params(**fold_parameters)

        estimator.fit(
            X_classification_development.iloc[
                train_index
            ][features],
            y_train_fold,
        )
        probabilities = estimator.predict_proba(
            X_classification_development.iloc[
                valid_index
            ][features]
        )[:, 1]

        feature_ablation_fold_rows.append({
            "task": "classification",
            "selected_model_family": classification_selected_name,
            "outer_fold": outer_fold,
            "variant": variant_name,
            "removed_feature": removed_feature,
            "features_retained": len(features),
            "mae_kg_co2": np.nan,
            "r2": np.nan,
            "pr_auc": average_precision_score(
                y_valid_fold,
                probabilities,
            ),
            "roc_auc": roc_auc_score(
                y_valid_fold,
                probabilities,
            ),
            "target_threshold_kg_co2": fold_threshold,
        })


feature_ablation_fold_results = pd.DataFrame(
    feature_ablation_fold_rows
)

regression_ablation_summary = (
    feature_ablation_fold_results.loc[
        lambda frame: frame["task"].eq("regression")
    ]
    .groupby(
        [
            "variant",
            "removed_feature",
        ],
        dropna=False,
        observed=True,
    )
    .agg(
        outer_folds=("outer_fold", "nunique"),
        mean_cv_mae_kg_co2=("mae_kg_co2", "mean"),
        std_cv_mae_kg_co2=("mae_kg_co2", "std"),
        mean_cv_r2=("r2", "mean"),
        std_cv_r2=("r2", "std"),
    )
    .reset_index()
)

classification_ablation_summary = (
    feature_ablation_fold_results.loc[
        lambda frame: frame[
            "task"
        ].eq("classification")
    ]
    .groupby(
        [
            "variant",
            "removed_feature",
        ],
        dropna=False,
        observed=True,
    )
    .agg(
        outer_folds=("outer_fold", "nunique"),
        mean_cv_pr_auc=("pr_auc", "mean"),
        std_cv_pr_auc=("pr_auc", "std"),
        mean_cv_roc_auc=("roc_auc", "mean"),
        std_cv_roc_auc=("roc_auc", "std"),
    )
    .reset_index()
)

full_regression_mae = float(
    regression_ablation_summary.loc[
        regression_ablation_summary[
            "variant"
        ].eq("all_features"),
        "mean_cv_mae_kg_co2",
    ].iloc[0]
)
full_regression_r2 = float(
    regression_ablation_summary.loc[
        regression_ablation_summary[
            "variant"
        ].eq("all_features"),
        "mean_cv_r2",
    ].iloc[0]
)
full_classification_pr_auc = float(
    classification_ablation_summary.loc[
        classification_ablation_summary[
            "variant"
        ].eq("all_features"),
        "mean_cv_pr_auc",
    ].iloc[0]
)
full_classification_roc_auc = float(
    classification_ablation_summary.loc[
        classification_ablation_summary[
            "variant"
        ].eq("all_features"),
        "mean_cv_roc_auc",
    ].iloc[0]
)

regression_ablation_summary[
    "mae_increase_vs_full_kg_co2"
] = (
    regression_ablation_summary[
        "mean_cv_mae_kg_co2"
    ]
    - full_regression_mae
)
regression_ablation_summary[
    "r2_loss_vs_full"
] = (
    full_regression_r2
    - regression_ablation_summary[
        "mean_cv_r2"
    ]
)

classification_ablation_summary[
    "pr_auc_loss_vs_full"
] = (
    full_classification_pr_auc
    - classification_ablation_summary[
        "mean_cv_pr_auc"
    ]
)
classification_ablation_summary[
    "roc_auc_loss_vs_full"
] = (
    full_classification_roc_auc
    - classification_ablation_summary[
        "mean_cv_roc_auc"
    ]
)

print(
    "\nDEVELOPMENT-ONLY FEATURE ABLATION — REGRESSION"
)
display(
    regression_ablation_summary.sort_values(
        "mae_increase_vs_full_kg_co2",
        ascending=False,
    ).round(4)
)

print(
    "\nDEVELOPMENT-ONLY FEATURE ABLATION — CLASSIFICATION"
)
display(
    classification_ablation_summary.sort_values(
        "pr_auc_loss_vs_full",
        ascending=False,
    ).round(4)
)

print(
    "Positive MAE increase / R2 loss / PR-AUC loss means "
    "performance deteriorated when that feature was removed. "
    "Ablation is diagnostic only and did not alter the locked model."
)


# -------------------------------------------------------------------------
# B. HELD-OUT REGRESSION RESIDUAL / ERROR DIAGNOSTICS
# -------------------------------------------------------------------------
selected_regression_predictions_for_diagnostics = (
    primary_regression_results[
        "test_predictions"
    ]
    .loc[
        lambda frame: frame["model"].eq(
            primary_regression_results[
                "selected_model_name"
            ]
        )
    ]
    .reset_index(drop=True)
    .copy()
)

selected_regression_predictions_for_diagnostics[
    "absolute_error_kg_co2"
] = selected_regression_predictions_for_diagnostics[
    "residual_actual_minus_predicted_kg"
].abs()

actual_emissions = selected_regression_predictions_for_diagnostics[
    "actual_mean_monthly_emissions_kg"
]
absolute_errors = selected_regression_predictions_for_diagnostics[
    "absolute_error_kg_co2"
]

if (
    actual_emissions.nunique() > 1
    and absolute_errors.nunique() > 1
):
    absolute_error_vs_actual_spearman, (
        absolute_error_vs_actual_p
    ) = spearmanr(
        actual_emissions,
        absolute_errors,
    )
else:
    absolute_error_vs_actual_spearman = np.nan
    absolute_error_vs_actual_p = np.nan

regression_residual_summary = pd.DataFrame([{
    "heldout_households": len(
        selected_regression_predictions_for_diagnostics
    ),
    "mean_residual_actual_minus_predicted_kg": (
        selected_regression_predictions_for_diagnostics[
            "residual_actual_minus_predicted_kg"
        ].mean()
    ),
    "median_residual_actual_minus_predicted_kg": (
        selected_regression_predictions_for_diagnostics[
            "residual_actual_minus_predicted_kg"
        ].median()
    ),
    "mae_kg_co2": absolute_errors.mean(),
    "median_absolute_error_kg_co2": absolute_errors.median(),
    "underprediction_fraction": float(
        (
            selected_regression_predictions_for_diagnostics[
                "residual_actual_minus_predicted_kg"
            ] > 0
        ).mean()
    ),
    "overprediction_fraction": float(
        (
            selected_regression_predictions_for_diagnostics[
                "residual_actual_minus_predicted_kg"
            ] < 0
        ).mean()
    ),
    "spearman_abs_error_vs_actual_emissions": (
        absolute_error_vs_actual_spearman
    ),
    "spearman_abs_error_vs_actual_p_value": (
        absolute_error_vs_actual_p
    ),
}])

quartile_codes = pd.qcut(
    actual_emissions,
    q=4,
    labels=False,
    duplicates="drop",
)

selected_regression_predictions_for_diagnostics[
    "actual_emission_quartile"
] = quartile_codes.astype("Int64") + 1

regression_error_by_actual_quartile = (
    selected_regression_predictions_for_diagnostics
    .groupby(
        "actual_emission_quartile",
        observed=True,
    )
    .agg(
        households=("household_ID", "count"),
        actual_emissions_min_kg=(
            "actual_mean_monthly_emissions_kg",
            "min",
        ),
        actual_emissions_median_kg=(
            "actual_mean_monthly_emissions_kg",
            "median",
        ),
        actual_emissions_max_kg=(
            "actual_mean_monthly_emissions_kg",
            "max",
        ),
        mae_kg_co2=("absolute_error_kg_co2", "mean"),
        median_absolute_error_kg_co2=(
            "absolute_error_kg_co2",
            "median",
        ),
        mean_residual_kg=(
            "residual_actual_minus_predicted_kg",
            "mean",
        ),
    )
    .reset_index()
)

print("\nHELD-OUT REGRESSION RESIDUAL SUMMARY")
display(regression_residual_summary.round(4))
print("\nHELD-OUT ERROR BY ACTUAL-EMISSION QUARTILE")
display(regression_error_by_actual_quartile.round(4))

plt.figure(figsize=(7, 5))
plt.scatter(
    selected_regression_predictions_for_diagnostics[
        "predicted_mean_monthly_emissions_kg"
    ],
    selected_regression_predictions_for_diagnostics[
        "residual_actual_minus_predicted_kg"
    ],
    alpha=0.7,
)
plt.axhline(0.0, linestyle="--")
plt.xlabel("Predicted mean monthly emissions (kg CO2)")
plt.ylabel("Residual: actual - predicted (kg CO2)")
plt.title("Held-out regression residual diagnostic")
plt.tight_layout()
plt.show()
plt.close()

plt.figure(figsize=(7, 5))
plt.bar(
    regression_error_by_actual_quartile[
        "actual_emission_quartile"
    ].astype(str),
    regression_error_by_actual_quartile[
        "mae_kg_co2"
    ],
)
plt.xlabel("Actual-emission quartile (Q1 = lowest)")
plt.ylabel("Held-out MAE (kg CO2)")
plt.title("Held-out regression error by emission level")
plt.tight_layout()
plt.show()
plt.close()


# -------------------------------------------------------------------------
# C. PAIRED BOOTSTRAP UNCERTAINTY FOR WAVE-1 AUGMENTATION
# -------------------------------------------------------------------------
# Both temporal-only and augmented models were selected on development data
# and evaluated on the same prespecified matched holdout. The bootstrap below
# quantifies uncertainty in their held-out PERFORMANCE DIFFERENCE. It is not
# used to select or retune either model.

def selected_regression_test_frame(results, label):
    frame = (
        results["test_predictions"]
        .loc[
            lambda data: data["model"].eq(
                results["selected_model_name"]
            )
        ]
        [
            [
                "household_ID",
                "actual_mean_monthly_emissions_kg",
                "predicted_mean_monthly_emissions_kg",
            ]
        ]
        .copy()
    )
    return frame.rename(columns={
        "predicted_mean_monthly_emissions_kg": (
            f"prediction_{label}"
        )
    })


def selected_classification_test_frame(results, label):
    frame = (
        results["test_predictions"]
        .loc[
            lambda data: data["model"].eq(
                results["selected_model_name"]
            )
        ]
        [
            [
                "household_ID",
                "actual_high_emission",
                "uncalibrated_model_probability",
            ]
        ]
        .copy()
    )
    return frame.rename(columns={
        "uncalibrated_model_probability": (
            f"probability_{label}"
        )
    })


paired_regression_predictions = (
    selected_regression_test_frame(
        survey_temporal_regression_results,
        "temporal",
    )
    .merge(
        selected_regression_test_frame(
            survey_augmented_regression_results,
            "augmented",
        ),
        on=[
            "household_ID",
            "actual_mean_monthly_emissions_kg",
        ],
        how="inner",
        validate="one_to_one",
    )
)

paired_classification_predictions = (
    selected_classification_test_frame(
        survey_temporal_classification_results,
        "temporal",
    )
    .merge(
        selected_classification_test_frame(
            survey_augmented_classification_results,
            "augmented",
        ),
        on=[
            "household_ID",
            "actual_high_emission",
        ],
        how="inner",
        validate="one_to_one",
    )
)

expected_survey_test_households = len(
    survey_split_plan["test_ids"]
)

if (
    len(paired_regression_predictions)
    != expected_survey_test_households
    or len(paired_classification_predictions)
    != expected_survey_test_households
):
    raise RuntimeError(
        "Paired Wave-1 bootstrap did not recover exactly "
        "the prespecified matched held-out households."
    )

paired_bootstrap_rng = np.random.default_rng(
    TEACHER_MATCHED_RANDOM_STATE + 7000
)
paired_bootstrap_rows = []

regression_n = len(paired_regression_predictions)
classification_n = len(
    paired_classification_predictions
)

for bootstrap_iteration in range(
    1,
    BOOTSTRAP_ITERATIONS + 1,
):
    regression_indices = (
        paired_bootstrap_rng.integers(
            0,
            regression_n,
            size=regression_n,
        )
    )
    regression_sample = (
        paired_regression_predictions.iloc[
            regression_indices
        ]
    )

    y_regression = regression_sample[
        "actual_mean_monthly_emissions_kg"
    ].to_numpy(dtype=float)
    temporal_regression_prediction = (
        regression_sample[
            "prediction_temporal"
        ].to_numpy(dtype=float)
    )
    augmented_regression_prediction = (
        regression_sample[
            "prediction_augmented"
        ].to_numpy(dtype=float)
    )

    if np.unique(y_regression).size > 1:
        temporal_r2 = r2_score(
            y_regression,
            temporal_regression_prediction,
        )
        augmented_r2 = r2_score(
            y_regression,
            augmented_regression_prediction,
        )
        paired_bootstrap_rows.append({
            "bootstrap_iteration": bootstrap_iteration,
            "metric": "R2 gain",
            "improvement": (
                augmented_r2 - temporal_r2
            ),
        })

    temporal_mae = mean_absolute_error(
        y_regression,
        temporal_regression_prediction,
    )
    augmented_mae = mean_absolute_error(
        y_regression,
        augmented_regression_prediction,
    )
    paired_bootstrap_rows.append({
        "bootstrap_iteration": bootstrap_iteration,
        "metric": "MAE reduction kg CO2",
        "improvement": (
            temporal_mae - augmented_mae
        ),
    })

    classification_indices = (
        paired_bootstrap_rng.integers(
            0,
            classification_n,
            size=classification_n,
        )
    )
    classification_sample = (
        paired_classification_predictions.iloc[
            classification_indices
        ]
    )
    y_classification = classification_sample[
        "actual_high_emission"
    ].to_numpy(dtype=int)

    if np.unique(y_classification).size == 2:
        temporal_pr_auc = average_precision_score(
            y_classification,
            classification_sample[
                "probability_temporal"
            ].to_numpy(dtype=float),
        )
        augmented_pr_auc = average_precision_score(
            y_classification,
            classification_sample[
                "probability_augmented"
            ].to_numpy(dtype=float),
        )
        paired_bootstrap_rows.append({
            "bootstrap_iteration": bootstrap_iteration,
            "metric": "PR-AUC gain",
            "improvement": (
                augmented_pr_auc - temporal_pr_auc
            ),
        })


survey_paired_bootstrap_long = pd.DataFrame(
    paired_bootstrap_rows
)

observed_r2_temporal = r2_score(
    paired_regression_predictions[
        "actual_mean_monthly_emissions_kg"
    ],
    paired_regression_predictions[
        "prediction_temporal"
    ],
)
observed_r2_augmented = r2_score(
    paired_regression_predictions[
        "actual_mean_monthly_emissions_kg"
    ],
    paired_regression_predictions[
        "prediction_augmented"
    ],
)
observed_mae_temporal = mean_absolute_error(
    paired_regression_predictions[
        "actual_mean_monthly_emissions_kg"
    ],
    paired_regression_predictions[
        "prediction_temporal"
    ],
)
observed_mae_augmented = mean_absolute_error(
    paired_regression_predictions[
        "actual_mean_monthly_emissions_kg"
    ],
    paired_regression_predictions[
        "prediction_augmented"
    ],
)
observed_pr_auc_temporal = average_precision_score(
    paired_classification_predictions[
        "actual_high_emission"
    ],
    paired_classification_predictions[
        "probability_temporal"
    ],
)
observed_pr_auc_augmented = average_precision_score(
    paired_classification_predictions[
        "actual_high_emission"
    ],
    paired_classification_predictions[
        "probability_augmented"
    ],
)

paired_observed = {
    "R2 gain": (
        observed_r2_augmented
        - observed_r2_temporal
    ),
    "MAE reduction kg CO2": (
        observed_mae_temporal
        - observed_mae_augmented
    ),
    "PR-AUC gain": (
        observed_pr_auc_augmented
        - observed_pr_auc_temporal
    ),
}

survey_paired_bootstrap_summary_rows = []

for metric_name, observed_improvement in (
    paired_observed.items()
):
    values = survey_paired_bootstrap_long.loc[
        survey_paired_bootstrap_long[
            "metric"
        ].eq(metric_name),
        "improvement",
    ].dropna().to_numpy(dtype=float)

    if len(values) < int(
        0.90 * BOOTSTRAP_ITERATIONS
    ):
        raise RuntimeError(
            "Too few valid paired bootstrap samples for "
            f"{metric_name}: {len(values)}."
        )

    ci_low, ci_high = np.quantile(
        values,
        [0.025, 0.975],
    )

    survey_paired_bootstrap_summary_rows.append({
        "metric": metric_name,
        "positive_value_means": (
            "temporal + Wave 1 performed better"
        ),
        "observed_improvement": observed_improvement,
        "bootstrap_mean_improvement": values.mean(),
        "ci_95_low": ci_low,
        "ci_95_high": ci_high,
        "valid_bootstrap_samples": len(values),
        "ci_entirely_above_zero": bool(
            ci_low > 0
        ),
    })

survey_paired_bootstrap_summary = pd.DataFrame(
    survey_paired_bootstrap_summary_rows
)

print(
    "\nPAIRED WAVE-1 AUGMENTATION BOOTSTRAP "
    "(SAME HELD-OUT HOUSEHOLDS)"
)
display(
    survey_paired_bootstrap_summary.round(4)
)

print(
    "These intervals quantify uncertainty in the matched "
    "held-out performance difference. They do not make "
    "causal claims about household characteristics."
)

In [ ]:
# ============================================================
# DEVELOPMENT-ONLY LEARNING CURVES
# Locked final regression + classification workflows
# ============================================================
# Purpose:
# - Diagnose whether performance is still improving as the
#   amount of training data increases.
# - Examine train-vs-validation gaps for the LOCKED models.
#
# Methodological rules:
# - Uses DEVELOPMENT households only.
# - Never touches the final held-out test set.
# - Uses the SAME outer CV structure already used in the
#   nested workflow.
# - Uses the hyperparameters selected inside each outer fold.
# - Does NOT retune models.
# - Does NOT change the final selected models.
# - For classification, the high-emission Q75 threshold is
#   derived from the OUTER TRAINING fold only and is held
#   constant across training fractions within that fold.
# ============================================================

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    average_precision_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# 1. Required data
# ------------------------------------------------------------

X_reg = (
    primary_regression_results["X_development"]
    .reset_index(drop=True)
)

y_reg = pd.Series(
    primary_regression_results["y_development"],
    dtype=float,
).reset_index(drop=True)


X_cls = (
    primary_classification_results["X_development"]
    .reset_index(drop=True)
)

emissions_cls = pd.Series(
    primary_classification_results[
        "emissions_development"
    ],
    dtype=float,
).reset_index(drop=True)


if len(X_reg) != len(y_reg):
    raise RuntimeError(
        "Regression development X/y length mismatch."
    )

if len(X_cls) != len(emissions_cls):
    raise RuntimeError(
        "Classification development X/outcome length mismatch."
    )


# ------------------------------------------------------------
# 2. Recover the LOCKED selected family and fold parameters
# ------------------------------------------------------------

regression_selected_name = (
    primary_regression_results["selected_model_name"]
)

classification_selected_name = (
    primary_classification_results["selected_model_name"]
)


regression_outer_rows = (
    primary_regression_results["outer_fold_results"]
    .loc[
        lambda df:
        df["model"].eq(regression_selected_name)
    ]
    .set_index("outer_fold")
)


classification_outer_rows = (
    primary_classification_results[
        "outer_fold_results"
    ]
    .loc[
        lambda df:
        df["model"].eq(classification_selected_name)
    ]
    .set_index("outer_fold")
)


print("=" * 80)
print("DEVELOPMENT-ONLY LEARNING CURVES")
print("=" * 80)

print(
    "Regression:",
    regression_selected_name
)

print(
    "Classification:",
    classification_selected_name
)

print(
    "Held-out test used:",
    False
)


# ------------------------------------------------------------
# 3. Learning-curve configuration
# ------------------------------------------------------------

TRAINING_FRACTIONS = [
    0.25,
    0.50,
    0.75,
    1.00,
]

SUBSET_REPEATS = 3

N_SPLITS = int(
    primary_regression_results["outer_cv_splits"]
)

learning_cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=primary_split_plan[
        "random_state"
    ],
)


def choose_training_subset(
    train_indices,
    fraction,
    random_state,
):
    """
    Randomly choose a subset from OUTER TRAINING households.

    Validation households are never sampled into training.
    """

    train_indices = np.asarray(
        train_indices,
        dtype=int,
    )

    if fraction >= 1.0:
        return train_indices.copy()

    subset_size = max(
        20,
        int(
            np.floor(
                len(train_indices) * fraction
            )
        ),
    )

    rng = np.random.default_rng(
        random_state
    )

    return np.sort(
        rng.choice(
            train_indices,
            size=subset_size,
            replace=False,
        )
    )


# ------------------------------------------------------------
# 4. REGRESSION learning curve
# ------------------------------------------------------------

regression_learning_rows = []


for outer_fold, (
    outer_train_index,
    outer_valid_index,
) in enumerate(
    learning_cv.split(X_reg),
    start=1,
):

    if outer_fold not in regression_outer_rows.index:
        raise RuntimeError(
            "Could not recover regression fold "
            f"{outer_fold} parameters."
        )

    fold_parameters = json.loads(
        regression_outer_rows.loc[
            outer_fold,
            "best_parameters",
        ]
    )

    for fraction in TRAINING_FRACTIONS:

        repeat_count = (
            1
            if fraction == 1.0
            else SUBSET_REPEATS
        )

        for repeat in range(
            1,
            repeat_count + 1,
        ):

            subset_indices = (
                choose_training_subset(
                    outer_train_index,
                    fraction,
                    random_state=(
                        RANDOM_STATE
                        + outer_fold * 100
                        + repeat * 10
                        + int(fraction * 100)
                    ),
                )
            )

            estimator = clone(
                primary_regression_results[
                    "selected_model"
                ]
            ).set_params(
                **fold_parameters
            )

            estimator.fit(
                X_reg.iloc[
                    subset_indices
                ][primary_feature_columns],
                y_reg.iloc[
                    subset_indices
                ],
            )

            # Training predictions
            train_predictions = np.clip(
                np.asarray(
                    estimator.predict(
                        X_reg.iloc[
                            subset_indices
                        ][
                            primary_feature_columns
                        ]
                    ),
                    dtype=float,
                ),
                0.0,
                None,
            )

            # Validation predictions
            validation_predictions = np.clip(
                np.asarray(
                    estimator.predict(
                        X_reg.iloc[
                            outer_valid_index
                        ][
                            primary_feature_columns
                        ]
                    ),
                    dtype=float,
                ),
                0.0,
                None,
            )

            train_mae = mean_absolute_error(
                y_reg.iloc[
                    subset_indices
                ],
                train_predictions,
            )

            validation_mae = mean_absolute_error(
                y_reg.iloc[
                    outer_valid_index
                ],
                validation_predictions,
            )

            train_r2 = r2_score(
                y_reg.iloc[
                    subset_indices
                ],
                train_predictions,
            )

            validation_r2 = r2_score(
                y_reg.iloc[
                    outer_valid_index
                ],
                validation_predictions,
            )

            regression_learning_rows.append({
                "outer_fold": outer_fold,
                "training_fraction": fraction,
                "repeat": repeat,
                "training_households": (
                    len(subset_indices)
                ),
                "validation_households": (
                    len(outer_valid_index)
                ),
                "train_mae_kg_co2": train_mae,
                "validation_mae_kg_co2": (
                    validation_mae
                ),
                "mae_gap_validation_minus_train": (
                    validation_mae
                    - train_mae
                ),
                "train_r2": train_r2,
                "validation_r2": validation_r2,
                "r2_gap_train_minus_validation": (
                    train_r2
                    - validation_r2
                ),
            })


regression_learning_raw = pd.DataFrame(
    regression_learning_rows
)


regression_learning_summary = (
    regression_learning_raw
    .groupby(
        "training_fraction",
        as_index=False,
    )
    .agg(
        mean_training_households=(
            "training_households",
            "mean",
        ),
        train_mae_mean=(
            "train_mae_kg_co2",
            "mean",
        ),
        train_mae_std=(
            "train_mae_kg_co2",
            "std",
        ),
        validation_mae_mean=(
            "validation_mae_kg_co2",
            "mean",
        ),
        validation_mae_std=(
            "validation_mae_kg_co2",
            "std",
        ),
        train_r2_mean=(
            "train_r2",
            "mean",
        ),
        validation_r2_mean=(
            "validation_r2",
            "mean",
        ),
        mae_gap_mean=(
            "mae_gap_validation_minus_train",
            "mean",
        ),
        r2_gap_mean=(
            "r2_gap_train_minus_validation",
            "mean",
        ),
    )
)


# ------------------------------------------------------------
# 5. CLASSIFICATION learning curve
# ------------------------------------------------------------

classification_learning_rows = []


classification_cv = KFold(
    n_splits=int(
        primary_classification_results[
            "outer_cv_splits"
        ]
    ),
    shuffle=True,
    random_state=primary_split_plan[
        "random_state"
    ],
)


for outer_fold, (
    outer_train_index,
    outer_valid_index,
) in enumerate(
    classification_cv.split(X_cls),
    start=1,
):

    if (
        outer_fold
        not in classification_outer_rows.index
    ):
        raise RuntimeError(
            "Could not recover classification fold "
            f"{outer_fold} parameters."
        )

    fold_parameters = json.loads(
        classification_outer_rows.loc[
            outer_fold,
            "best_parameters",
        ]
    )

    # IMPORTANT:
    # Q75 is derived using OUTER TRAINING data only.
    # Validation outcomes are not used.
    fold_threshold = float(
        emissions_cls.iloc[
            outer_train_index
        ].quantile(
            HIGH_EMISSION_QUANTILE
        )
    )

    y_outer_train = (
        emissions_cls.iloc[
            outer_train_index
        ]
        >= fold_threshold
    ).astype(int)

    y_outer_valid = (
        emissions_cls.iloc[
            outer_valid_index
        ]
        >= fold_threshold
    ).astype(int)

    if (
        y_outer_train.nunique() != 2
        or y_outer_valid.nunique() != 2
    ):
        raise RuntimeError(
            "A learning-curve outer fold contains "
            "only one classification class."
        )

    for fraction in TRAINING_FRACTIONS:

        repeat_count = (
            1
            if fraction == 1.0
            else SUBSET_REPEATS
        )

        for repeat in range(
            1,
            repeat_count + 1,
        ):

            subset_indices = (
                choose_training_subset(
                    outer_train_index,
                    fraction,
                    random_state=(
                        RANDOM_STATE
                        + 5000
                        + outer_fold * 100
                        + repeat * 10
                        + int(fraction * 100)
                    ),
                )
            )

            y_subset = (
                emissions_cls.iloc[
                    subset_indices
                ]
                >= fold_threshold
            ).astype(int)

            # Extremely defensive check.
            if y_subset.nunique() != 2:
                continue

            estimator = clone(
                primary_classification_results[
                    "selected_model"
                ]
            ).set_params(
                **fold_parameters
            )

            estimator.fit(
                X_cls.iloc[
                    subset_indices
                ][primary_feature_columns],
                y_subset,
            )

            train_probability = (
                estimator.predict_proba(
                    X_cls.iloc[
                        subset_indices
                    ][
                        primary_feature_columns
                    ]
                )[:, 1]
            )

            validation_probability = (
                estimator.predict_proba(
                    X_cls.iloc[
                        outer_valid_index
                    ][
                        primary_feature_columns
                    ]
                )[:, 1]
            )

            train_pr_auc = (
                average_precision_score(
                    y_subset,
                    train_probability,
                )
            )

            validation_pr_auc = (
                average_precision_score(
                    y_outer_valid,
                    validation_probability,
                )
            )

            train_roc_auc = roc_auc_score(
                y_subset,
                train_probability,
            )

            validation_roc_auc = (
                roc_auc_score(
                    y_outer_valid,
                    validation_probability,
                )
            )

            classification_learning_rows.append({
                "outer_fold": outer_fold,
                "training_fraction": fraction,
                "repeat": repeat,
                "training_households": (
                    len(subset_indices)
                ),
                "validation_households": (
                    len(outer_valid_index)
                ),
                "target_threshold_kg_co2": (
                    fold_threshold
                ),
                "train_pr_auc": train_pr_auc,
                "validation_pr_auc": (
                    validation_pr_auc
                ),
                "pr_auc_gap_train_minus_validation": (
                    train_pr_auc
                    - validation_pr_auc
                ),
                "train_roc_auc": train_roc_auc,
                "validation_roc_auc": (
                    validation_roc_auc
                ),
                "roc_auc_gap_train_minus_validation": (
                    train_roc_auc
                    - validation_roc_auc
                ),
            })


classification_learning_raw = pd.DataFrame(
    classification_learning_rows
)


classification_learning_summary = (
    classification_learning_raw
    .groupby(
        "training_fraction",
        as_index=False,
    )
    .agg(
        mean_training_households=(
            "training_households",
            "mean",
        ),
        train_pr_auc_mean=(
            "train_pr_auc",
            "mean",
        ),
        train_pr_auc_std=(
            "train_pr_auc",
            "std",
        ),
        validation_pr_auc_mean=(
            "validation_pr_auc",
            "mean",
        ),
        validation_pr_auc_std=(
            "validation_pr_auc",
            "std",
        ),
        pr_auc_gap_mean=(
            "pr_auc_gap_train_minus_validation",
            "mean",
        ),
        train_roc_auc_mean=(
            "train_roc_auc",
            "mean",
        ),
        validation_roc_auc_mean=(
            "validation_roc_auc",
            "mean",
        ),
        roc_auc_gap_mean=(
            "roc_auc_gap_train_minus_validation",
            "mean",
        ),
    )
)


# ------------------------------------------------------------
# 6. Display summaries
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("REGRESSION LEARNING CURVE")
print("=" * 80)

display(
    regression_learning_summary.round(4)
)


print("\n" + "=" * 80)
print("CLASSIFICATION LEARNING CURVE")
print("=" * 80)

display(
    classification_learning_summary.round(4)
)


# ------------------------------------------------------------
# 7. Plot regression learning curve
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    regression_learning_summary[
        "mean_training_households"
    ],
    regression_learning_summary[
        "train_mae_mean"
    ],
    marker="o",
    label="Training MAE",
)

ax.plot(
    regression_learning_summary[
        "mean_training_households"
    ],
    regression_learning_summary[
        "validation_mae_mean"
    ],
    marker="o",
    label="Validation MAE",
)

ax.set_xlabel(
    "Number of training households"
)

ax.set_ylabel(
    "MAE (kg CO₂/month)"
)

ax.set_title(
    "Development-Only Regression Learning Curve"
)

ax.legend()

ax.grid(
    alpha=0.25
)

plt.tight_layout()

regression_curve_path = (
    OUTPUT_DIR
    / "development_learning_curve_regression.png"
)

plt.savefig(
    regression_curve_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# 8. Plot classification learning curve
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    classification_learning_summary[
        "mean_training_households"
    ],
    classification_learning_summary[
        "train_pr_auc_mean"
    ],
    marker="o",
    label="Training PR-AUC",
)

ax.plot(
    classification_learning_summary[
        "mean_training_households"
    ],
    classification_learning_summary[
        "validation_pr_auc_mean"
    ],
    marker="o",
    label="Validation PR-AUC",
)

ax.set_xlabel(
    "Number of training households"
)

ax.set_ylabel(
    "PR-AUC"
)

ax.set_title(
    "Development-Only High-Emission Classification Learning Curve"
)

ax.legend()

ax.grid(
    alpha=0.25
)

plt.tight_layout()

classification_curve_path = (
    OUTPUT_DIR
    / "development_learning_curve_classification.png"
)

plt.savefig(
    classification_curve_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# 9. Save numerical outputs
# ------------------------------------------------------------

regression_learning_raw.to_csv(
    OUTPUT_DIR
    / "development_learning_curve_regression_raw.csv",
    index=False,
)

regression_learning_summary.to_csv(
    OUTPUT_DIR
    / "development_learning_curve_regression_summary.csv",
    index=False,
)

classification_learning_raw.to_csv(
    OUTPUT_DIR
    / "development_learning_curve_classification_raw.csv",
    index=False,
)

classification_learning_summary.to_csv(
    OUTPUT_DIR
    / "development_learning_curve_classification_summary.csv",
    index=False,
)


# ------------------------------------------------------------
# 10. Final full-data diagnostic
# ------------------------------------------------------------

final_regression_curve = (
    regression_learning_summary
    .sort_values(
        "training_fraction"
    )
    .iloc[-1]
)

final_classification_curve = (
    classification_learning_summary
    .sort_values(
        "training_fraction"
    )
    .iloc[-1]
)


print("\n" + "=" * 80)
print("FINAL LEARNING-CURVE DIAGNOSTIC")
print("=" * 80)

print(
    "Regression final validation MAE:",
    round(
        float(
            final_regression_curve[
                "validation_mae_mean"
            ]
        ),
        3,
    ),
    "kg CO2/month",
)

print(
    "Regression final train-validation MAE gap:",
    round(
        float(
            final_regression_curve[
                "mae_gap_mean"
            ]
        ),
        3,
    ),
)

print(
    "Classification final validation PR-AUC:",
    round(
        float(
            final_classification_curve[
                "validation_pr_auc_mean"
            ]
        ),
        4,
    ),
)

print(
    "Classification final train-validation PR-AUC gap:",
    round(
        float(
            final_classification_curve[
                "pr_auc_gap_mean"
            ]
        ),
        4,
    ),
)

print(
    "\nIMPORTANT: These are secondary development-only "
    "diagnostics. They do NOT replace the nested-CV or "
    "held-out test performance reported by the final workflow."
)

print(
    "No held-out test household was used in this learning-curve analysis."
)

In [ ]:
# ============================================================
# STEP 16 — RECONSTRUCTION, COHORT AND TARGET SENSITIVITY
# ============================================================

# A. Cross-month gap-rule sensitivity. Every rule uses the same
# minimum-month, coverage and actual-reading boundary requirements.
sensitivity_household_rows = []
gap_cohort_rows = []

for maximum_gap in SENSITIVITY_MAX_GAP_HOURS:
    eligible = sixhour_monthly_all.loc[
        sixhour_monthly_all[
            "cross_month_max_gap_hours"
        ].eq(float(maximum_gap))
        & sixhour_monthly_all[
            "month_quality_pass"
        ]
    ].copy()

    eligibility_audit = build_cohort_eligibility_audit(
        eligible,
        minimum_valid_months=PRIMARY_MIN_VALID_MONTHS,
    )

    selected_ids = set(
        eligibility_audit.loc[
            eligibility_audit[
                "eligible_primary_cohort"
            ],
            "household_ID",
        ]
    )

    selected = eligible.loc[
        eligible["household_ID"].isin(selected_ids)
    ].copy()

    household_summary = (
        selected.groupby(
            "household_ID",
            as_index=False,
            observed=True,
        )
        .agg(
            observed_months=("month_period", "nunique"),
            mean_monthly_kwh=(
                "reconstructed_kwh",
                "mean",
            ),
            minimum_coverage=(
                "coverage_fraction",
                "min",
            ),
            maximum_start_boundary_gap_hours=(
                "start_boundary_gap_hours",
                "max",
            ),
            maximum_end_boundary_gap_hours=(
                "end_boundary_gap_hours",
                "max",
            ),
        )
    )

    household_summary[
        "cross_month_max_gap_hours"
    ] = float(maximum_gap)

    sensitivity_household_rows.append(
        household_summary
    )

    gap_cohort_rows.append({
        "cross_month_max_gap_hours": float(maximum_gap),
        "minimum_valid_months_required": (
            PRIMARY_MIN_VALID_MONTHS
        ),
        "eligible_households": len(household_summary),
        "median_observed_months": (
            household_summary[
                "observed_months"
            ].median()
            if len(household_summary)
            else np.nan
        ),
        "median_mean_monthly_kwh": (
            household_summary[
                "mean_monthly_kwh"
            ].median()
            if len(household_summary)
            else np.nan
        ),
    })

gap_household_sensitivity = pd.concat(
    sensitivity_household_rows,
    ignore_index=True,
)

gap_cohort_summary = pd.DataFrame(
    gap_cohort_rows
)

gap_comparison = (
    gap_household_sensitivity.pivot(
        index="household_ID",
        columns="cross_month_max_gap_hours",
        values="mean_monthly_kwh",
    )
)

for required_gap_column in [
    float(value)
    for value in SENSITIVITY_MAX_GAP_HOURS
]:
    if required_gap_column not in gap_comparison.columns:
        gap_comparison[required_gap_column] = np.nan

gap_summary_rows = []

for comparison_gap in SENSITIVITY_MAX_GAP_HOURS:
    comparison_gap = float(comparison_gap)

    common = gap_comparison[
        [
            float(PRIMARY_MAX_GAP_HOURS),
            comparison_gap,
        ]
    ].dropna()

    if comparison_gap == float(PRIMARY_MAX_GAP_HOURS):
        difference = pd.Series(
            np.zeros(len(common)),
            index=common.index,
            dtype=float,
        )
    else:
        difference = (
            common[comparison_gap]
            - common[float(PRIMARY_MAX_GAP_HOURS)]
        )

    gap_summary_rows.append({
        "comparison_gap_hours": comparison_gap,
        "reference_gap_hours": float(
            PRIMARY_MAX_GAP_HOURS
        ),
        "common_households": len(common),
        "pearson_r": (
            pearsonr(
                common[float(PRIMARY_MAX_GAP_HOURS)],
                common[comparison_gap],
            )[0]
            if (
                len(common) >= 10
                and comparison_gap
                != float(PRIMARY_MAX_GAP_HOURS)
                and common[
                    float(PRIMARY_MAX_GAP_HOURS)
                ].nunique() > 1
                and common[comparison_gap].nunique() > 1
            )
            else (
                1.0
                if len(common)
                and comparison_gap
                == float(PRIMARY_MAX_GAP_HOURS)
                else np.nan
            )
        ),
        "spearman_r": (
            spearmanr(
                common[float(PRIMARY_MAX_GAP_HOURS)],
                common[comparison_gap],
            )[0]
            if (
                len(common) >= 10
                and comparison_gap
                != float(PRIMARY_MAX_GAP_HOURS)
                and common[
                    float(PRIMARY_MAX_GAP_HOURS)
                ].nunique() > 1
                and common[comparison_gap].nunique() > 1
            )
            else (
                1.0
                if len(common)
                and comparison_gap
                == float(PRIMARY_MAX_GAP_HOURS)
                else np.nan
            )
        ),
        "median_difference_kwh_comparison_minus_reference": (
            difference.median()
            if len(difference)
            else np.nan
        ),
        "median_absolute_difference_kwh": (
            difference.abs().median()
            if len(difference)
            else np.nan
        ),
    })

gap_sensitivity_summary = pd.DataFrame(
    gap_summary_rows
)

display(gap_cohort_summary)
display(gap_sensitivity_summary.round(4))


# B. Monthly-coverage threshold sensitivity, holding the selected
# cross-month rule, actual-reading boundary requirement and minimum
# valid-month rule fixed.
coverage_sensitivity_rows = []

primary_gap_months = sixhour_monthly_all.loc[
    sixhour_monthly_all[
        "cross_month_max_gap_hours"
    ].eq(float(PRIMARY_MAX_GAP_HOURS))
].copy()

for coverage_threshold in [0.95, 0.98, 0.99]:
    eligible = primary_gap_months.loc[
        primary_gap_months[
            "coverage_fraction"
        ].ge(coverage_threshold)
        & primary_gap_months[
            "start_boundary_gap_hours"
        ].le(MAX_BOUNDARY_GAP_HOURS)
        & primary_gap_months[
            "end_boundary_gap_hours"
        ].le(MAX_BOUNDARY_GAP_HOURS)
    ].copy()

    eligibility_audit = build_cohort_eligibility_audit(
        eligible,
        minimum_valid_months=PRIMARY_MIN_VALID_MONTHS,
    )

    selected_ids = set(
        eligibility_audit.loc[
            eligibility_audit[
                "eligible_primary_cohort"
            ],
            "household_ID",
        ]
    )

    selected = eligible.loc[
        eligible["household_ID"].isin(
            selected_ids
        )
    ].copy()

    household_values = (
        selected.groupby(
            "household_ID",
            as_index=False,
            observed=True,
        )
        .agg(
            observed_months=("month_period", "nunique"),
            mean_monthly_kwh=(
                "reconstructed_kwh",
                "mean",
            )
        )
    )

    coverage_sensitivity_rows.append({
        "minimum_month_coverage": coverage_threshold,
        "minimum_valid_months_required": (
            PRIMARY_MIN_VALID_MONTHS
        ),
        "eligible_households": len(
            household_values
        ),
        "median_observed_months": (
            household_values[
                "observed_months"
            ].median()
            if len(household_values)
            else np.nan
        ),
        "median_mean_monthly_kwh": (
            household_values[
                "mean_monthly_kwh"
            ].median()
            if len(household_values)
            else np.nan
        ),
        "mean_mean_monthly_kwh": (
            household_values[
                "mean_monthly_kwh"
            ].mean()
            if len(household_values)
            else np.nan
        ),
    })

coverage_sensitivity_summary = pd.DataFrame(
    coverage_sensitivity_rows
)

display(coverage_sensitivity_summary.round(4))


# C. Minimum-valid-month sensitivity under the selected reconstruction
# and quality rules. The primary rule remains five months.
minimum_month_sensitivity_rows = []

for minimum_months in [3, 4, 5, 6, 7]:
    eligibility_audit = build_cohort_eligibility_audit(
        primary_monthly_eligible,
        minimum_valid_months=minimum_months,
    )

    selected_ids = set(
        eligibility_audit.loc[
            eligibility_audit[
                "eligible_primary_cohort"
            ],
            "household_ID",
        ]
    )

    selected = primary_monthly_eligible.loc[
        primary_monthly_eligible[
            "household_ID"
        ].isin(selected_ids)
    ].copy()

    household_values = (
        selected.groupby(
            "household_ID",
            as_index=False,
            observed=True,
        )
        .agg(
            observed_months=("month_period", "nunique"),
            mean_monthly_kwh=(
                "reconstructed_kwh",
                "mean",
            ),
        )
    )

    minimum_month_sensitivity_rows.append({
        "minimum_valid_months_required": minimum_months,
        "eligible_households": len(household_values),
        "median_observed_months": (
            household_values[
                "observed_months"
            ].median()
            if len(household_values)
            else np.nan
        ),
        "median_mean_monthly_kwh": (
            household_values[
                "mean_monthly_kwh"
            ].median()
            if len(household_values)
            else np.nan
        ),
        "mean_mean_monthly_kwh": (
            household_values[
                "mean_monthly_kwh"
            ].mean()
            if len(household_values)
            else np.nan
        ),
        "is_primary_rule": (
            minimum_months
            == PRIMARY_MIN_VALID_MONTHS
        ),
    })

minimum_month_sensitivity_summary = pd.DataFrame(
    minimum_month_sensitivity_rows
)

display(minimum_month_sensitivity_summary.round(4))


# D. High-emission target sensitivity. The household split is held
# fixed. Each emission threshold and decision threshold are estimated
# from development households only.
quantile_sensitivity_rows = []
quantile_test_labels = {}

indexed_primary = primary_model_table.set_index(
    "household_ID",
    drop=False,
)

development_ids = primary_split_plan["development_ids"]
test_ids = primary_split_plan["test_ids"]

X_development_fixed = indexed_primary.loc[
    development_ids,
    primary_feature_columns,
].reset_index(drop=True)

X_test_fixed = indexed_primary.loc[
    test_ids,
    primary_feature_columns,
].reset_index(drop=True)

emissions_development_fixed = indexed_primary.loc[
    development_ids,
    "mean_monthly_emissions_kg",
].reset_index(drop=True)

emissions_test_fixed = indexed_primary.loc[
    test_ids,
    "mean_monthly_emissions_kg",
].reset_index(drop=True)

sensitivity_cv = StratifiedKFold(
    n_splits=SENSITIVITY_CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

for target_quantile in [0.70, 0.75, 0.80]:
    threshold_emissions = float(
        emissions_development_fixed.quantile(
            target_quantile
        )
    )

    y_development_sensitivity = (
        emissions_development_fixed
        >= threshold_emissions
    ).astype(int)

    y_test_sensitivity = (
        emissions_test_fixed
        >= threshold_emissions
    ).astype(int)

    sensitivity_model_template = clone(
        primary_classification_results[
            "selected_model"
        ]
    )

    oof_probabilities = cross_val_predict(
        sensitivity_model_template,
        X_development_fixed,
        y_development_sensitivity,
        cv=sensitivity_cv,
        method="predict_proba",
        n_jobs=SEARCH_N_JOBS,
    )[:, 1]

    decision_threshold, _ = (
        select_development_threshold(
            y_development_sensitivity,
            oof_probabilities,
        )
    )

    fitted_sensitivity_model = clone(
        sensitivity_model_template
    )
    fitted_sensitivity_model.fit(
        X_development_fixed,
        y_development_sensitivity,
    )

    sensitivity_probabilities = (
        fitted_sensitivity_model.predict_proba(
            X_test_fixed
        )[:, 1]
    )

    sensitivity_metrics = classification_metrics(
        y_test_sensitivity,
        sensitivity_probabilities,
        threshold=decision_threshold,
    )

    quantile_test_labels[target_quantile] = (
        y_test_sensitivity.to_numpy()
    )

    quantile_sensitivity_rows.append({
        "target_quantile": target_quantile,
        "threshold_kg_co2_from_development_only": (
            threshold_emissions
        ),
        "threshold_kwh_from_development_only": (
            threshold_emissions
            / GRID_FACTOR_KG_CO2_PER_KWH
        ),
        "decision_threshold_from_development_oof": (
            decision_threshold
        ),
        "development_positive_rate": (
            y_development_sensitivity.mean()
        ),
        "test_positive_rate": (
            y_test_sensitivity.mean()
        ),
        **sensitivity_metrics,
    })

quantile_sensitivity_summary = pd.DataFrame(
    quantile_sensitivity_rows
)

reference_labels = quantile_test_labels[0.75]

quantile_sensitivity_summary[
    "test_label_kappa_vs_q75"
] = [
    cohen_kappa_score(
        reference_labels,
        quantile_test_labels[row_quantile],
    )
    for row_quantile in quantile_sensitivity_summary[
        "target_quantile"
    ]
]

display(quantile_sensitivity_summary.round(4))


# E. Transparent arithmetic scenario only.
reduction_scenario = household_carbon[
    [
        "household_ID",
        "mean_monthly_kwh",
        "mean_monthly_emissions_kg",
    ]
].copy()

reduction_scenario[
    "illustrative_reduction_fraction"
] = 0.10

reduction_scenario["illustrative_reduction_kwh"] = (
    reduction_scenario["mean_monthly_kwh"] * 0.10
)

reduction_scenario[
    "illustrative_reduction_kg_co2"
] = (
    reduction_scenario["illustrative_reduction_kwh"]
    * GRID_FACTOR_KG_CO2_PER_KWH
)

reduction_scenario[
    "emissions_after_illustrative_reduction_kg"
] = (
    reduction_scenario["mean_monthly_emissions_kg"]
    - reduction_scenario[
        "illustrative_reduction_kg_co2"
    ]
)

reduction_scenario["interpretation"] = (
    "Illustrative 10% reduction in total electricity only; "
    "not a causal or model-derived household saving."
)

fixed_factor_timing_check = pd.DataFrame([{
    "total_kwh_before_shift": 8.0,
    "total_kwh_after_shift": 8.0,
    "emissions_before_kg": (
        8.0 * GRID_FACTOR_KG_CO2_PER_KWH
    ),
    "emissions_after_kg": (
        8.0 * GRID_FACTOR_KG_CO2_PER_KWH
    ),
    "timing_only_carbon_saving_kg": 0.0,
    "conclusion": (
        "A fixed factor cannot support a carbon-saving claim "
        "from shifting unchanged kWh to another time."
    ),
}])

display(reduction_scenario.head())
display(fixed_factor_timing_check)

print(
    "No arbitrary kg-CO2-to-LKR conversion is created."
)


In [ ]:
# ============================================================
# STEP 17 — FINAL METHODOLOGICAL RELEASE GATE
# ============================================================

classification_split = (
    primary_classification_results["split_audit"].iloc[0]
)
regression_split = (
    primary_regression_results["split_audit"].iloc[0]
)


regression_best_params = (
    primary_regression_results[
        "fitted_searches"
    ][
        primary_regression_results[
            "selected_model_name"
        ]
    ].best_params_
)

classification_best_params = (
    primary_classification_results[
        "fitted_searches"
    ][
        primary_classification_results[
            "selected_model_name"
        ]
    ].best_params_
)

survey_pairing_valid = True

if (
    survey_temporal_regression_results is not None
    and survey_augmented_regression_results is not None
    and survey_temporal_classification_results is not None
    and survey_augmented_classification_results is not None
):
    survey_pairing_valid = bool(
        survey_temporal_regression_results[
            "id_development"
        ].equals(
            survey_augmented_regression_results[
                "id_development"
            ]
        )
        and survey_temporal_regression_results[
            "id_test"
        ].equals(
            survey_augmented_regression_results[
                "id_test"
            ]
        )
        and survey_temporal_classification_results[
            "id_development"
        ].equals(
            survey_augmented_classification_results[
                "id_development"
            ]
        )
        and survey_temporal_classification_results[
            "id_test"
        ].equals(
            survey_augmented_classification_results[
                "id_test"
            ]
        )
    )

teacher_model_results_complete = bool(
    survey_temporal_regression_results is not None
    and survey_augmented_regression_results is not None
    and survey_temporal_classification_results is not None
    and survey_augmented_classification_results is not None
)

teacher_features_complete = bool(
    survey_model_table is not None
    and survey_feature_columns == TEACHER_SOCIO_FEATURES
    and set(TEACHER_SOCIO_FEATURES).issubset(
        survey_model_table.columns
    )
)

teacher_augmented_feature_set_is_leakage_safe = bool(
    teacher_features_complete
    and set(survey_augmented_features)
    == set(primary_feature_columns + TEACHER_SOCIO_FEATURES)
    and not any(
        forbidden in feature.lower()
        for feature in survey_augmented_features
        for forbidden in FORBIDDEN_PREDICTOR_TERMS
    )
)

def regression_searches_use_clipped_scorers(results):
    """Verify every fitted regression GridSearchCV uses the locked scorers."""
    searches = results.get("fitted_searches", {})
    if not searches:
        return False

    for search in searches.values():
        scoring = getattr(search, "scoring", None)
        if not isinstance(scoring, dict):
            return False
        if scoring.get("neg_mae") is not clipped_neg_mae_scorer:
            return False
        if scoring.get("neg_mse") is not clipped_neg_mse_scorer:
            return False
        if scoring.get("r2") is not clipped_r2_scorer:
            return False

    return True


def classifier_selected_by_outer_fold_pr_auc(results):
    """Recompute the declared family winner from fold-level PR-AUC only."""
    table = results["model_selection"].copy()
    expected = (
        table.sort_values(
            [
                "outer_fold_pr_auc_mean",
                "outer_fold_pr_auc_std",
                "model",
            ],
            ascending=[False, True, True],
        )
        .iloc[0]["model"]
    )
    return bool(
        results["selected_model_name"] == expected
        and results["model_family_selection_statistic"]
        == "mean_outer_fold_pr_auc"
    )


classification_model_names = set(
    primary_classification_results[
        "model_selection"
    ]["model"]
)

regression_model_names = set(
    primary_regression_results[
        "model_selection"
    ]["model"]
)

release_checks = {
    "exactly_five_sixhour_files": (
        len(sixhour_files) == EXPECTED_SIXHOUR_FILE_COUNT
    ),
    "locked_population_is_2227": (
        len(cluster_df) == EXPECTED_LOCKED_HOUSEHOLDS
    ),
    "unclipped_feature_population_is_2227": (
        len(raw_feature_df) == EXPECTED_LOCKED_HOUSEHOLDS
    ),
    "unclipped_and_locked_household_ids_match": (
        raw_locked_id_match
    ),
    "locked_archetype_counts_match_master": (
        observed_cluster_counts == EXPECTED_CLUSTER_COUNTS
    ),
    "no_conflicting_sixhour_duplicates": (
        conflicting_duplicate_count == 0
    ),
    "synthetic_full_month_test_passed": (
        bool(synthetic_may["month_quality_pass"])
        and np.isclose(
            synthetic_may["reconstructed_kwh"],
            744.0,
        )
    ),
    "synthetic_long_boundary_gap_is_rejected": (
        long_cross_boundary_test_passed
    ),
    "primary_allocation_conserves_energy": (
        np.isclose(
            allocation_audit_table.loc[
                allocation_audit_table[
                    "cross_month_max_gap_hours"
                ].eq(PRIMARY_MAX_GAP_HOURS),
                "conservation_difference_kwh",
            ].iloc[0],
            0.0,
            atol=1e-6,
        )
    ),
    "sixhour_billing_validation_passed": (
        primary_validation_pass
    ),
    "every_primary_household_has_required_valid_months": (
        bool(
            np.all(
                household_carbon[
                    "observed_months"
                ].to_numpy()
                >= PRIMARY_MIN_VALID_MONTHS
            )
        )
    ),
    "no_primary_household_exceeds_target_month_count": (
        bool(
            np.all(
                household_carbon[
                    "observed_months"
                ].to_numpy()
                <= len(TARGET_MONTHS)
            )
        )
    ),
    "every_primary_household_spans_early_and_late_window": (
        bool(
            np.all(
                selected_eligibility[
                    "has_early_window_month"
                ].to_numpy(dtype=bool)
            )
            and np.all(
                selected_eligibility[
                    "has_late_window_month"
                ].to_numpy(dtype=bool)
            )
        )
    ),
    "primary_cohort_exceeds_minimum_household_count": (
        len(household_carbon) >= MIN_PRIMARY_HOUSEHOLDS
    ),
    "every_primary_month_passes_quality_rule": (
        bool(
            np.all(
                primary_analysis_panel[
                    "month_quality_pass"
                ].to_numpy(dtype=bool)
            )
        )
    ),
    "every_primary_month_has_required_coverage": (
        bool(
            np.all(
                primary_analysis_panel[
                    "coverage_fraction"
                ].ge(MIN_MONTH_COVERAGE)
                .to_numpy(dtype=bool)
            )
        )
    ),
    "every_primary_month_has_real_boundary_anchors": (
        bool(
            np.all(
                primary_analysis_panel[
                    "nearest_start_actual_reading_time"
                ].notna().to_numpy(dtype=bool)
            )
            and np.all(
                primary_analysis_panel[
                    "nearest_end_actual_reading_time"
                ].notna().to_numpy(dtype=bool)
            )
        )
    ),
    "every_primary_boundary_anchor_is_within_6_5h": (
        bool(
            np.all(
                primary_analysis_panel[
                    "start_boundary_gap_hours"
                ].le(MAX_BOUNDARY_GAP_HOURS)
                .to_numpy(dtype=bool)
            )
            and np.all(
                primary_analysis_panel[
                    "end_boundary_gap_hours"
                ].le(MAX_BOUNDARY_GAP_HOURS)
                .to_numpy(dtype=bool)
            )
        )
    ),
    "target_months_are_within_may_to_november_2024": (
        set(
            carbon_monthly[
                "month_period"
            ].astype(str).unique()
        ).issubset({
            str(period)
            for period in TARGET_MONTHS
        })
        and carbon_monthly[
            "month_period"
        ].nunique() >= PRIMARY_MIN_VALID_MONTHS
    ),
    "only_0_71_factor_is_used": (
        len(factor_audit) == 1
        and np.isclose(
            factor_audit.loc[
                0,
                "factor_kg_co2_per_kwh",
            ],
            0.71,
        )
        and bool(
            factor_audit.loc[
                0,
                "used_for_primary_results",
            ]
        )
    ),
    "continuous_target_equals_kwh_times_0_71": (
        np.allclose(
            household_carbon[
                "mean_monthly_emissions_kg"
            ],
            household_carbon["mean_monthly_kwh"] * 0.71,
        )
    ),
    "factor_does_not_change_household_ranking": (
        factor_rank_invariance
    ),
    "no_direct_target_predictors": (
        not any(
            forbidden in feature.lower()
            for feature in primary_feature_columns
            for forbidden in FORBIDDEN_PREDICTOR_TERMS
        )
    ),
    "log_daily_kwh_is_excluded": (
        "log_daily_kwh" not in primary_feature_columns
    ),
    "cluster_labels_are_excluded": (
        not any(
            "cluster" in feature.lower()
            or "archetype" in feature.lower()
            for feature in primary_feature_columns
        )
    ),
    "one_row_per_model_household": (
        not primary_model_table[
            "household_ID"
        ].duplicated().any()
    ),
    "regression_and_classification_share_split": (
        primary_regression_results[
            "id_development"
        ].equals(
            primary_classification_results[
                "id_development"
            ]
        )
        and primary_regression_results[
            "id_test"
        ].equals(
            primary_classification_results[
                "id_test"
            ]
        )
    ),
    "test_not_used_for_regression_selection": (
        not bool(
            regression_split[
                "test_used_for_hyperparameter_selection"
            ]
        )
        and not bool(
            regression_split[
                "test_used_for_model_family_selection"
            ]
        )
    ),
    "test_not_used_for_classification_selection": (
        not bool(
            classification_split[
                "test_used_for_hyperparameter_selection"
            ]
        )
        and not bool(
            classification_split[
                "test_used_for_model_family_selection"
            ]
        )
        and not bool(
            classification_split[
                "test_used_for_decision_threshold_selection"
            ]
        )
    ),
    "holdout_split_did_not_use_outcome_stratification": (
        not bool(
            regression_split[
                "test_outcomes_used_to_stratify_split"
            ]
        )
        and not bool(
            classification_split[
                "test_outcomes_used_to_stratify_split"
            ]
        )
    ),
    "all_required_regression_models_were_tuned": (
        regression_model_names
        == {
            "Dummy mean baseline",
            "Ridge Regression",
            "Random Forest Regressor",
            "Histogram Gradient Boosting Regressor",
        }
    ),
    "all_required_classification_models_were_tuned": (
        classification_model_names
        == {
            "Dummy prior baseline",
            "Logistic Regression",
            "Random Forest",
            "Histogram Gradient Boosting",
        }
    ),
    "regression_selection_uses_nested_outer_cv": (
        primary_regression_results["selection_procedure"]
        == "nested_outer_cv_with_inner_hyperparameter_tuning"
        and len(primary_regression_results["outer_fold_results"])
        >= OUTER_CV_SPLITS * len(regression_model_names)
    ),
    "classification_selection_uses_nested_outer_cv": (
        primary_classification_results["selection_procedure"]
        == "nested_outer_cv_with_inner_hyperparameter_tuning"
        and len(primary_classification_results["outer_fold_results"])
        >= OUTER_CV_SPLITS * len(classification_model_names)
    ),
    "classification_target_is_outer_training_defined": (
        bool(primary_classification_results["adaptive_outer_target"])
        and primary_classification_results[
            "outer_target_thresholds"
        ]["target_threshold_source"].eq(
            "outer-training Q75"
        ).all()
    ),
    "classification_threshold_uses_complete_nested_oof_predictions": (
        len(primary_classification_results["oof_predictions"])
        == len(primary_classification_results["X_development"])
        * len(classification_model_names)
        and primary_classification_results[
            "oof_predictions"
        ]["nested_oof_probability"].notna().all()
    ),
    "class_weight_none_and_balanced_were_compared": (
        set(
            build_classification_search_spaces()[
                "Logistic Regression"
            ]["param_grid"]["model__class_weight"]
        ) == {None, "balanced"}
        and set(
            build_classification_search_spaces()[
                "Random Forest"
            ]["param_grid"]["model__class_weight"]
        ) == {None, "balanced"}
        and not primary_classification_results[
            "class_weight_ablation"
        ].empty
    ),
    "three_split_seeds_were_audited_without_reselecting_on_test": (
        split_stability_results["split_seed"].nunique() == 3
        and split_stability_results[
            "is_locked_primary_split"
        ].sum() == 1
    ),
    "sample_size_and_ci_widths_are_explicitly_audited": (
        len(sample_size_audit) == 1
        and sample_size_audit[
            "heldout_test_positive_count"
        ].notna().all()
        and sample_size_audit[
            "classification_bootstrap_max_ci_width"
        ].notna().all()
    ),
    "regression_model_selected_by_nested_development_cv": (
        primary_regression_results[
            "model_selection"
        ]["selected_by_development_cv"].sum() == 1
    ),
    "classification_model_selected_by_nested_development_cv": (
        primary_classification_results[
            "model_selection"
        ]["selected_by_development_cv"].sum() == 1
    ),
    "selected_regression_has_valid_locked_parameters": (
        primary_regression_results["selected_model_name"]
        == "Dummy mean baseline"
        or len(regression_best_params) > 0
    ),
    "selected_classification_has_valid_locked_parameters": (
        primary_classification_results["selected_model_name"]
        == "Dummy prior baseline"
        or len(classification_best_params) > 0
    ),
    "classification_threshold_is_nested_development_oof_derived": (
        0.0
        < primary_classification_results[
            "selected_threshold"
        ]
        < 1.0
    ),
    "regression_metrics_are_complete": (
        {
            "mae_kg_co2",
            "rmse_kg_co2",
            "median_absolute_error_kg_co2",
            "r2",
            "normalized_mae_percent_of_test_mean",
            "spearman_r",
        }.issubset(
            primary_regression_results[
                "test_metrics"
            ].columns
        )
    ),
    "classification_metrics_are_complete": (
        {
            "pr_auc",
            "roc_auc",
            "balanced_accuracy",
            "high_emission_precision",
            "high_emission_recall",
            "high_emission_f1",
            "mcc",
            "brier_score",
            "log_loss",
            "ece_10_bins",
        }.issubset(
            primary_classification_results[
                "test_metrics"
            ].columns
        )
    ),
    "regression_bootstrap_completed": (
        primary_regression_results[
            "bootstrap_intervals"
        ]["valid_bootstrap_samples"].min() >= 900
    ),
    "classification_bootstrap_completed": (
        primary_classification_results[
            "bootstrap_intervals"
        ]["valid_bootstrap_samples"].min() >= 900
    ),
    "heldout_regression_permutation_importance_completed": (
        len(regression_permutation_importance)
        == len(primary_feature_columns)
    ),
    "heldout_classification_permutation_importance_completed": (
        len(classification_permutation_importance)
        == len(primary_feature_columns)
    ),
    "survey_models_are_paired_when_run": (
        survey_pairing_valid
    ),
    "all_eight_teacher_household_variables_are_used": (
        teacher_features_complete
    ),
    "teacher_aligned_models_completed": (
        teacher_model_results_complete
    ),
    "teacher_augmented_feature_set_is_leakage_safe": (
        teacher_augmented_feature_set_is_leakage_safe
    ),
    "teacher_current_period_kwh_and_log_daily_are_excluded": (
        "avg_monthly_kwh" not in survey_augmented_features
        and "avg_log_daily_kwh" not in survey_augmented_features
        and "log_daily_kwh" not in survey_augmented_features
    ),
    "teacher_numeric_cluster_label_is_excluded": (
        "final_cluster" not in survey_augmented_features
    ),
    "teacher_variable_availability_was_checked_on_development_only": (
        teacher_development_availability is not None
        and teacher_development_availability[
            "development_non_missing_fraction"
        ].ge(0.60).all()
        and teacher_development_availability[
            "development_unique_non_missing_values"
        ].gt(1).all()
    ),
    "teacher_models_use_fresh_prespecified_matched_split": (
        survey_split_plan["random_state"]
        == TEACHER_MATCHED_RANDOM_STATE
        and survey_split_plan["name"]
        == "teacher_aligned_survey_matched_fresh_split"
    ),
    "teacher_regression_uses_training_fold_missing_indicators": (
        bool(
            survey_temporal_regression_results[
                "add_missing_indicators"
            ]
        )
        and bool(
            survey_augmented_regression_results[
                "add_missing_indicators"
            ]
        )
    ),
    "teacher_regression_compared_original_and_log_targets": (
        bool(
            survey_temporal_regression_results[
                "include_log_target"
            ]
        )
        and bool(
            survey_augmented_regression_results[
                "include_log_target"
            ]
        )
        and {"original", "log1p"}.issubset(
            set(
                survey_augmented_regression_results[
                    "model_selection"
                ]["target_transform"]
            )
        )
    ),
    "teacher_regression_selected_by_nested_original_scale_r2": (
        survey_temporal_regression_results[
            "selection_objective"
        ] == "r2"
        and survey_augmented_regression_results[
            "selection_objective"
        ] == "r2"
    ),
    "teacher_regression_reports_original_and_log_scale_r2": (
        {
            "r2_original_kg_co2_scale",
            "r2_log1p_scale",
        }.issubset(
            survey_augmented_regression_results[
                "test_metrics"
            ].columns
        )
    ),
    "teacher_regression_month_count_sensitivity_completed": (
        set(
            teacher_regression_month_count_sensitivity[
                "minimum_valid_months"
            ]
        ) == {5, 6, 7}
    ),
    "teacher_regression_permutation_importance_completed": (
        len(teacher_regression_permutation_importance)
        == len(survey_augmented_features)
    ),
    "teacher_classification_target_is_outer_training_defined": (
        bool(
            survey_augmented_classification_results[
                "adaptive_outer_target"
            ]
        )
        and survey_augmented_classification_results[
            "outer_target_thresholds"
        ]["target_threshold_source"].eq(
            "outer-training Q75"
        ).all()
    ),
    "all_gap_sensitivity_rules_completed": (
        set(
            gap_cohort_summary[
                "cross_month_max_gap_hours"
            ].astype(float)
        )
        == set(
            float(value)
            for value in SENSITIVITY_MAX_GAP_HOURS
        )
    ),
    "all_coverage_sensitivity_rules_completed": (
        set(
            coverage_sensitivity_summary[
                "minimum_month_coverage"
            ].round(2)
        )
        == {0.95, 0.98, 0.99}
    ),
    "all_minimum_month_sensitivity_rules_completed": (
        set(
            minimum_month_sensitivity_summary[
                "minimum_valid_months_required"
            ].astype(int)
        )
        == {3, 4, 5, 6, 7}
        and minimum_month_sensitivity_summary.loc[
            minimum_month_sensitivity_summary[
                "is_primary_rule"
            ],
            "minimum_valid_months_required",
        ].tolist() == [PRIMARY_MIN_VALID_MONTHS]
    ),
    "all_target_quantile_sensitivity_rules_completed": (
        set(
            quantile_sensitivity_summary[
                "target_quantile"
            ].round(2)
        )
        == {0.70, 0.75, 0.80}
    ),
    "no_timing_only_carbon_saving_claim": (
        np.isclose(
            fixed_factor_timing_check.loc[
                0,
                "timing_only_carbon_saving_kg",
            ],
            0.0,
        )
    ),
    "no_lkr_conversion": (
        not any(
            "lkr" in column.lower()
            for column in reduction_scenario.columns
        )
    ),

    "classification_inner_target_is_training_fold_defined": (
        primary_classification_results[
            "outer_split_strategy"
        ] == "shuffled KFold independent of global target labels"
        and primary_classification_results[
            "inner_target_definition"
        ] == "inner-training Q75"
        and primary_classification_results[
            "outer_fold_results"
        ]["inner_target_definition"].eq(
            "inner-training Q75"
        ).all()
        and primary_classification_results[
            "tuning_results"
        ].loc[
            primary_classification_results[
                "tuning_results"
            ]["selection_stage"].eq(
                "nested_outer_inner_search"
            ),
            "inner_target_source",
        ].eq("inner-training Q75").all()
    ),
    "teacher_classification_inner_target_is_training_fold_defined": (
        survey_temporal_classification_results[
            "inner_target_definition"
        ] == "inner-training Q75"
        and survey_augmented_classification_results[
            "inner_target_definition"
        ] == "inner-training Q75"
    ),
    "regression_scoring_matches_nonnegative_prediction_rule": (
        primary_regression_results[
            "regression_scoring_rule"
        ] == REGRESSION_SCORING_RULE
        and survey_temporal_regression_results[
            "regression_scoring_rule"
        ] == REGRESSION_SCORING_RULE
        and survey_augmented_regression_results[
            "regression_scoring_rule"
        ] == REGRESSION_SCORING_RULE
        and regression_searches_use_clipped_scorers(
            primary_regression_results
        )
        and regression_searches_use_clipped_scorers(
            survey_temporal_regression_results
        )
        and regression_searches_use_clipped_scorers(
            survey_augmented_regression_results
        )
    ),
    "classification_family_selected_by_mean_outer_fold_pr_auc": (
        classifier_selected_by_outer_fold_pr_auc(
            primary_classification_results
        )
        and classifier_selected_by_outer_fold_pr_auc(
            survey_temporal_classification_results
        )
        and classifier_selected_by_outer_fold_pr_auc(
            survey_augmented_classification_results
        )
    ),
    "development_feature_ablation_completed": (
        set(
            regression_ablation_summary[
                "variant"
            ]
        ) == {
            "all_features",
            *{
                f"without_{feature}"
                for feature in primary_feature_columns
            },
        }
        and set(
            classification_ablation_summary[
                "variant"
            ]
        ) == {
            "all_features",
            *{
                f"without_{feature}"
                for feature in primary_feature_columns
            },
        }
        and feature_ablation_fold_results[
            "outer_fold"
        ].nunique()
        >= min(
            primary_regression_results[
                "outer_cv_splits"
            ],
            primary_classification_results[
                "outer_cv_splits"
            ],
        )
    ),
    "heldout_regression_residual_diagnostics_completed": (
        len(regression_residual_summary) == 1
        and regression_residual_summary[
            "heldout_households"
        ].iloc[0] == len(
            primary_regression_results[
                "id_test"
            ]
        )
        and regression_error_by_actual_quartile[
            "households"
        ].sum() == len(
            primary_regression_results[
                "id_test"
            ]
        )
        and regression_error_by_actual_quartile[
            "actual_emission_quartile"
        ].nunique() >= 3
    ),
    "paired_wave1_bootstrap_difference_intervals_completed": (
        set(
            survey_paired_bootstrap_summary[
                "metric"
            ]
        ) == {
            "R2 gain",
            "MAE reduction kg CO2",
            "PR-AUC gain",
        }
        and survey_paired_bootstrap_summary[
            "valid_bootstrap_samples"
        ].min() >= int(
            0.90 * BOOTSTRAP_ITERATIONS
        )
        and len(
            paired_regression_predictions
        ) == len(
            survey_split_plan["test_ids"]
        )
        and len(
            paired_classification_predictions
        ) == len(
            survey_split_plan["test_ids"]
        )
    ),

}

release_gate = pd.DataFrame([
    {
        "check": check_name,
        "passed": bool(passed),
    }
    for check_name, passed in release_checks.items()
])

display(release_gate)

failed_checks = release_gate.loc[
    ~release_gate["passed"],
    "check",
].tolist()

release_gate.to_csv(
    OUTPUT_DIR / "FINAL_RELEASE_GATE.csv",
    index=False,
)

if failed_checks:
    raise RuntimeError(
        "Final methodological release gate failed: "
        + ", ".join(failed_checks)
    )

print(
    "FINAL CARBON REGRESSION + CLASSIFICATION + XAI "
    "RELEASE GATE: PASS"
)


In [ ]:
# ============================================================
# STEP 18 — SAVE TABLES, FINAL MODELS, JOBLIB AND PICKLE BUNDLE
# ============================================================

def save_classification_outputs(results, prefix):
    results["split_audit"].to_csv(
        OUTPUT_DIR / f"{prefix}_split_audit.csv",
        index=False,
    )
    results["model_selection"].to_csv(
        OUTPUT_DIR / f"{prefix}_model_selection.csv",
        index=False,
    )
    results["tuning_results"].to_csv(
        OUTPUT_DIR / f"{prefix}_nested_and_final_grid_results.csv",
        index=False,
    )
    results["outer_fold_results"].to_csv(
        OUTPUT_DIR / f"{prefix}_nested_outer_fold_results.csv",
        index=False,
    )
    results["oof_predictions"].to_csv(
        OUTPUT_DIR / f"{prefix}_nested_oof_predictions.csv",
        index=False,
    )
    results["test_metrics"].to_csv(
        OUTPUT_DIR / f"{prefix}_heldout_test_metrics.csv",
        index=False,
    )
    results["test_predictions"].to_csv(
        OUTPUT_DIR / f"{prefix}_heldout_test_predictions.csv",
        index=False,
    )
    results["bootstrap_long"].to_csv(
        OUTPUT_DIR / f"{prefix}_heldout_bootstrap_long.csv",
        index=False,
    )
    results["bootstrap_intervals"].to_csv(
        OUTPUT_DIR / f"{prefix}_heldout_bootstrap_95ci.csv",
        index=False,
    )
    results["calibration_table"].to_csv(
        OUTPUT_DIR / f"{prefix}_heldout_reliability.csv",
        index=False,
    )
    results["development_correlation"].to_csv(
        OUTPUT_DIR / f"{prefix}_development_correlation.csv"
    )
    results["correlated_pairs"].to_csv(
        OUTPUT_DIR / f"{prefix}_high_correlation_pairs.csv",
        index=False,
    )
    results["outer_target_thresholds"].to_csv(
        OUTPUT_DIR / f"{prefix}_nested_outer_target_thresholds.csv",
        index=False,
    )
    results["class_weight_ablation"].to_csv(
        OUTPUT_DIR / f"{prefix}_class_weight_ablation.csv",
        index=False,
    )

    for model_name, threshold_table in results[
        "threshold_tables"
    ].items():
        safe_name = (
            model_name.lower()
            .replace(" ", "_")
            .replace("-", "_")
        )
        threshold_table.to_csv(
            OUTPUT_DIR
            / f"{prefix}_{safe_name}_development_threshold_search.csv",
            index=False,
        )


def save_regression_outputs(results, prefix):
    results["split_audit"].to_csv(
        OUTPUT_DIR / f"{prefix}_split_audit.csv",
        index=False,
    )
    results["model_selection"].to_csv(
        OUTPUT_DIR / f"{prefix}_model_selection.csv",
        index=False,
    )
    results["tuning_results"].to_csv(
        OUTPUT_DIR / f"{prefix}_nested_and_final_grid_results.csv",
        index=False,
    )
    results["outer_fold_results"].to_csv(
        OUTPUT_DIR / f"{prefix}_nested_outer_fold_results.csv",
        index=False,
    )
    results["oof_predictions"].to_csv(
        OUTPUT_DIR / f"{prefix}_nested_oof_predictions.csv",
        index=False,
    )
    results["test_metrics"].to_csv(
        OUTPUT_DIR / f"{prefix}_heldout_test_metrics.csv",
        index=False,
    )
    results["test_predictions"].to_csv(
        OUTPUT_DIR / f"{prefix}_heldout_test_predictions.csv",
        index=False,
    )
    results["bootstrap_long"].to_csv(
        OUTPUT_DIR / f"{prefix}_heldout_bootstrap_long.csv",
        index=False,
    )
    results["bootstrap_intervals"].to_csv(
        OUTPUT_DIR / f"{prefix}_heldout_bootstrap_95ci.csv",
        index=False,
    )
    results["relative_improvement"].to_csv(
        OUTPUT_DIR / f"{prefix}_baseline_improvement.csv",
        index=False,
    )


input_audit.to_csv(
    OUTPUT_DIR / "01_input_audit.csv",
    index=False,
)
cluster_population.to_csv(
    OUTPUT_DIR / "02_locked_cluster_population.csv",
    index=False,
)
raw_feature_audit.to_csv(
    OUTPUT_DIR / "02B_unclipped_feature_source_audit.csv",
    index=False,
)
sixhour_file_audit.to_csv(
    OUTPUT_DIR / "03_sixhour_file_audit.csv",
    index=False,
)
sixhour_quality_audit.to_csv(
    OUTPUT_DIR / "04_sixhour_quality_audit.csv",
    index=False,
)
allocation_audit_table.to_csv(
    OUTPUT_DIR / "05_month_allocation_energy_conservation.csv",
    index=False,
)
coverage_summary.to_csv(
    OUTPUT_DIR / "06_monthly_coverage_by_gap_rule.csv",
    index=False,
)
primary_panel_audit.to_csv(
    OUTPUT_DIR / "07_primary_minimum_month_panel_audit.csv",
    index=False,
)
primary_cohort_eligibility_audit.to_csv(
    OUTPUT_DIR / "07A_primary_cohort_eligibility_audit.csv",
    index=False,
)
primary_month_availability_summary.to_csv(
    OUTPUT_DIR / "07B_primary_month_availability_summary.csv",
    index=False,
)
november_availability_audit.to_csv(
    OUTPUT_DIR / "07C_november_availability_audit.csv",
    index=False,
)
sixhour_monthly_all.to_csv(
    OUTPUT_DIR / "08_sixhour_monthly_gap_sensitivity.csv",
    index=False,
)
billing_validation_all.to_csv(
    OUTPUT_DIR / "09_sixhour_billing_validation_rows.csv",
    index=False,
)
billing_validation_summary.to_csv(
    OUTPUT_DIR / "10_sixhour_billing_validation_summary.csv",
    index=False,
)
validation_gate.to_csv(
    OUTPUT_DIR / "11_sixhour_validation_gate.csv",
    index=False,
)
factor_audit.to_csv(
    OUTPUT_DIR / "12_emission_factor_audit.csv",
    index=False,
)
carbon_monthly.to_csv(
    OUTPUT_DIR / "13_household_month_carbon.csv",
    index=False,
)
household_carbon.to_csv(
    OUTPUT_DIR / "14_household_carbon_summary.csv",
    index=False,
)
household_month_count_target_audit.to_csv(
    OUTPUT_DIR / "14B_household_month_count_target_audit.csv",
    index=False,
)
archetype_carbon_summary.to_csv(
    OUTPUT_DIR / "15_archetype_carbon_summary.csv",
    index=False,
)
archetype_global_test.to_csv(
    OUTPUT_DIR / "16_archetype_global_test.csv",
    index=False,
)
pairwise_tests.to_csv(
    OUTPUT_DIR / "17_archetype_pairwise_holm_effect_sizes.csv",
    index=False,
)
match_table.to_csv(
    OUTPUT_DIR / "18_archetype_match_table.csv"
)
match_bias_result.to_csv(
    OUTPUT_DIR / "19_archetype_match_bias_result.csv",
    index=False,
)
inclusion_feature_audit.to_csv(
    OUTPUT_DIR / "19B_inclusion_feature_bias_audit.csv",
    index=False,
)

primary_model_table[
    [
        "household_ID",
        "mean_monthly_kwh",
        "mean_monthly_emissions_kg",
    ]
    + primary_feature_columns
].to_csv(
    OUTPUT_DIR
    / "20_primary_direct_target_excluded_modelling_table.csv",
    index=False,
)

save_regression_outputs(
    primary_regression_results,
    "21_primary_continuous_emission_regression",
)
save_classification_outputs(
    primary_classification_results,
    "22_secondary_high_emission_classification",
)

classification_permutation_importance.to_csv(
    OUTPUT_DIR
    / "23_classification_heldout_permutation_importance.csv",
    index=False,
)
regression_permutation_importance.to_csv(
    OUTPUT_DIR
    / "24_regression_heldout_permutation_importance.csv",
    index=False,
)


feature_ablation_fold_results.to_csv(
    OUTPUT_DIR / "24A_development_feature_ablation_outer_folds.csv",
    index=False,
)
regression_ablation_summary.to_csv(
    OUTPUT_DIR / "24B_regression_feature_ablation_summary.csv",
    index=False,
)
classification_ablation_summary.to_csv(
    OUTPUT_DIR / "24C_classification_feature_ablation_summary.csv",
    index=False,
)
regression_residual_summary.to_csv(
    OUTPUT_DIR / "24D_regression_heldout_residual_summary.csv",
    index=False,
)
regression_error_by_actual_quartile.to_csv(
    OUTPUT_DIR / "24E_regression_heldout_error_by_emission_quartile.csv",
    index=False,
)

if classification_shap_global is not None:
    classification_shap_global.to_csv(
        OUTPUT_DIR
        / "25_classification_heldout_shap_global.csv",
        index=False,
    )

if classification_shap_households is not None:
    classification_shap_households.to_csv(
        OUTPUT_DIR
        / "25B_classification_heldout_shap_households.csv",
        index=False,
    )

if regression_shap_global is not None:
    regression_shap_global.to_csv(
        OUTPUT_DIR
        / "26_regression_heldout_shap_global.csv",
        index=False,
    )

if regression_shap_households is not None:
    regression_shap_households.to_csv(
        OUTPUT_DIR
        / "26B_regression_heldout_shap_households.csv",
        index=False,
    )

if classification_linear_coefficients is not None:
    classification_linear_coefficients.to_csv(
        OUTPUT_DIR
        / "25C_classification_linear_coefficients.csv",
        index=False,
    )

if regression_linear_coefficients is not None:
    regression_linear_coefficients.to_csv(
        OUTPUT_DIR
        / "26C_regression_linear_coefficients.csv",
        index=False,
    )

if survey_feature_audit is not None:
    survey_feature_audit.to_csv(
        OUTPUT_DIR / "27_survey_feature_audit.csv",
        index=False,
    )

if survey_feature_missingness is not None:
    survey_feature_missingness.to_csv(
        OUTPUT_DIR / "27B_survey_feature_missingness.csv",
        index=False,
    )

teacher_source_audit.to_csv(
    OUTPUT_DIR / "27C_teacher_variable_source_file_audit.csv",
    index=False,
)
teacher_variable_source_audit.to_csv(
    OUTPUT_DIR / "27D_teacher_variable_column_mapping.csv",
    index=False,
)
teacher_variable_availability.to_csv(
    OUTPUT_DIR / "27E_teacher_variable_availability.csv",
    index=False,
)
teacher_development_availability.to_csv(
    OUTPUT_DIR / "27F_teacher_variable_development_audit.csv",
    index=False,
)
survey_model_table[
    [
        "household_ID",
        "mean_monthly_kwh",
        "mean_monthly_emissions_kg",
    ]
    + survey_augmented_features
].to_csv(
    OUTPUT_DIR
    / "27G_teacher_aligned_matched_modelling_table.csv",
    index=False,
)

if survey_temporal_regression_results is not None:
    save_regression_outputs(
        survey_temporal_regression_results,
        "28A_survey_temporal_only_regression",
    )

if survey_augmented_regression_results is not None:
    save_regression_outputs(
        survey_augmented_regression_results,
        "28B_survey_temporal_plus_survey_regression",
    )

if survey_temporal_classification_results is not None:
    save_classification_outputs(
        survey_temporal_classification_results,
        "28C_survey_temporal_only_classification",
    )

if survey_augmented_classification_results is not None:
    save_classification_outputs(
        survey_augmented_classification_results,
        "28D_survey_temporal_plus_survey_classification",
    )

if not survey_paired_comparison.empty:
    survey_paired_comparison.to_csv(
        OUTPUT_DIR / "28E_survey_paired_comparison.csv",
        index=False,
    )


survey_paired_bootstrap_long.to_csv(
    OUTPUT_DIR / "28H_survey_paired_bootstrap_difference_long.csv",
    index=False,
)
survey_paired_bootstrap_summary.to_csv(
    OUTPUT_DIR / "28I_survey_paired_bootstrap_difference_95ci.csv",
    index=False,
)


teacher_regression_month_count_sensitivity.to_csv(
    OUTPUT_DIR / "28F_teacher_regression_valid_month_sensitivity.csv",
    index=False,
)
teacher_regression_permutation_importance.to_csv(
    OUTPUT_DIR / "28G_teacher_regression_heldout_permutation_importance.csv",
    index=False,
)

gap_household_sensitivity.to_csv(
    OUTPUT_DIR / "29_gap_sensitivity_household_values.csv",
    index=False,
)
gap_cohort_summary.to_csv(
    OUTPUT_DIR / "30A_gap_sensitivity_cohort_summary.csv",
    index=False,
)
gap_sensitivity_summary.to_csv(
    OUTPUT_DIR / "30B_gap_sensitivity_primary_comparison.csv",
    index=False,
)
coverage_sensitivity_summary.to_csv(
    OUTPUT_DIR / "30C_coverage_threshold_sensitivity.csv",
    index=False,
)
minimum_month_sensitivity_summary.to_csv(
    OUTPUT_DIR / "30D_minimum_valid_month_sensitivity.csv",
    index=False,
)
quantile_sensitivity_summary.to_csv(
    OUTPUT_DIR / "30E_high_emission_quantile_sensitivity.csv",
    index=False,
)
reduction_scenario.to_csv(
    OUTPUT_DIR / "31_illustrative_reduction_scenario.csv",
    index=False,
)
fixed_factor_timing_check.to_csv(
    OUTPUT_DIR / "32_fixed_factor_timing_check.csv",
    index=False,
)

sample_size_audit.to_csv(
    OUTPUT_DIR / "33A_primary_sample_size_and_ci_audit.csv",
    index=False,
)
split_stability_results.to_csv(
    OUTPUT_DIR / "33B_split_seed_stability_results.csv",
    index=False,
)
split_stability_summary.to_csv(
    OUTPUT_DIR / "33C_split_seed_stability_summary.csv",
    index=False,
)

split_ids = pd.concat([
    pd.DataFrame({
        "household_ID": primary_split_plan[
            "development_ids"
        ],
        "split": "development",
    }),
    pd.DataFrame({
        "household_ID": primary_split_plan["test_ids"],
        "split": "heldout_test",
    }),
], ignore_index=True)

split_ids.to_csv(
    OUTPUT_DIR / "33_locked_household_split_ids.csv",
    index=False,
)

# ------------------------------------------------------------
# Final fitted artifacts
# ------------------------------------------------------------
# Evaluation pipelines remain fitted on development households only.
regression_evaluation_pipeline = (
    primary_regression_results["selected_model"]
)
classification_evaluation_pipeline = (
    primary_classification_results["selected_model"]
)

# Deployment pipelines are refitted only after the held-out evaluation
# has been completed and reported. They use the locked model family,
# locked hyperparameters and, for classification, the locked
# development-derived high-emission definition.
all_primary_X = primary_model_table[
    primary_feature_columns
].copy()

all_regression_y = primary_model_table[
    "mean_monthly_emissions_kg"
].copy()

locked_high_emission_threshold_kg = float(
    primary_split_plan[
        "development_threshold_emissions"
    ]
)

all_classification_y = (
    all_regression_y
    >= locked_high_emission_threshold_kg
).astype(int)

regression_deployment_pipeline = clone(
    regression_evaluation_pipeline
)
regression_deployment_pipeline.fit(
    all_primary_X,
    all_regression_y,
)

classification_deployment_pipeline = clone(
    classification_evaluation_pipeline
)
classification_deployment_pipeline.fit(
    all_primary_X,
    all_classification_y,
)

regression_best_params = (
    primary_regression_results[
        "fitted_searches"
    ][
        primary_regression_results[
            "selected_model_name"
        ]
    ].best_params_
)

classification_best_params = (
    primary_classification_results[
        "fitted_searches"
    ][
        primary_classification_results[
            "selected_model_name"
        ]
    ].best_params_
)

model_metadata = {
    "created_by_notebook": (
        "Household Electricity Carbon XAI final workflow"
    ),
    "study_period": "May-November 2024",
    "primary_cross_month_max_gap_hours": (
        PRIMARY_MAX_GAP_HOURS
    ),
    "minimum_valid_full_months_per_household": (
        PRIMARY_MIN_VALID_MONTHS
    ),
    "cohort_rule": (
        "At least five quality-passing full months within "
        "May-November, with early- and late-window representation"
    ),
    "emission_factor_kg_co2_per_kwh": (
        GRID_FACTOR_KG_CO2_PER_KWH
    ),
    "regression_target": (
        "mean_monthly_emissions_kg = "
        "mean_monthly_kwh * 0.71"
    ),
    "classification_target": (
        "mean_monthly_emissions_kg at or above the "
        "development-set Q75 threshold"
    ),
    "locked_high_emission_threshold_kg": (
        locked_high_emission_threshold_kg
    ),
    "locked_high_emission_threshold_kwh": (
        primary_split_plan[
            "development_threshold_kwh"
        ]
    ),
    "locked_classification_decision_threshold": (
        primary_classification_results[
            "selected_threshold"
        ]
    ),
    "feature_columns": primary_feature_columns,
    "teacher_aligned_feature_columns": survey_augmented_features,
    "teacher_household_variables": TEACHER_SOCIO_FEATURES,
    "teacher_variable_units": TEACHER_VARIABLE_UNITS,
    "teacher_aligned_matched_households": len(survey_model_table),
    "teacher_code_variables_intentionally_excluded": {
        "avg_monthly_kwh": "directly constructs the carbon target",
        "avg_log_daily_kwh": "closely encodes current-period target magnitude",
        "final_cluster": (
            "derived from the temporal features and not an independent "
            "household predictor"
        ),
    },
    "excluded_direct_target_terms": (
        FORBIDDEN_PREDICTOR_TERMS
    ),
    "selected_regression_model": (
        primary_regression_results[
            "selected_model_name"
        ]
    ),
    "selected_regression_best_params": (
        regression_best_params
    ),
    "selected_classification_model": (
        primary_classification_results[
            "selected_model_name"
        ]
    ),
    "selected_classification_best_params": (
        classification_best_params
    ),
    "selection_rules": {
        "regression": (
            "lowest honest nested development OOF MAE; "
            "RMSE then R2 tie-break"
        ),
        "classification": (
            "highest honest nested development OOF PR-AUC; "
            "nested OOF balanced accuracy then F1 tie-break"
        ),
        "classification_decision_threshold": (
            "maximized balanced accuracy using honest nested "
            "development OOF probabilities"
        ),
    },
    "nested_cv": {
        "primary_outer_splits": OUTER_CV_SPLITS,
        "primary_inner_splits": INNER_CV_SPLITS,
        "final_tuning_splits": FINAL_TUNING_CV_SPLITS,
        "classification_outer_split_strategy": (
            "shuffled KFold independent of global target labels"
        ),
        "classification_outer_target": "outer-training Q75",
        "classification_inner_target": "inner-training Q75",
    },
    "split_stability_seeds": split_stability_results[
        "split_seed"
    ].tolist(),
    "deployment_ready": False,
    "artifact_status": "saved research prototype",
    "evaluation_scope": (
        "Internal household-level held-out test; "
        "cross-sectional, not future forecasting"
    ),
}

with (
    OUTPUT_DIR / "FINAL_model_metadata.json"
).open("w", encoding="utf-8") as handle:
    json.dump(
        model_metadata,
        handle,
        indent=2,
        default=str,
    )

model_bundle = {
    "metadata": model_metadata,
    "feature_columns": primary_feature_columns,
    "regression_evaluation_pipeline": (
        regression_evaluation_pipeline
    ),
    "regression_deployment_pipeline": (
        regression_deployment_pipeline
    ),
    "classification_evaluation_pipeline": (
        classification_evaluation_pipeline
    ),
    "classification_deployment_pipeline": (
        classification_deployment_pipeline
    ),
    "classification_decision_threshold": (
        primary_classification_results[
            "selected_threshold"
        ]
    ),
    "high_emission_target_threshold_kg": (
        locked_high_emission_threshold_kg
    ),
    "development_household_ids": (
        primary_split_plan["development_ids"].tolist()
    ),
    "heldout_test_household_ids": (
        primary_split_plan["test_ids"].tolist()
    ),
    "sample_size_audit": sample_size_audit.to_dict(
        orient="records"
    ),
    "primary_panel_audit": primary_panel_audit.to_dict(
        orient="records"
    ),
    "month_availability_summary": (
        primary_month_availability_summary.to_dict(
            orient="records"
        )
    ),
    "split_stability_summary": split_stability_summary.to_dict(
        orient="records"
    ),
    "regression_feature_ablation": (
        regression_ablation_summary.to_dict(orient="records")
    ),
    "classification_feature_ablation": (
        classification_ablation_summary.to_dict(orient="records")
    ),
    "regression_residual_summary": (
        regression_residual_summary.to_dict(orient="records")
    ),
}

# Wave-1 household-context matched-cohort artifacts.
teacher_regression_evaluation_pipeline = (
    survey_augmented_regression_results["selected_model"]
)
teacher_classification_evaluation_pipeline = (
    survey_augmented_classification_results["selected_model"]
)

teacher_all_X = survey_model_table[
    survey_augmented_features
].copy()
teacher_all_regression_y = survey_model_table[
    "mean_monthly_emissions_kg"
].copy()
teacher_all_classification_y = (
    teacher_all_regression_y
    >= float(
        survey_split_plan[
            "development_threshold_emissions"
        ]
    )
).astype(int)

teacher_regression_deployment_pipeline = clone(
    teacher_regression_evaluation_pipeline
)
teacher_regression_deployment_pipeline.fit(
    teacher_all_X,
    teacher_all_regression_y,
)

teacher_classification_deployment_pipeline = clone(
    teacher_classification_evaluation_pipeline
)
teacher_classification_deployment_pipeline.fit(
    teacher_all_X,
    teacher_all_classification_y,
)

teacher_model_metadata = {
    "artifact_status": "saved research prototype",
    "cohort": "survey-matched subset with a fresh prespecified holdout split",
    "households": len(survey_model_table),
    "features": survey_augmented_features,
    "teacher_household_variables": TEACHER_SOCIO_FEATURES,
    "emission_factor_kg_co2_per_kwh": (
        GRID_FACTOR_KG_CO2_PER_KWH
    ),
    "regression_target": (
        "mean_monthly_emissions_kg = mean_monthly_kwh * 0.71"
    ),
    "regression_primary_selection_metric": (
        "honest nested development OOF original-scale R2"
    ),
    "regression_missing_value_strategy": (
        "median imputation plus missingness indicators, fitted inside "
        "each training fold"
    ),
    "regression_target_formulations_compared": [
        "original kg CO2",
        "log1p target with predictions back-transformed to kg CO2",
    ],
    "selected_regression_target_transform": (
        survey_augmented_regression_results["split_audit"].iloc[0][
            "selected_target_transform"
        ]
    ),
    "r2_reporting_rule": (
        "Original-scale R2 is principal; log1p-scale R2 is supplementary"
    ),
    "classification_target_threshold_kg": float(
        survey_split_plan[
            "development_threshold_emissions"
        ]
    ),
    "classification_decision_threshold": float(
        survey_augmented_classification_results[
            "selected_threshold"
        ]
    ),
    "selected_regression_model": (
        survey_augmented_regression_results[
            "selected_model_name"
        ]
    ),
    "selected_classification_model": (
        survey_augmented_classification_results[
            "selected_model_name"
        ]
    ),
    "excluded_teacher_code_variables": [
        "avg_monthly_kwh",
        "avg_log_daily_kwh",
        "final_cluster",
    ],
    "evaluation_scope": (
        "Fresh prespecified internal matched-cohort held-out evaluation; "
        "not external validation and not future forecasting"
    ),
}

teacher_model_bundle = {
    "metadata": teacher_model_metadata,
    "feature_columns": survey_augmented_features,
    "regression_evaluation_pipeline": (
        teacher_regression_evaluation_pipeline
    ),
    "regression_deployment_pipeline": (
        teacher_regression_deployment_pipeline
    ),
    "classification_evaluation_pipeline": (
        teacher_classification_evaluation_pipeline
    ),
    "classification_deployment_pipeline": (
        teacher_classification_deployment_pipeline
    ),
    "classification_decision_threshold": (
        survey_augmented_classification_results[
            "selected_threshold"
        ]
    ),
    "regression_valid_month_sensitivity": (
        teacher_regression_month_count_sensitivity.to_dict(
            orient="records"
        )
    ),
    "regression_permutation_importance": (
        teacher_regression_permutation_importance.to_dict(
            orient="records"
        )
    ),
    "paired_augmentation_bootstrap_95ci": (
        survey_paired_bootstrap_summary.to_dict(
            orient="records"
        )
    ),
    "high_emission_target_threshold_kg": float(
        survey_split_plan[
            "development_threshold_emissions"
        ]
    ),
    "development_household_ids": (
        survey_split_plan["development_ids"].tolist()
    ),
    "heldout_test_household_ids": (
        survey_split_plan["test_ids"].tolist()
    ),
}

model_bundle["teacher_aligned_augmented_model"] = (
    teacher_model_bundle
)

joblib.dump(
    teacher_regression_evaluation_pipeline,
    OUTPUT_DIR
    / "FINAL_teacher_aligned_regression_evaluation_pipeline.joblib",
)
joblib.dump(
    teacher_regression_deployment_pipeline,
    OUTPUT_DIR
    / "FINAL_teacher_aligned_regression_deployment_pipeline.joblib",
)
joblib.dump(
    teacher_classification_evaluation_pipeline,
    OUTPUT_DIR
    / "FINAL_teacher_aligned_classification_evaluation_pipeline.joblib",
)
joblib.dump(
    teacher_classification_deployment_pipeline,
    OUTPUT_DIR
    / "FINAL_teacher_aligned_classification_deployment_pipeline.joblib",
)
joblib.dump(
    teacher_model_bundle,
    OUTPUT_DIR / "FINAL_teacher_aligned_model_bundle.joblib",
)
with (
    OUTPUT_DIR / "FINAL_teacher_aligned_model_bundle.pkl"
).open("wb") as handle:
    pickle.dump(
        teacher_model_bundle,
        handle,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

with (
    OUTPUT_DIR / "FINAL_teacher_aligned_model_metadata.json"
).open("w", encoding="utf-8") as handle:
    json.dump(
        teacher_model_metadata,
        handle,
        indent=2,
        default=str,
    )

teacher_feature_schema = pd.DataFrame({
    "feature_order": np.arange(
        1,
        len(survey_augmented_features) + 1,
    ),
    "feature_name": survey_augmented_features,
    "feature_group": [
        (
            "temporal_shape"
            if feature in primary_feature_columns
            else "teacher_household"
        )
        for feature in survey_augmented_features
    ],
    "required_for_prediction": True,
    "expected_type": "numeric",
    "direct_target_variable": False,
})
teacher_feature_schema.to_csv(
    OUTPUT_DIR / "FINAL_teacher_aligned_feature_schema.csv",
    index=False,
)

# Individual evaluation and deployment artifacts.
joblib.dump(
    regression_evaluation_pipeline,
    OUTPUT_DIR
    / "FINAL_regression_evaluation_pipeline.joblib",
)
joblib.dump(
    regression_deployment_pipeline,
    OUTPUT_DIR
    / "FINAL_regression_deployment_pipeline.joblib",
)
joblib.dump(
    classification_evaluation_pipeline,
    OUTPUT_DIR
    / "FINAL_classification_evaluation_pipeline.joblib",
)
joblib.dump(
    classification_deployment_pipeline,
    OUTPUT_DIR
    / "FINAL_classification_deployment_pipeline.joblib",
)

# Combined bundle in both requested serialization formats.
joblib.dump(
    model_bundle,
    OUTPUT_DIR / "FINAL_carbon_xai_model_bundle.joblib",
)

with (
    OUTPUT_DIR / "FINAL_carbon_xai_model_bundle.pkl"
).open("wb") as handle:
    pickle.dump(
        model_bundle,
        handle,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

print("All tables and final model artifacts saved to:", OUTPUT_DIR)
print(
    "Primary full-cohort regression research prototype:",
    OUTPUT_DIR / "FINAL_regression_deployment_pipeline.joblib",
)
print(
    "Secondary full-cohort classification research prototype:",
    OUTPUT_DIR / "FINAL_classification_deployment_pipeline.joblib",
)
print(
    "Combined joblib bundle:",
    OUTPUT_DIR / "FINAL_carbon_xai_model_bundle.joblib",
)
print(
    "Combined pickle bundle:",
    OUTPUT_DIR / "FINAL_carbon_xai_model_bundle.pkl",
)


# Reproducibility and responsible-use files.
feature_schema = pd.DataFrame({
    "feature_order": np.arange(1, len(primary_feature_columns) + 1),
    "feature_name": primary_feature_columns,
    "required_for_prediction": True,
    "expected_type": "numeric",
    "direct_target_variable": False,
})
feature_schema.to_csv(
    OUTPUT_DIR / "FINAL_feature_schema.csv",
    index=False,
)

environment_versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "joblib": joblib.__version__,
    "platform": platform.platform(),
}
with (
    OUTPUT_DIR / "FINAL_environment_versions.json"
).open("w", encoding="utf-8") as handle:
    json.dump(environment_versions, handle, indent=2)

model_card_text = f"""# Carbon-XAI Saved Research Prototype

## Purpose
Estimate calculated mean monthly electricity-related operational CO2 emissions
and identify households above the development-set Q75 emission threshold. The
full-cohort model uses five target-excluded temporal load-shape predictors. A
paired Wave-1 household-context matched-cohort model additionally uses eight genuine
household, demographic and appliance variables.

## Data and scope
- Study period: May-November 2024
- Primary cohort: at least {PRIMARY_MIN_VALID_MONTHS} quality-passing full months
- Adjacent cross-month allocation limit: {PRIMARY_MAX_GAP_HOURS} hours
- Missing months: not imputed
- Emission factor: {GRID_FACTOR_KG_CO2_PER_KWH} kg CO2/kWh
- Evaluation: internal household-level held-out test
- Design: cross-sectional explanatory modelling, not future forecasting

## Selection safeguards
- Final test split created before model development
- Nested development CV for model-family and hyperparameter selection
- Classification target threshold derived inside each outer training fold
- Hyperparameter-tuning labels derive Q75 again inside each inner training fold
- Classification model-selection folds are not stratified using a global target
- Probability decision threshold selected from honest nested OOF predictions
- Leave-one-feature-out ablation is restricted to development folds
- Held-out residual diagnostics characterize errors without retuning the model
- Matched Wave 1 improvement is accompanied by paired bootstrap confidence intervals
- Logistic Regression and Random Forest compared with both unweighted and
  class-weight-balanced training
- Three split seeds audited as a sensitivity analysis
- Dummy baselines remained eligible if no trained model improved the primary
  nested out-of-fold metric

## Prohibited interpretations
- Not external validation
- Not a complete household carbon footprint
- Not causal evidence
- Not calibrated household risk unless calibration is separately established
- Not operational deployment ready

## Required inputs for the full-cohort model
{chr(10).join(f'- {feature}' for feature in primary_feature_columns)}

## Additional inputs for the Wave-1 household-context matched-cohort model
{chr(10).join(f'- {feature}' for feature in TEACHER_SOCIO_FEATURES)}

## Direct-target variables deliberately excluded
- avg_monthly_kwh: directly constructs the target
- avg_log_daily_kwh: directly encodes current-period magnitude
- final_cluster: derived from the temporal features and not independent
"""
with (
    OUTPUT_DIR / "FINAL_MODEL_CARD.md"
).open("w", encoding="utf-8") as handle:
    handle.write(model_card_text)

# Verify that the saved artifacts can be loaded immediately.
required_artifacts = [
    OUTPUT_DIR / "FINAL_regression_evaluation_pipeline.joblib",
    OUTPUT_DIR / "FINAL_regression_deployment_pipeline.joblib",
    OUTPUT_DIR / "FINAL_classification_evaluation_pipeline.joblib",
    OUTPUT_DIR / "FINAL_classification_deployment_pipeline.joblib",
    OUTPUT_DIR / "FINAL_carbon_xai_model_bundle.joblib",
    OUTPUT_DIR / "FINAL_carbon_xai_model_bundle.pkl",
    OUTPUT_DIR / "FINAL_model_metadata.json",
    OUTPUT_DIR / "FINAL_feature_schema.csv",
    OUTPUT_DIR / "FINAL_environment_versions.json",
    OUTPUT_DIR / "FINAL_MODEL_CARD.md",
    OUTPUT_DIR / "FINAL_teacher_aligned_regression_evaluation_pipeline.joblib",
    OUTPUT_DIR / "FINAL_teacher_aligned_regression_deployment_pipeline.joblib",
    OUTPUT_DIR / "FINAL_teacher_aligned_classification_evaluation_pipeline.joblib",
    OUTPUT_DIR / "FINAL_teacher_aligned_classification_deployment_pipeline.joblib",
    OUTPUT_DIR / "FINAL_teacher_aligned_model_bundle.joblib",
    OUTPUT_DIR / "FINAL_teacher_aligned_model_bundle.pkl",
    OUTPUT_DIR / "FINAL_teacher_aligned_model_metadata.json",
    OUTPUT_DIR / "FINAL_teacher_aligned_feature_schema.csv",
]

missing_artifacts = [
    str(path)
    for path in required_artifacts
    if not path.exists()
]

if missing_artifacts:
    raise RuntimeError(
        "Final artifact export failed: "
        + ", ".join(missing_artifacts)
    )

loaded_joblib_bundle = joblib.load(
    OUTPUT_DIR / "FINAL_carbon_xai_model_bundle.joblib"
)

with (
    OUTPUT_DIR / "FINAL_carbon_xai_model_bundle.pkl"
).open("rb") as handle:
    loaded_pickle_bundle = pickle.load(handle)

loaded_teacher_joblib_bundle = joblib.load(
    OUTPUT_DIR / "FINAL_teacher_aligned_model_bundle.joblib"
)
with (
    OUTPUT_DIR / "FINAL_teacher_aligned_model_bundle.pkl"
).open("rb") as handle:
    loaded_teacher_pickle_bundle = pickle.load(handle)

if (
    loaded_teacher_joblib_bundle["feature_columns"]
    != survey_augmented_features
    or loaded_teacher_pickle_bundle["feature_columns"]
    != survey_augmented_features
):
    raise RuntimeError(
        "Saved Wave-1 household-context model bundle feature metadata "
        "is inconsistent."
    )

if (
    loaded_joblib_bundle["feature_columns"]
    != primary_feature_columns
    or loaded_pickle_bundle["feature_columns"]
    != primary_feature_columns
):
    raise RuntimeError(
        "Saved model bundle feature metadata is inconsistent."
    )

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


manifest_rows = []
for artifact_path in sorted(OUTPUT_DIR.iterdir()):
    if artifact_path.is_file() and artifact_path.name != (
        "FINAL_artifact_manifest_sha256.csv"
    ):
        manifest_rows.append({
            "file": artifact_path.name,
            "bytes": artifact_path.stat().st_size,
            "sha256": sha256_file(artifact_path),
        })

artifact_manifest = pd.DataFrame(manifest_rows)
artifact_manifest.to_csv(
    OUTPUT_DIR / "FINAL_artifact_manifest_sha256.csv",
    index=False,
)

if artifact_manifest.empty or artifact_manifest["sha256"].str.len().ne(64).any():
    raise RuntimeError("Artifact SHA-256 manifest validation failed.")

print("FINAL JOBLIB/PICKLE/REPRODUCIBILITY ARTIFACT GATE: PASS")




### Core source

1. **Sri Lanka Sustainable Energy Authority.** *Carbon Footprint*. The official
   page lists electricity at **0.71 kg CO₂/kWh**. The reference year and full
   derivation boundary are not stated.


> The six-hour source did not support a sufficiently large all-seven-month
> complete-case cohort under the strict month-quality rules. The primary carbon
> cohort therefore required at least five quality-passing full months within
> May-November 2024, with representation in both the early and later parts of
> the window. Missing months were not imputed. Household targets were calculated
> as the mean across valid months, and observed-month count and month composition
> were reported as availability audits.

> Electricity-related operational CO₂ emissions were calculated by multiplying
> reconstructed imported electricity by 0.71 kg CO₂/kWh. The primary supervised
> task estimated continuous mean monthly household emissions, while a secondary
> classification task identified households above the development-derived 75th
> percentile of emissions. These are two representations of the same underlying
> electricity-derived outcome and must not be presented as independent evidence.

> Direct consumption magnitude, `log_daily_kwh`, calculated emissions, coverage
> and cluster labels were excluded from the predictor set. The five temporal
> load-shape predictors were exported before thesis-wide percentile clipping.
> Median imputation and any scaling were fitted within model pipelines.

> A household-level development/test split was created before model development.
> Model-family and hyperparameter selection used nested development
> cross-validation. Classification model-selection folds use shuffled KFold
> splits rather than stratification on a target derived from the full
> development sample. Within each outer fold, the high-emission Q75 threshold is
> calculated from outer-training households only. Within each inner tuning fold,
> Q75 is recalculated from that inner-training subset and then applied unchanged
> to the corresponding inner-validation subset. Honest outer out-of-fold
> probabilities select the classifier family and probability decision threshold.
> The final test set is evaluated only after all development choices are locked.

> The dummy baseline remained eligible for selection; if no trained model
> improved the primary nested out-of-fold metric, the correct conclusion was
> that the available behavioural predictors did not outperform the baseline.
> Logistic Regression and Random Forest explicitly compared `class_weight=None`
> with `class_weight='balanced'`. Weighting was treated as a development-only
> hyperparameter rather than assumed to be beneficial.

> MAE was the primary continuous-emission regression metric. RMSE, median
> absolute error, R², normalized MAE and Spearman correlation were supplementary.
> PR-AUC was the primary classification metric. Balanced accuracy, recall,
> precision, F1-score, MCC, ROC-AUC, Brier score, log loss and reliability
> analysis were supplementary. Bootstrap confidence intervals and the number of
> positive held-out cases must be reported with the point estimates.

> The primary split was fixed at random state 42. Two alternative household
> split seeds were analysed as a development-selection sensitivity check. This
> stability analysis did not replace the locked primary model and did not select
> models using alternative holdout performance.

> The reported performance is an internal household-level held-out evaluation
> within the same thesis cohort. Because predictors and outcomes represent the
> same May-November 2024 study window, the models are cross-sectional explanatory
> models and must not be described as future-emission forecasting.



The matched-cohort augmentation uses `n_members`, `mean_age`, `floor_area`,
`total_monthly_expenditure_of_last_month`, `n_acs`, `n_appliances`,
`avg_hours_home`, and `occupant_density`, together with the five temporal
load-shape predictors. Current-period kWh, `log_daily_kwh`, emissions and
numeric cluster labels are excluded. The temporal-only and augmented models are compared on the same matched households using a fresh prespecified matched-cohort split, because the earlier full-cohort test results had already been inspected.


### Final regression interpretation rule

For the fresh teacher-matched regression, model family, hyperparameters and
original-versus-log target formulation are selected using nested development
cross-validation with **original-scale R² as the prespecified primary
criterion**. Missing household variables are median-imputed within each
training fold and accompanied by missingness indicators. The principal result
is always held-out **R² on the original kg CO₂ scale** after any log-target
prediction is back-transformed. Log-scale R² is supplementary only. A positive
held-out R² is not guaranteed and must not be manufactured by including
current-period electricity consumption, `log_daily_kwh`, emissions, or cluster
labels as predictors.

> Leave-one-feature-out ablation is performed on development folds only and is
> diagnostic rather than a second model-selection stage. The same selected model
> family and fold-specific hyperparameters from the full-feature nested-CV
> analysis are reused so that performance change reflects feature removal rather
> than compensating retuning.

> Held-out residual diagnostics characterize where the locked regression model
> errs, including error by actual-emission quartile. They are descriptive
> evaluation diagnostics and are not used to retune the model.

> For the matched temporal-only versus temporal + Wave 1 comparison, paired
> bootstrap confidence intervals are calculated from the same held-out
> households. Positive R² gain, MAE reduction, or PR-AUC gain indicates better
> held-out performance for the augmented model. These intervals quantify
> uncertainty in predictive-performance differences and do not imply causality.

